# Subtitle Generator v0.1
**使用顺序 / Workflow**
1. 先运行“初始化界面 / Init notebook UI”  
2. 在“选择文件”里二选一：Google Drive 或本地上传  
3. 设置“通用参数 / Required settings”  
4. 然后运行“生成字幕”  
5. 最后在“下载结果 / Download outputs”下载字幕

**说明 / Notes**
- 适合日语 / 中文语音字幕生成  
- 主流程：**faster-whisper/Qwen ASR → Qwen Forced Align → NeMo diarization → 导出 SRT / ASS**  
- 若不需要说话人区分，可在参数里关闭 diarization


**<font size='4'>以下单元格请按顺序执行，不可跳过步骤</font>**  
**【重要】:** 务必在“修改”→“笔记本设置”→“硬件加速器”中选择 GPU！否则处理速度会非常慢。
**</br>【IMPORTANT】:** Make sure you select GPU as hardware accelerator in notebook settings, otherwise processing will be very slow.


In [ ]:
#@title **初始化界面 / Init notebook UI**
import os
import re
import json
import math
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

COLAB_CONSTRAINTS = "/tmp/subtitle_generator_colab_constraints.txt"
with open(COLAB_CONSTRAINTS, "w", encoding="utf-8") as f:
    f.write(
        "numpy>=1.26,<2.2\n"
        "pandas==2.2.2\n"
        "fsspec==2025.3.0\n"
        "numexpr>=2.14.1\n"
        "numba>=0.60,<0.62\n"
    )

!apt update -y
!pip -q install -c {COLAB_CONSTRAINTS} "numpy>=1.26,<2.2" "pandas==2.2.2" "fsspec==2025.3.0" "numexpr>=2.14.1" "numba>=0.60,<0.62" faster-whisper transformers accelerate librosa soundfile qwen-asr ipywidgets

BTN_STYLE = widgets.ButtonStyle(
    button_color="#2563eb",
    text_color="#ffffff"
)
BTN_LAYOUT = widgets.Layout(width="200px", height="42px")

globals()["BTN_STYLE"] = BTN_STYLE
globals()["BTN_LAYOUT"] = BTN_LAYOUT

SPLIT_PUNCT = "。！？!?…"

DEFAULTS = {
    "AUDIO_FILE": "audio_16k_mono.wav",
    "input_file": "",
    "LANG": "ja",
    "WHISPER_MODEL": "large-v3",
    "BEAM_SIZE": 5,
    "WHISPER_BEAM_FLOOR": 8,
    "WHISPER_STYLE_PROMPT": "",
    "MIN_SEG_DUR": 0.35,
    "PAD_LEFT": 0.20,
    "MIN_CLIP_SEC": 0.05,
    "MIN_TOKEN_DUR": 0.02,
    "MAX_END_EXTEND": 1.0,
    "MAX_END_SHRINK": 0.2,
    "MIN_MERGE_DUR": 0.8,
    "MAX_GAP_MERGE": 0.22,
    "MAX_MERGE_DUR": 7.5,
    "ASS_FORCE_SINGLE_LINE": True,
    "ENABLE_WORD_LEVEL_FALLBACK": True,
    "ENABLE_ASR_CHUNKING": True,
    "HOTWORDS": "",

    # diarization 默认值
    "ENABLE_DIARIZATION": True,
    "DIAR_SCENARIO": "auto_route",
    "DIAR_SPEAKER_MODE": "auto",
    "DIAR_FIXED_SPK": 2,
    "DIAR_MIN_SPK": 1,
    "DIAR_MAX_SPK": 4,
    "ENABLE_DIAR_DIAGNOSTICS": True,
    "DIAR_LONG_AUDIO_SEC": 1080,
    "DIAR_CHUNK_SEC": 720,
    "DIAR_CHUNK_OVERLAP_SEC": 25,
    "DIAR_BOUNDARY_SEARCH_SEC": 20,
    "DIAR_CHUNK_MIN_SEC": 180,

    "MAX_SENT_DUR": 8.5,
    "MIN_SENT_DUR": 0.8,
    "MAX_CHARS_PER_CHUNK": 34,
    "SR": 16000,
}

for k, v in DEFAULTS.items():
    if k not in globals():
        globals()[k] = v

def clean_token_text(text):
    return re.sub(r"\s+", "", (text or "").strip())

def looks_like_long_tail(text):
    t = re.sub(r"[。．，、【】！？!?…\s]+$", "", text or "")
    return t.endswith(("ー", "～", "ね", "よ", "な", "わ", "さ", "ぞ", "か"))

def pad_right(text):
    if looks_like_long_tail(text):
        return 1.6
    if len(text or "") <= 6:
        return 1.0
    return 0.8

display(HTML("""
<style>
/* 整个 button 元素 */
.widget-button,
.jupyter-widgets .widget-button,
button.widget-button,
button.jp-Button,
.widget-button > button {
    background: #2563eb !important;
    background-color: #2563eb !important;
    color: #ffffff !important;
    border: 1px solid #1d4ed8 !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    text-shadow: none !important;
    box-shadow: none !important;
    opacity: 1 !important;
}

/* 内层文字/图标 */
.widget-button * ,
.jupyter-widgets .widget-button *,
button.widget-button *,
button.jp-Button * {
    color: #ffffff !important;
    fill: #ffffff !important;
}

/* hover */
.widget-button:hover,
.jupyter-widgets .widget-button:hover,
button.widget-button:hover,
button.jp-Button:hover {
    background: #1d4ed8 !important;
    background-color: #1d4ed8 !important;
    color: #ffffff !important;
}

/* active */
.widget-button:active,
.jupyter-widgets .widget-button:active,
button.widget-button:active,
button.jp-Button:active {
    background: #1e40af !important;
    background-color: #1e40af !important;
    color: #ffffff !important;
}

/* disabled */
.widget-button:disabled,
.jupyter-widgets .widget-button:disabled,
button.widget-button:disabled,
button.jp-Button:disabled {
    background: #4b5563 !important;
    background-color: #4b5563 !important;
    color: #d1d5db !important;
    border-color: #4b5563 !important;
    opacity: 1 !important;
}
</style>
"""))

print("UI 初始化完成。请在下面二选一选择文件，然后运行参数单元。")


## 选择文件
**完成文件选择后，请务必再执行一次下面的“通用参数 / Required settings”单元**


In [ ]:
# @markdown <font size="2">Choose one input source below. Switching source only updates the UI and will not overwrite the current selected file until you click the action button.</font>
# @markdown <br/><font size="2">请先选择输入来源。切换来源只会更新界面，不会自动覆盖当前已选文件；只有点击对应按钮后才会更新。</font>

import os
import ipywidgets as widgets
from google.colab import drive, files
from IPython.display import display, clear_output

source_selector = widgets.ToggleButtons(
    options=[("Google Drive", "drive"), ("Local Upload", "local")],
    value="local",
    description="Source:",
    layout=widgets.Layout(width='360px'),
    style={'description_width': 'initial'}
)

file_input_panel = widgets.Output()


def _render_file_input(mode):
    with file_input_panel:
        clear_output()
        # Safe access to global styles to avoid TraitError if Init cell wasn't run
        btn_layout = globals().get("BTN_LAYOUT", widgets.Layout())
        btn_style = globals().get("BTN_STYLE", widgets.ButtonStyle())

        if mode == "drive":
            mount_btn = widgets.Button(
                description='挂载 Drive / Mount',
                icon='hdd-o',
                layout=btn_layout,
                style=btn_style
            )
            mount_output = widgets.Output()
            drive_path_input = widgets.Text(
                value=globals().get("input_file", ""),
                placeholder='/content/drive/MyDrive/your_folder/your_audio.mp4',
                description='Path:',
                layout=widgets.Layout(width='920px')
            )
            drive_pick_btn = widgets.Button(
                description='确定文件 / Select',
                icon='check',
                layout=btn_layout,
                style=btn_style
            )
            drive_output = widgets.Output()

            is_mounted = os.path.isdir('/content/drive/MyDrive')
            drive_path_input.disabled = not is_mounted
            drive_pick_btn.disabled = not is_mounted

            def _set_drive_file(_):
                path = drive_path_input.value.strip()
                with drive_output:
                    clear_output()
                    if path and os.path.exists(path):
                        globals()["input_file"] = path
                        print(f"✅ 已选择文件: {path}")
                    else:
                        print("❌ 路径不存在，请检查后重试。")

            def _mount_drive(_):
                with mount_output:
                    clear_output()
                    try:
                        drive.mount('/content/drive', force_remount=False)
                    except Exception as e:
                        err_msg = str(e).strip() or repr(e)
                        lower_msg = err_msg.lower()
                        if "cancel" in lower_msg or "aborted" in lower_msg or "denied" in lower_msg:
                            print("⚠️ 挂载已取消或授权未完成，请重新点击按钮并完成授权。")
                        elif "path" in lower_msg or "no such file" in lower_msg or "not found" in lower_msg:
                            print("❌ Drive 挂载路径不可用，请稍后重试或检查 Colab 运行环境。")
                        else:
                            print("❌ 挂载失败，请重试。")
                        print(f"详情: {err_msg}")
                        return

                    if os.path.isdir('/content/drive/MyDrive'):
                        drive_path_input.disabled = False
                        drive_pick_btn.disabled = False
                        print("✅ Drive 已挂载。现在可以粘贴文件路径并点击“确定文件”。")
                    else:
                        print("⚠️ 未检测到 /content/drive/MyDrive，请确认挂载是否完成。")

            drive_pick_btn.on_click(_set_drive_file)
            mount_btn.on_click(_mount_drive)
            display(widgets.HTML('<font size="2">Use Drive mode in two steps: (1) click "挂载 Drive / Mount" first, then (2) paste the full file path and click select.<br/>Drive 模式请先点击“挂载 Drive / Mount”，挂载成功后再粘贴完整路径并点击“确定文件”。</font>'))
            display(widgets.VBox([mount_btn, mount_output, drive_path_input, drive_pick_btn, drive_output]))
        else:
            upload_btn = widgets.Button(
                description='上传并选择 / Upload',
                icon='upload',
                layout=btn_layout,
                style=btn_style
            )
            upload_output = widgets.Output()

            def _upload_local_file(_):
                with upload_output:
                    clear_output()
                    uploaded = files.upload()
                    if uploaded:
                        picked = next(iter(uploaded.keys()))
                        globals()["input_file"] = picked
                        print(f"✅ 已选择文件: {picked}")
                    else:
                        print("未选择文件。")

            upload_btn.on_click(_upload_local_file)
            display(widgets.HTML('<font size="2">Use this mode to upload a local file. You can run it again to replace the current selected file.<br/>如需从本地上传文件，请点击下方按钮；如需重新选择本地文件，可再次执行本单元格。</font>'))
            display(widgets.VBox([upload_btn, upload_output]))


def _on_source_change(change):
    if change.get('name') == 'value':
        _render_file_input(change['new'])


source_selector.observe(_on_source_change, names='value')
display(widgets.VBox([source_selector, file_input_panel]))
_render_file_input(source_selector.value)

## 设置参数

In [ ]:
#@title **通用参数**
# @markdown **【重要】**：选择文件后，请执行一次本单元格。修改参数后，重新运行本格即可生效。
# @markdown 运行顺序：**先选文件 → 再运行本参数格 → 然后顺序执行后续单元**

# ========= 基础参数 =========
语言 = "ja"  # @param {type:"string"}
Whisper模型 = "large-v3"  # @param ["medium", "large-v3", "large-v3-turbo"]
束搜索大小 = 5  # @param {type:"slider", min:1, max:8, step:1}
ASS强制单行 = True  # @param {type:"boolean"}
启用词级回退 = True  # @param {type:"boolean"}

启用ASR长音频切分 = True  # @param {type:"boolean"}

# ========= 说话人识别 =========
启用说话人识别 = True  # @param {type:"boolean"}
# @markdown - `auto`：自动估计说话人数（默认）
# @markdown - `fixed`：使用下面的“固定说话人数”
# @markdown - `range`：使用下面的“说话人数范围”
说话人场景 = "auto_route"  # @param ["auto_route", "single_low_overlap", "multi_overlap"]
说话人数模式 = "auto"  # @param ["auto", "fixed", "range"]
固定说话人数 = 4  # @param {type:"slider", min:1, max:8, step:1}
说话人数范围 = [1,4]  # @param {type:"raw"}
长音频阈值秒 = 1080  # @param {type:"integer"}
diar分块秒 = 720  # @param {type:"integer"}
diar分块重叠秒 = 25  # @param {type:"integer"}
diar边界搜索秒 = 20  # @param {type:"integer"}
diar最小分块秒 = 180  # @param {type:"integer"}

def _parse_speaker_range(v):
    if isinstance(v, (list, tuple)) and len(v) == 2:
        a, b = int(v[0]), int(v[1])
        return (a, b) if a <= b else (b, a)
    if isinstance(v, str):
        s = v.strip().replace("[", "").replace("]", "").replace("(", "").replace(")", "")
        parts = [x.strip() for x in s.split(",") if x.strip()]
        if len(parts) == 2:
            a, b = int(parts[0]), int(parts[1])
            return (a, b) if a <= b else (b, a)
    return (
        int(globals().get("DIAR_MIN_SPK", 1)),
        int(globals().get("DIAR_MAX_SPK", 4)),
    )

# ========= 热词 =========
热词预设 = "无" # @param ["无","原神","星铁","自定义"]

# @markdown **自定义热词（仅在“自定义”时生效；多个词用顿号分隔）**
自定义热词 = ""  # @param {type:"string"}

# ========= 对齐高级参数 =========
最短分段时长 = 0.35
左侧补偿 = 0.20
最短裁剪时长 = 0.05
最短token时长 = 0.02
句尾最大延长 = 1.0
句尾最大收缩 = 0.2
最短合并时长 = 0.8
最大合并间隔 = 0.22
最大合并时长 = 7.5
与下一句安全间隔 = 0.03
最短句子时长 = 0.15

# ========= 热词预设字典 =========
HOTWORDS_PRESETS = {
    "无": "",
    "原神": """
\\ 以下は『原神』関連の日本語固有名詞です。固有名詞はできるだけこの表記を優先してください。

\\ 作品・基本用語：
原神、テイワット、元素、元素反応、元素スキル、元素爆発、聖遺物、祈願、原石、モラ、天然樹脂、濃縮樹脂、七天神像、地脈、深境螺旋、伝説任務、魔神任務、世界任務、デイリー依頼、神の瞳、アビス

\\ 地域・国：
モンド、璃月、稲妻、スメール、フォンテーヌ、ナタ、スネージナヤ、セレスティア、ナド・クライ

\\ 主要キャラクター：
旅人、空、蛍、パイモン
ジン、アンバー、リサ、ガイア、バーバラ、ディルック、レザー、ウェンティ、クレー、アルベド、ロサリア、エウルア、ミカ、モナ、ディオナ、スクロース、ノエル、ベネット、フィッシュル
刻晴、甘雨、七七、魈、鍾離、胡桃、夜蘭、申鶴、白朮、ヨォーヨ、凝光、北斗、行秋、香菱、煙緋、辛炎、雲菫
雷電将軍、神里綾華、神里綾人、宵宮、楓原万葉、珊瑚宮心海、八重神子、荒瀧一斗、久岐忍、ゴロー、トーマ、早柚、鹿野院平蔵、九条裟羅
ナヒーダ、アルハイゼン、セノ、ニィロウ、放浪者、ティナリ、ディシア、キャンディス、ファルザン、レイラ、コレイ、ドリー、カーヴェ
フリーナ、ヌヴィレット、リオセスリ、ナヴィア、クロリンデ、千織、リネ、リネット、フレミネ、シャルロット、エスコフィエ
マーヴィカ、シトラリ、キィニチ、ムアラニ、チャスカ、シロネン、イファ、ヴァレサ、オロルン、イアンサ、カチーナ
アイノ、イネファ、ラウマ、フリンズ、コロンビーナ、ネフェル、ドゥリン、ファルカ、ヤフォダ、アリス、ニコ、ローエン、リンネア
スカーク、タルタリヤ、アルレッキーノ

\\ 声優：
前野智昭、村瀬歩、古賀葵、堀江瞬、ほりえる、木村良平、小松昌平、こまちょえ
""",
    "星铁": """
\\ 以下は『崩壊：スターレイル』関連の日本語固有名詞です。固有名詞はできるだけこの表記を優先してください。

\\ 作品・基本用語：
崩壊：スターレイル、スターレイル、星穹列車、開拓者、星核、星神、運命、軌跡、光円錐、星魂、模擬宇宙、階差宇宙、混沌の記憶、虚構叙事、末日の幻影、開拓クエスト、同行クエスト、均衡レベル、遺物、次元界オーナメント、開拓力、歴戦余韻

\\ 勢力・地域：
宇宙ステーション「ヘルタ」、ヤリーロVI、ベロブルグ、仙舟「羅浮」、ピノコニー、オンパロス、星核ハンター、スターピースカンパニー、雲上の五騎士、二相楽園

\\ 運命：
壊滅、知恵、巡狩、調和、虚無、存護、豊穣、記憶、愉悦

\\ キャラクター：
開拓者、三月なのか、丹恒、姫子、ヴェルト、パム、ヘルタ、アスター、アーラン
ブローニャ、ゼーレ、ジェパード、ペラ、サンポ、クラーラ、フック、セーバル、ルカ、リンクス
景元、符玄、刃、銀狼、カフカ、羅刹、飲月、鏡流、桂乃芬、素裳、停雲、白露、彦卿、雪衣、寒鴉、フォフォ
ルアン・メェイ、Dr.レイシオ、花火、黄泉、ブラックスワン、アベンチュリン、ロビン、ブートヒル、ホタル、ジェイド、雲璃、椒丘、霊砂、乱破
サンデー、アグライア、トリビー、モーディス、キャストリス、アナイクス、ファイノン

\\ 声優名：
榎木淳弥、石川由依、田中理恵、伊東健人、沢城みゆき、武内駿輔、斎藤千和、阿座上洋平、内田雄馬、遠藤綾、諏訪部順一
""",
}

# ========= 工具函数 =========
def _parse_range(v, default_min=1, default_max=4):
    if isinstance(v, (list, tuple)) and len(v) == 2:
        a, b = int(v[0]), int(v[1])
        return (a, b) if a <= b else (b, a)
    if isinstance(v, str):
        s = v.strip().replace("[", "").replace("]", "").replace("(", "").replace(")", "")
        parts = [x.strip() for x in s.split(",") if x.strip()]
        if len(parts) == 2:
            a, b = int(parts[0]), int(parts[1])
            return (a, b) if a <= b else (b, a)
    return default_min, default_max

# ========= 生成热词 =========
def build_hotwords():
    if 热词预设 == "自定义":
        final_prompt = f"{自定义热词.strip()}"
    else:
        final_prompt = HOTWORDS_PRESETS.get(热词预设, "")
    
    words = []
    for line in final_prompt.splitlines():
        if line.startswith("\\") or line.startswith("#") or line.startswith("//"):
            continue
        #split by ,、.。 ，。
        words.extend(re.split(r"[，、,.。]", line.strip()))

    return ' '.join([w for w in words if w])

# ========= 写回全局变量 =========
globals()["LANG"] = str(语言).strip() or "ja"
globals()["WHISPER_MODEL"] = Whisper模型
globals()["BEAM_SIZE"] = int(束搜索大小)
globals()["ASS_FORCE_SINGLE_LINE"] = bool(ASS强制单行)
globals()["ENABLE_WORD_LEVEL_FALLBACK"] = bool(启用词级回退)

globals()["ENABLE_ASR_CHUNKING"] = bool(启用ASR长音频切分)

globals()["ENABLE_DIARIZATION"] = bool(启用说话人识别)
globals()["DIAR_SCENARIO"] = 说话人场景
globals()["DIAR_SPEAKER_MODE"] = 说话人数模式
globals()["DIAR_LONG_AUDIO_SEC"] = max(60, int(长音频阈值秒))
globals()["DIAR_CHUNK_SEC"] = max(120, int(diar分块秒))
globals()["DIAR_CHUNK_OVERLAP_SEC"] = max(5, int(diar分块重叠秒))
globals()["DIAR_BOUNDARY_SEARCH_SEC"] = max(0, int(diar边界搜索秒))
globals()["DIAR_CHUNK_MIN_SEC"] = max(60, int(diar最小分块秒))

if 说话人数模式 == "fixed":
    globals()["DIAR_FIXED_SPK"] = int(固定说话人数)

elif 说话人数模式 == "range":
    _min_spk, _max_spk = _parse_speaker_range(说话人数范围)
    globals()["DIAR_MIN_SPK"] = int(_min_spk)
    globals()["DIAR_MAX_SPK"] = int(_max_spk)

globals()["MIN_SEG_DUR"] = float(最短分段时长)
globals()["PAD_LEFT"] = float(左侧补偿)
globals()["MIN_CLIP_SEC"] = float(最短裁剪时长)
globals()["MIN_TOKEN_DUR"] = float(最短token时长)
globals()["MAX_END_EXTEND"] = float(句尾最大延长)
globals()["MAX_END_SHRINK"] = float(句尾最大收缩)
globals()["MIN_MERGE_DUR"] = float(最短合并时长)
globals()["MAX_GAP_MERGE"] = float(最大合并间隔)
globals()["MAX_MERGE_DUR"] = float(最大合并时长)
globals()["MIN_SENT_DUR"] = float(最短句子时长)
ENABLE_OVERLAP_RESCUE = True

assert globals().get("input_file"), "请先在上面的文件选择单元里选择文件。"

print("✅ 参数已应用")
print("input_file =", globals().get("input_file"))
print("语言 =", globals().get("LANG"))
print("Whisper模型 =", globals().get("WHISPER_MODEL"))
print("说话人数模式 =", globals().get("DIAR_SPEAKER_MODE"))
print("长音频阈值秒 =", globals().get("DIAR_LONG_AUDIO_SEC"))
print("diar分块秒 =", globals().get("DIAR_CHUNK_SEC"))
print("diar分块重叠秒 =", globals().get("DIAR_CHUNK_OVERLAP_SEC"))

if globals()["DIAR_SPEAKER_MODE"] == "fixed":
    print("固定说话人数 =", globals().get("DIAR_FIXED_SPK"))
elif globals()["DIAR_SPEAKER_MODE"] == "range":
    print("说话人数范围 =", (globals().get("DIAR_MIN_SPK"), globals().get("DIAR_MAX_SPK")))
else:
    print("说话人数 = 自动")

print("ASS强制单行 =", globals().get("ASS_FORCE_SINGLE_LINE"))

print("ASR长音频切分 =", globals().get("ENABLE_ASR_CHUNKING"))


## 生成字幕

In [ ]:
#@title **音频转换与共享切分**

# ===== 转换为 16k 单声道 wav，并提前生成 ASR / diarization 共用长音频切片 =====
assert globals().get("input_file"), "请先在前置设置单元里选择文件"

import json
import os
import shutil
import subprocess

import soundfile as sf

input_path = globals()["input_file"]
shared_audio_source = "audio_16k_mono.wav"
shared_chunk_dir = "shared_audio_chunks"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        shared_audio_source,
    ],
    check=True,
)

info = sf.info(shared_audio_source)
audio_duration = float(info.frames) / float(info.samplerate)

# 清理旧切片，避免短音频或参数变化后误用上一次运行的 chunk。
if os.path.isdir(shared_chunk_dir):
    shutil.rmtree(shared_chunk_dir)

long_audio_sec = max(60.0, float(globals().get("DIAR_LONG_AUDIO_SEC", 1080)))
chunk_sec = max(120.0, float(globals().get("DIAR_CHUNK_SEC", 720)))
chunk_overlap_sec = max(0.0, float(globals().get("DIAR_CHUNK_OVERLAP_SEC", 25)))
chunk_overlap_sec = min(chunk_overlap_sec, max(0.0, chunk_sec / 2.0 - 1.0))
chunk_min_sec = max(60.0, float(globals().get("DIAR_CHUNK_MIN_SEC", 180)))


def _round_sec(v):
    return round(float(v), 3)


def _plan_shared_audio_chunks(audio_len: float):
    if audio_len <= long_audio_sec:
        return [
            {
                "index": 0,
                "base_start": 0.0,
                "base_end": _round_sec(audio_len),
                "extract_start": 0.0,
                "extract_end": _round_sec(audio_len),
                "overlap_start": 0.0,
                "overlap_end": 0.0,
                "wav_path": os.path.abspath(shared_audio_source),
            }
        ]

    chunks = []
    base_start = 0.0
    idx = 0
    while base_start < audio_len - 1e-6:
        base_end = min(audio_len, base_start + chunk_sec)
        if audio_len - base_end < chunk_min_sec:
            base_end = float(audio_len)
        if base_end <= base_start:
            base_end = min(audio_len, base_start + chunk_sec)

        extract_start = 0.0 if idx == 0 else max(0.0, base_start - chunk_overlap_sec)
        chunk = {
            "index": idx,
            "base_start": _round_sec(base_start),
            "base_end": _round_sec(base_end),
            "extract_start": _round_sec(extract_start),
            "extract_end": _round_sec(base_end),
            "overlap_start": _round_sec(extract_start if idx > 0 else 0.0),
            "overlap_end": _round_sec(base_start if idx > 0 else 0.0),
            "wav_path": os.path.abspath(os.path.join(shared_chunk_dir, f"chunk_{idx:03d}.wav")),
        }
        chunks.append(chunk)
        base_start = base_end
        idx += 1

    return chunks


shared_chunks = _plan_shared_audio_chunks(audio_duration)

if len(shared_chunks) > 1:
    os.makedirs(shared_chunk_dir, exist_ok=True)
    for chunk in shared_chunks:
        duration = max(0.0, float(chunk["extract_end"]) - float(chunk["extract_start"]))
        subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", shared_audio_source,
                "-ss", f"{float(chunk['extract_start']):.3f}",
                "-t", f"{duration:.3f}",
                "-ac", "1",
                "-ar", "16000",
                chunk["wav_path"],
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

shared_debug = {
    "audio_source": os.path.abspath(shared_audio_source),
    "audio_duration": round(audio_duration, 3),
    "chunk_count": len(shared_chunks),
    "chunked": len(shared_chunks) > 1,
    "asr_chunking_enabled": bool(globals().get("ENABLE_ASR_CHUNKING", True)),
    "params": {
        "long_audio_sec": long_audio_sec,
        "chunk_sec": chunk_sec,
        "chunk_overlap_sec": chunk_overlap_sec,
        "chunk_min_sec": chunk_min_sec,
    },
}

with open("shared_audio_chunks.json", "w", encoding="utf-8") as f:
    json.dump({"debug": shared_debug, "chunks": shared_chunks}, f, ensure_ascii=False, indent=2)

globals()["AUDIO_FILE"] = shared_audio_source
globals()["SHARED_AUDIO_SOURCE"] = shared_audio_source
globals()["SHARED_AUDIO_CHUNKS"] = shared_chunks
globals()["SHARED_AUDIO_DEBUG"] = shared_debug
globals()["AUDIO_LEN"] = audio_duration
globals()["audio_len"] = audio_duration
globals()["ASR_CHUNKING_ACTIVE"] = bool(globals().get("ENABLE_ASR_CHUNKING", True)) and len(shared_chunks) > 1

print("saved:", shared_audio_source)
print("audio_len:", round(audio_duration, 3), "sec")
print("shared chunks:", len(shared_chunks), "ASR chunking active:", globals()["ASR_CHUNKING_ACTIVE"])
if len(shared_chunks) > 1:
    print("first chunk:", shared_chunks[0])
    print("last chunk:", shared_chunks[-1])


### Whisper转写

In [ ]:
#@title **模型加载**
# Cell 4: faster-whisper transcription + bad-segment cleanup + local rescue
from faster_whisper import WhisperModel
import json, gc, torch, os, re, soundfile as sf
from importlib.metadata import PackageNotFoundError, version
from qwen_asr import Qwen3ASRModel, Qwen3ForcedAligner

AUDIO_FILE = globals().get("AUDIO_FILE", "audio.wav")

# ===== free VRAM first =====
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Before load VRAM free: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

HOTWORDS = build_hotwords()

fw_model = WhisperModel(
    WHISPER_MODEL,
    device="cuda",
    compute_type="float16"
)

# Qwen ASR: lazy load only when tail supplement needs extra ASR
qwen_asr_model = None

def ensure_qwen_asr_model():
    global qwen_asr_model
    if qwen_asr_model is None:
        print("[qwen asr] loading on demand for tail supplement")
        qwen_asr_model = Qwen3ASRModel.from_pretrained(
            "Qwen/Qwen3-ASR-0.6B",
            dtype=torch.float16,
            device_map="cuda:0",
            max_inference_batch_size=1,
            max_new_tokens=256,
        )
    return qwen_asr_model

aligner = Qwen3ForcedAligner.from_pretrained(
    "Qwen/Qwen3-ForcedAligner-0.6B",
    dtype=torch.float16,
    device_map="cuda:0"
)

def clean_compact_text(s):
    return re.sub(r"\s+", "", (s or "").strip())


def build_hotword_terms(hotwords):
    terms = []
    seen = set()
    for raw in re.split(r"[\s,，、.。/／|;；:：]+", hotwords or ""):
        term = clean_compact_text(raw)
        if len(term) < 2:
            continue
        if term in seen:
            continue
        seen.add(term)
        terms.append(term)
    terms.sort(key=len, reverse=True)
    return terms

def pkg_ver(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "missing"
    except Exception as e:
        return f"error:{type(e).__name__}"

hotwords_compact = clean_compact_text(HOTWORDS)
HOTWORD_TERMS = build_hotword_terms(HOTWORDS)
globals()["HOTWORD_TERMS"] = HOTWORD_TERMS
HOTWORD_HALLUCINATION_ENABLE = bool(globals().get("HOTWORD_HALLUCINATION_ENABLE", True))
HOTWORD_COVER_RATIO = float(globals().get("HOTWORD_COVER_RATIO", 0.72))
HOTWORD_LOW_CPS = float(globals().get("HOTWORD_LOW_CPS", 4.5))
HOTWORD_REPEAT_MIN = int(globals().get("HOTWORD_REPEAT_MIN", 2))
SUBTITLE_GAP_DIAG_THRESHOLD_SEC = float(globals().get("SUBTITLE_GAP_DIAG_THRESHOLD_SEC", 5.0))
SUBTITLE_GAP_CHUNK_BOUNDARY_WINDOW_SEC = float(globals().get("SUBTITLE_GAP_CHUNK_BOUNDARY_WINDOW_SEC", 30.0))
WHISPER_BEAM_FLOOR = max(1, int(globals().get("WHISPER_BEAM_FLOOR", 8)))
WHISPER_STYLE_PROMPT = str(globals().get("WHISPER_STYLE_PROMPT", "") or "").strip()
print("[runtime] faster-whisper=", pkg_ver("faster-whisper"), "ctranslate2=", pkg_ver("ctranslate2"), "transformers=", pkg_ver("transformers"), "qwen-asr=", pkg_ver("qwen-asr"))
print("[runtime] hotwords_enabled=", bool(hotwords_compact), "hotwords_chars=", len(hotwords_compact), "hotword_terms=", len(HOTWORD_TERMS))
print("[runtime] whisper_beam_floor=", WHISPER_BEAM_FLOOR, "style_prompt=", bool(WHISPER_STYLE_PROMPT))
if len(hotwords_compact) > 120:
    print("[runtime] warning: hotwords prompt is long; tail dropout can become more likely")

TAIL_RESCUE_ENABLE = bool(globals().get("TAIL_RESCUE_ENABLE", True))
TAIL_RESCUE_SEC = float(globals().get("TAIL_RESCUE_SEC", 18.0))
TAIL_RESCUE_LEFT_PAD = float(globals().get("TAIL_RESCUE_LEFT_PAD", 1.5))
TAIL_RESCUE_MIN_NEW_CHARS = int(globals().get("TAIL_RESCUE_MIN_NEW_CHARS", 2))
TAIL_RESCUE_MAX_LOOKBACK = float(globals().get("TAIL_RESCUE_MAX_LOOKBACK", 24.0))
TAIL_RESCUE_STRICT_START_TOL = float(globals().get("TAIL_RESCUE_STRICT_START_TOL", 0.10))
TAIL_RESCUE_DEDUP_SIM = float(globals().get("TAIL_RESCUE_DEDUP_SIM", 0.78))
TAIL_MERGE_GAP_SEC = float(globals().get("TAIL_MERGE_GAP_SEC", 0.6))
TAIL_SUPPLEMENT_ENABLE = bool(globals().get("TAIL_SUPPLEMENT_ENABLE", True))
TAIL_SUPPLEMENT_TRIGGER_SEC = float(globals().get("TAIL_SUPPLEMENT_TRIGGER_SEC", 0.55))
TAIL_SUPPLEMENT_LOOKBACK = float(globals().get("TAIL_SUPPLEMENT_LOOKBACK", 28.0))
TAIL_SUPPLEMENT_LEFT_CONTEXT = float(globals().get("TAIL_SUPPLEMENT_LEFT_CONTEXT", 4.0))
TAIL_SUPPLEMENT_MIN_NEW_CHARS = int(globals().get("TAIL_SUPPLEMENT_MIN_NEW_CHARS", 4))
MAIN_HOTWORDS_FALLBACK_ENABLE = bool(globals().get("MAIN_HOTWORDS_FALLBACK_ENABLE", True))
MAIN_HOTWORDS_FALLBACK_TRIGGER_SEC = float(globals().get("MAIN_HOTWORDS_FALLBACK_TRIGGER_SEC", 8.0))
MAIN_HOTWORDS_FALLBACK_MIN_IMPROVE_SEC = float(globals().get("MAIN_HOTWORDS_FALLBACK_MIN_IMPROVE_SEC", 4.0))



In [ ]:
#@title **Whisper 主转写**

# ===== 主识别：支持共享 chunk 转写与尾部缺失 fallback =====

class _ChunkedTranscribeInfo:
    def __init__(self, duration=0.0, language=None, language_probability=0.0):
        self.duration = float(duration or 0.0)
        self.language = language
        self.language_probability = float(language_probability or 0.0)


class _OffsetWord:
    def __init__(self, word, start, end):
        self.word = word
        self.start = start
        self.end = end


class _OffsetSegment:
    def __init__(self, start, end, text, words):
        self.start = start
        self.end = end
        self.text = text
        self.words = words


def _transcribe_audio_input(audio_input, hotwords):
    return fw_model.transcribe(
        audio_input,
        language=LANG,
        beam_size=max(WHISPER_BEAM_FLOOR, int(BEAM_SIZE)),
        best_of=max(WHISPER_BEAM_FLOOR, int(BEAM_SIZE)),
        temperature=0.0,
        condition_on_previous_text=False,
        compression_ratio_threshold=2.4,
        log_prob_threshold=-1.0,
        no_speech_threshold=0.3,
        max_initial_timestamp=0,
        word_timestamps=True,
        vad_filter=False,
        initial_prompt=WHISPER_STYLE_PROMPT,
        hotwords=hotwords,
    )


def _shift_whisper_segment(seg, offset_sec):
    offset_sec = float(offset_sec or 0.0)
    words = []
    for w in (getattr(seg, "words", None) or []):
        ws = getattr(w, "start", None)
        we = getattr(w, "end", None)
        wt = getattr(w, "word", "")
        if ws is None or we is None:
            continue
        ws = float(ws) + offset_sec
        we = float(we) + offset_sec
        if we <= ws:
            continue
        words.append(_OffsetWord(wt, ws, we))
    return _OffsetSegment(
        float(getattr(seg, "start", 0.0) or 0.0) + offset_sec,
        float(getattr(seg, "end", 0.0) or 0.0) + offset_sec,
        getattr(seg, "text", "") or "",
        words,
    )


def _segment_midpoint(seg):
    return 0.5 * (float(seg.start) + float(seg.end))


def _keep_chunk_segment(seg, chunk, is_last_chunk=False):
    mid = _segment_midpoint(seg)
    base_start = float(chunk["base_start"])
    base_end = float(chunk["base_end"])
    if mid < base_start - 1e-6:
        return False
    if is_last_chunk:
        return mid <= base_end + 1e-6
    return mid < base_end - 1e-6


def _build_chunked_info(chunk_infos, duration):
    best = None
    for info_obj in chunk_infos:
        if info_obj is None:
            continue
        prob = float(getattr(info_obj, "language_probability", 0.0) or 0.0)
        if best is None or prob > float(getattr(best, "language_probability", 0.0) or 0.0):
            best = info_obj
    return _ChunkedTranscribeInfo(
        duration=duration,
        language=getattr(best, "language", None) if best is not None else None,
        language_probability=getattr(best, "language_probability", 0.0) if best is not None else 0.0,
    )


def _shared_asr_chunks():
    chunks = globals().get("SHARED_AUDIO_CHUNKS", []) or []
    if not bool(globals().get("ENABLE_ASR_CHUNKING", True)):
        return []
    if len(chunks) <= 1:
        return []
    usable = []
    for chunk in chunks:
        wav_path = chunk.get("wav_path")
        if wav_path and os.path.exists(wav_path):
            usable.append(chunk)
    return usable if len(usable) == len(chunks) else []


def run_main_transcribe(hotwords):
    chunks = _shared_asr_chunks()
    audio_duration = float(globals().get("AUDIO_LEN", globals().get("audio_len", 0.0)) or 0.0)

    if not chunks:
        segments_gen, info = _transcribe_audio_input(AUDIO_FILE, hotwords)
        segments = list(segments_gen)
        globals()["_asr_chunk_debug"] = {
            "chunked": False,
            "chunk_count": 1,
            "hotwords_enabled": bool(clean_compact_text(hotwords)),
            "fallback_run": False,
            "merged_segments": len(segments),
            "dedup_dropped": 0,
            "chunks": [],
        }
        return segments, info

    merged_segments = []
    chunk_infos = []
    dedup_dropped = 0
    chunk_records = []

    print(f"[ASR chunk] enabled: chunks={len(chunks)}")
    for idx, chunk in enumerate(chunks):
        wav_path = chunk["wav_path"]
        segments_gen, info = _transcribe_audio_input(wav_path, hotwords)
        local_segments = list(segments_gen)
        chunk_infos.append(info)

        kept = []
        dropped = 0
        for seg in local_segments:
            shifted = _shift_whisper_segment(seg, float(chunk["extract_start"]))
            if _keep_chunk_segment(shifted, chunk, is_last_chunk=(idx == len(chunks) - 1)):
                kept.append(shifted)
            else:
                dropped += 1

        dedup_dropped += dropped
        merged_segments.extend(kept)
        chunk_records.append(
            {
                "index": int(chunk["index"]),
                "base_start": float(chunk["base_start"]),
                "base_end": float(chunk["base_end"]),
                "extract_start": float(chunk["extract_start"]),
                "extract_end": float(chunk["extract_end"]),
                "raw_segments": len(local_segments),
                "kept_segments": len(kept),
                "dedup_dropped": dropped,
            }
        )
        print(
            "[ASR chunk]",
            int(chunk["index"]),
            f"{float(chunk['extract_start']):.3f}->{float(chunk['extract_end']):.3f}",
            "raw=", len(local_segments),
            "kept=", len(kept),
            "dropped=", dropped,
        )

    merged_segments.sort(key=lambda x: (float(x.start), float(x.end)))
    info = _build_chunked_info(chunk_infos, audio_duration)
    globals()["_asr_chunk_debug"] = {
        "chunked": True,
        "chunk_count": len(chunks),
        "hotwords_enabled": bool(clean_compact_text(hotwords)),
        "fallback_run": False,
        "merged_segments": len(merged_segments),
        "dedup_dropped": dedup_dropped,
        "chunks": chunk_records,
    }
    print("[ASR chunk] merged_segments=", len(merged_segments), "dedup_dropped=", dedup_dropped)
    return merged_segments, info


def log_raw_transcribe(label, segments, info):
    audio_len = float(getattr(info, "duration", 0.0) or 0.0)
    last_end = float(segments[-1].end) if segments else 0.0
    tail_gap = audio_len - last_end if audio_len > 0 else None
    print(label, "segments=", len(segments), "last_end=", round(last_end, 3), "audio_len=", round(audio_len, 3) if audio_len > 0 else "?", "tail_gap=", round(tail_gap, 3) if tail_gap is not None else "?")
    if getattr(info, "language", None) is not None:
        print(label, "detected_language=", info.language, "prob=", round(float(getattr(info, "language_probability", 0.0) or 0.0), 4))
    tail_label = label + " tail seg"
    for seg in segments[-4:]:
        seg_text = re.sub(r"\s+", " ", (getattr(seg, "text", "") or "")).strip()
        print(tail_label, round(float(seg.start), 3), "->", round(float(seg.end), 3), repr(seg_text[:120]))
    return audio_len, last_end, tail_gap


raw_segments, info = run_main_transcribe(HOTWORDS)
raw_audio_len, raw_last_end, raw_tail_gap = log_raw_transcribe("[raw whisper]", raw_segments, info)
raw_asr_debug = dict(globals().get("_asr_chunk_debug", {}) or {})
globals()["_main_hotwords_fallback_used"] = False

if MAIN_HOTWORDS_FALLBACK_ENABLE and hotwords_compact and raw_tail_gap is not None and raw_tail_gap >= MAIN_HOTWORDS_FALLBACK_TRIGGER_SEC:
    print("[raw whisper] large tail gap detected; retrying full transcription without hotwords")
    fallback_segments, fallback_info = run_main_transcribe("")
    fallback_debug = dict(globals().get("_asr_chunk_debug", {}) or {})
    fallback_audio_len, fallback_last_end, fallback_tail_gap = log_raw_transcribe("[raw whisper fallback]", fallback_segments, fallback_info)
    gap_improve = raw_tail_gap - fallback_tail_gap if fallback_tail_gap is not None else 0.0
    if fallback_segments and fallback_tail_gap is not None and gap_improve >= MAIN_HOTWORDS_FALLBACK_MIN_IMPROVE_SEC and fallback_last_end > raw_last_end + 1.0:
        print("[raw whisper fallback] using no-hotwords result; tail gap improved by", round(gap_improve, 3), "sec")
        raw_segments = fallback_segments
        info = fallback_info
        raw_audio_len = fallback_audio_len
        raw_last_end = fallback_last_end
        raw_tail_gap = fallback_tail_gap
        fallback_debug["fallback_run"] = True
        fallback_debug["fallback_used"] = True
        globals()["_asr_chunk_debug"] = fallback_debug
        globals()["_main_hotwords_fallback_used"] = True
    else:
        print("[raw whisper fallback] keeping original hotwords result")
        raw_asr_debug["fallback_run"] = True
        raw_asr_debug["fallback_used"] = False
        raw_asr_debug["fallback_candidate"] = fallback_debug
        globals()["_asr_chunk_debug"] = raw_asr_debug

print("[ASR debug]", json.dumps(globals().get("_asr_chunk_debug", {}), ensure_ascii=False)[:2000])


In [ ]:
#@title **Whisper rescue 辅助函数**
def clean_compact_text(s):
    return re.sub(r"\s+", "", (s or "").strip())

def repeated_ngram_score(s, min_n=2, max_n=5):
    s = clean_compact_text(s)
    if len(s) < min_n * 2:
        return 0.0
    best = 0.0
    upper = min(max_n, len(s) // 2)
    for n in range(min_n, upper + 1):
        grams = [s[i:i+n] for i in range(len(s) - n + 1)]
        if not grams:
            continue
        cnt = {}
        for g in grams:
            cnt[g] = cnt.get(g, 0) + 1
        top = max(cnt.values())
        score = (top * n) / max(1, len(s))
        best = max(best, score)
    return best

def _hotword_terms():
    return list(globals().get("HOTWORD_TERMS", []) or [])


def hotword_cover_ratio(text):
    txt = clean_compact_text(text)
    terms = _hotword_terms()
    if not txt or not terms:
        return 0.0
    covered = [False] * len(txt)
    for term in terms:
        if not term:
            continue
        pos = 0
        while True:
            idx = txt.find(term, pos)
            if idx < 0:
                break
            for j in range(idx, min(len(txt), idx + len(term))):
                covered[j] = True
            pos = idx + max(1, len(term))
    return sum(1 for x in covered if x) / max(1, len(txt))


def hotword_repeat_count(text):
    txt = clean_compact_text(text)
    best = 0
    for term in _hotword_terms():
        if not term:
            continue
        best = max(best, txt.count(term))
    return best


def is_hotword_hallucination_text(text, start=None, end=None, recent_texts=None, strict=False):
    if not bool(globals().get("HOTWORD_HALLUCINATION_ENABLE", True)):
        return False
    txt = clean_compact_text(text)
    if not txt or not _hotword_terms():
        return False

    cover = hotword_cover_ratio(txt)
    repeat_count = hotword_repeat_count(txt)
    repeated = repeated_ngram_score(txt)
    dur = None
    if start is not None and end is not None:
        dur = max(0.0, float(end) - float(start))
    cps = len(txt) / max(dur, 1e-6) if dur is not None else None

    cover_threshold = float(globals().get("HOTWORD_COVER_RATIO", 0.72))
    low_cps = float(globals().get("HOTWORD_LOW_CPS", 4.5))
    repeat_min = int(globals().get("HOTWORD_REPEAT_MIN", 2))

    if cover >= cover_threshold:
        if repeat_count >= repeat_min or repeated >= 0.55:
            return True
        if dur is not None and (dur >= 4.0 or (dur >= 2.0 and cps <= low_cps)):
            return True
        if strict and dur is not None and len(txt) <= 6 and dur >= 2.0 and cps <= low_cps:
            return True

    if cover >= 0.55 and dur is not None and dur >= 4.0 and len(txt) <= 18:
        return True

    if recent_texts and cover >= 0.55:
        for rt in recent_texts[-6:]:
            if text_similarity_loose(txt, rt) >= 0.86:
                return True

    return False


def extract_words(seg):
    whisper_words = []
    for w in (getattr(seg, "words", None) or []):
        ws = getattr(w, "start", None)
        we = getattr(w, "end", None)
        wt = getattr(w, "word", "")
        if ws is None or we is None:
            continue
        ws = float(ws)
        we = float(we)
        if we <= ws:
            continue
        wt = (wt or "").strip()
        if not wt:
            continue
        whisper_words.append({
            "text": wt,
            "start": ws,
            "end": we
        })
    return whisper_words

def is_suspect_whisper_segment(seg):
    start = float(seg.start)
    end = float(seg.end)
    text = (seg.text or "").strip()
    dur = end - start
    txt = re.sub(r"\s+", "", text)

    if not txt:
        return True

    cps = len(txt) / max(dur, 1e-6)

    # 长段但字很少
    if dur >= 2.5 and len(txt) <= 6:
        return True

    # 很长但信息密度极低
    if dur >= 2.0 and cps < 2.2:
        return True

    # 热词在静音 / BGM / 转场段容易被模型当作填充文本吐出。
    if is_hotword_hallucination_text(text, start, end):
        return True

    return False

# ===== 读整段音频，给 rescue 用 =====
wav, sr = sf.read(AUDIO_FILE)
if wav.ndim > 1:
    wav = wav.mean(axis=1)

def transcribe_clip(clip_wav, hotwords=None, no_speech_threshold=0.40, log_prob_threshold=-1.0):
    rescue_gen, _ = fw_model.transcribe(
        clip_wav,
        language=LANG,
        beam_size=max(WHISPER_BEAM_FLOOR, int(BEAM_SIZE)),
        best_of=max(WHISPER_BEAM_FLOOR, int(BEAM_SIZE)),
        patience=1.2,
        temperature=0.0,
        compression_ratio_threshold=2.4,
        log_prob_threshold=log_prob_threshold,
        no_speech_threshold=no_speech_threshold,
        vad_filter=False,
        condition_on_previous_text=False,
        initial_prompt=WHISPER_STYLE_PROMPT,
        hotwords=HOTWORDS if hotwords is None else hotwords,
        word_timestamps=True,
    )
    return list(rescue_gen)

def compact_text(s):
    return re.sub(r"\s+", "", (s or "").strip())

def text_similarity_loose(a, b):
    a = compact_text(a)
    b = compact_text(b)
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    # 简单包含关系
    short = a if len(a) <= len(b) else b
    long_ = b if len(a) <= len(b) else a
    if short and short in long_:
        return len(short) / max(len(long_), 1)
    # 字符集合近似
    sa, sb = set(a), set(b)
    inter = len(sa & sb)
    union = max(len(sa | sb), 1)
    return inter / union


def is_tail_hallucination(cand_text, recent_texts, min_sim=0.82):
    t = compact_text(cand_text)
    if not t:
        return True

    # 太短、套话、结尾常见废句，严格过滤
    bad_phrases = [
        "ご視聴ありがとうございました",
        "ご覧いただきありがとうございました",
        "最後までご視聴ありがとうございました",
    ]
    if t in [compact_text(x) for x in bad_phrases]:
        return True

    if is_hotword_hallucination_text(cand_text, recent_texts=recent_texts, strict=True):
        return True

    # 和最近几句高度重复，也过滤
    for rt in recent_texts[-4:]:
        if text_similarity_loose(t, rt) >= min_sim:
            return True

    return False

def get_audio_len_sec():
    return len(wav) / sr

def overlap_ratio(a0, a1, b0, b1):
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    union = max(a1, b1) - min(a0, b0)
    return inter / max(union, 1e-6)

def _recent_tail_compact_text(clean_segments, limit=4):
    return "".join(compact_text(x.get("text", "")) for x in (clean_segments or [])[-limit:])


def _longest_compact_seen_prefix_overlap(prev_texts, tail_text):
    prev_compact = "".join(compact_text(x) for x in (prev_texts or []))
    tail_compact = compact_text(tail_text)
    max_k = min(len(prev_compact), len(tail_compact))
    for k in range(max_k, 5, -1):
        if tail_compact[:k] in prev_compact:
            return k
    return 0

def _trim_tail_rescue_by_words(words, min_start):
    kept = [dict(w) for w in (words or []) if float(w.get("end", 0.0) or 0.0) > float(min_start)]
    if not kept:
        return [], ""
    return kept, "".join(str(w.get("text") or "") for w in kept).strip()

def _finalize_tail_rescue_candidate(clean_segments, last_end, rs_start, rs_end, rs_text, rs_words):
    recent_compact = _recent_tail_compact_text(clean_segments, limit=5)
    words = [dict(w) for w in (rs_words or [])]
    deduped = False
    crop_from = None

    if rs_start < float(last_end) - TAIL_RESCUE_STRICT_START_TOL:
        crop_from = float(last_end) - 0.02

    cand_compact = compact_text(rs_text)
    overlap_k = max(
        _longest_compact_prefix_overlap([x.get("text", "") for x in (clean_segments or [])[-5:]], rs_text),
        _longest_compact_seen_prefix_overlap([x.get("text", "") for x in (clean_segments or [])[-8:]], rs_text),
    )
    if overlap_k >= 6:
        deduped = True
        if words:
            crop_from = max(float(crop_from or -1e9), float(last_end) - 0.02)
        else:
            rs_text = _slice_text_by_compact_prefix(rs_text, overlap_k)
            cand_compact = compact_text(rs_text)

    if cand_compact and recent_compact and text_similarity_loose(cand_compact, recent_compact[-max(8, min(len(recent_compact), len(cand_compact) * 2)):]) >= TAIL_RESCUE_DEDUP_SIM:
        deduped = True
        crop_from = max(float(crop_from or -1e9), float(last_end) - 0.02)

    if crop_from is not None and words:
        words, rs_text = _trim_tail_rescue_by_words(words, crop_from)
        if words:
            rs_start = max(float(last_end) - 0.02, float(words[0]["start"]))
            rs_end = max(rs_start, float(words[-1]["end"]))
            deduped = True

    rs_text = (rs_text or "").strip()
    if not rs_text:
        return None, deduped
    if rs_end - rs_start < MIN_SEG_DUR:
        return None, deduped
    if compact_text(rs_text) and recent_compact and text_similarity_loose(compact_text(rs_text), recent_compact[-max(8, min(len(recent_compact), len(compact_text(rs_text)) * 2)):]) >= 0.90:
        return None, True

    return {
        "start": rs_start,
        "end": rs_end,
        "text": rs_text,
        "whisper_words": words,
        "is_tail_rescue": True,
    }, deduped

def diagnose_tail_dropout(last_end, audio_len):
    gap = float(audio_len) - float(last_end)
    if gap <= 0.80:
        return

    clip_l = max(0.0, min(float(last_end) - 4.0, float(audio_len) - 28.0))
    clip_r = float(audio_len)
    if clip_r - clip_l < 2.0:
        return

    clip_wav = wav[int(clip_l * sr):int(clip_r * sr)]
    print(f"[tail diag] clip=({clip_l:.3f}, {clip_r:.3f})")

    trials = [
        ("same_hotwords", {"hotwords": HOTWORDS, "no_speech_threshold": 0.3, "log_prob_threshold": -1.0}),
        ("same_no_hotwords", {"hotwords": "", "no_speech_threshold": 0.3, "log_prob_threshold": -1.0}),
        ("relaxed_no_hotwords", {"hotwords": "", "no_speech_threshold": 0.6, "log_prob_threshold": -1.2}),
    ]
    first_hit = None

    for label, kwargs in trials:
        try:
            segs = transcribe_clip(clip_wav, **kwargs)
        except Exception as e:
            print("[tail diag]", label, "error:", e)
            continue

        joined = "".join((x.text or "").strip() for x in segs)
        joined = re.sub(r"\s+", "", joined)
        rel_last_end = float(segs[-1].end) if segs else 0.0
        print("[tail diag]", label, "segments=", len(segs), "rel_last_end=", round(rel_last_end, 3), "chars=", len(joined))
        if segs and first_hit is None:
            first_hit = label

    if first_hit == "same_no_hotwords":
        print("[tail diag] likely cause: hotwords suppressed tail decoding")
    elif first_hit == "relaxed_no_hotwords":
        print("[tail diag] likely cause: tail was filtered by no_speech/log_prob gating or dependency drift")
    elif first_hit is None:
        print("[tail diag] whisper still misses the tail even on isolated clip; this points to model/runtime drift more than post-processing")

def append_tail_rescue(clean_segments):
    globals()["_tail_rescue_deduped"] = 0
    globals()["_tail_rescue_no_new_tail"] = False
    globals()["_tail_rescue_near_last_end"] = False
    if not TAIL_RESCUE_ENABLE or not clean_segments:
        return clean_segments

    audio_len = get_audio_len_sec()
    last_seg = clean_segments[-1]
    last_end = float(last_seg["end"])
    tail_missing = audio_len - last_end

    print(f"[tail rescue] audio_len={audio_len:.3f} last_end={last_end:.3f} missing={tail_missing:.3f}")

    # 末尾缺太少就不救
    if tail_missing <= 0.80:
        return clean_segments

    lookback = min(TAIL_RESCUE_MAX_LOOKBACK, max(TAIL_RESCUE_SEC, tail_missing + 6.0))
    clip_l = max(0.0, audio_len - lookback - TAIL_RESCUE_LEFT_PAD)
    clip_r = audio_len
    print(f"[tail rescue] clip=({clip_l:.3f}, {clip_r:.3f})")

    clip_wav = wav[int(clip_l * sr):int(clip_r * sr)]

    try:
        rescue_segments = transcribe_clip(clip_wav)
    except Exception as e:
        print("tail rescue failed:", e)
        return clean_segments

    def _tail_segments_hotword_suspect(segs):
        return any(
            is_hotword_hallucination_text(
                (x.text or "").strip(),
                float(x.start) + clip_l,
                float(x.end) + clip_l,
                recent_texts=[seg.get("text", "") for seg in clean_segments[-8:]],
                strict=True,
            )
            for x in (segs or [])
        )

    if hotwords_compact and (not rescue_segments or _tail_segments_hotword_suspect(rescue_segments)):
        reason = "empty" if not rescue_segments else "hotword hallucination"
        print(f"[tail rescue] retry without hotwords ({reason})")
        no_hotword_segments = transcribe_clip(clip_wav, hotwords="")
        if no_hotword_segments:
            rescue_segments = no_hotword_segments
            print("[tail rescue diagnosis] using no-hotwords tail result")

    if not rescue_segments:
        print("[tail rescue] no segments")
        diagnose_tail_dropout(last_end, audio_len)
        return clean_segments

    recent_texts = [seg.get("text", "") for seg in clean_segments[-5:]]
    accepted = []
    deduped_n = 0
    near_last_end_seen = False

    for rs in rescue_segments:
        rs_start = float(rs.start) + clip_l
        rs_end = float(rs.end) + clip_l
        rs_text = (rs.text or "").strip()
        rs_words = [
            {
                "text": (getattr(w, "word", "") or "").strip(),
                "start": float(getattr(w, "start")) + clip_l,
                "end": float(getattr(w, "end")) + clip_l,
            }
            for w in (getattr(rs, "words", None) or [])
            if getattr(w, "start", None) is not None
            and getattr(w, "end", None) is not None
            and float(getattr(w, "end")) > float(getattr(w, "start"))
            and (getattr(w, "word", "") or "").strip()
        ]

        print("[tail raw]", round(rs_start, 3), "->", round(rs_end, 3), repr(rs_text))
        if rs_end >= last_end - 0.35:
            near_last_end_seen = True

        if not rs_text:
            continue
        if rs_end - rs_start < MIN_SEG_DUR:
            continue

        # 必须明显落在现有最后一句之后，才考虑
        if rs_end <= last_end + 0.20:
            continue

        # 尾段 hallucination / 热词填充过滤
        if is_tail_hallucination(rs_text, recent_texts):
            print("[tail reject hallucination]", repr(rs_text))
            continue
        if is_hotword_hallucination_text(rs_text, rs_start, rs_end, recent_texts=recent_texts, strict=True):
            print("[tail reject hotword hallucination]", repr(rs_text))
            deduped_n += 1
            continue

        cand, deduped = _finalize_tail_rescue_candidate(clean_segments, last_end, rs_start, rs_end, rs_text, rs_words)
        if deduped:
            deduped_n += 1
        if not cand:
            continue

        accepted.append({
            "id": -1,
            "orig_index": 10**9,
            "start": cand["start"],
            "end": cand["end"],
            "text": cand["text"],
            "whisper_words": cand["whisper_words"],
            "is_tail_rescue": True,
            "tail_rescue_source": "whisper_tail_rescue",
        })

    if not accepted:
        print("[tail rescue] nothing accepted")
        globals()["_tail_rescue_deduped"] = deduped_n
        globals()["_tail_rescue_near_last_end"] = bool(near_last_end_seen)
        globals()["_tail_rescue_no_new_tail"] = bool(near_last_end_seen)
        return clean_segments

    # 只追加真正新的尾句
    for seg in accepted:
        seg["id"] = len(clean_segments)
        clean_segments.append(seg)

    clean_segments.sort(key=lambda x: (x["start"], x["end"]))
    for i, seg in enumerate(clean_segments):
        seg["id"] = i

    globals()["_tail_rescue_deduped"] = deduped_n
    print("[tail rescue] accepted:", len(accepted))
    return clean_segments

LANG_NAME = "Japanese" if LANG == "ja" else "Chinese"

#qwen尾段转写


In [ ]:
#@title **Whisper tail supplement 与分段工具**
def qwen_tail_transcribe(seg_wav):
    # seg_wav: numpy array, SR: 采样率
    model = ensure_qwen_asr_model()
    results = model.transcribe(
        audio=[(seg_wav, SR)],
        language=[LANG_NAME],
    )
    return (results[0].text or "").strip()

def align_tail_text(seg_wav, text, clip_start):
    tokens = []
    try:
        result = aligner.align(
            audio=[(seg_wav, SR)],
            text=[text],
            language=[LANG_NAME]
        )
        if result and len(result) > 0:
            for tk in result[0]:
                s = getattr(tk,"start_time",None) or getattr(tk,"start",None)
                e = getattr(tk,"end_time",None) or getattr(tk,"end",None)

                if s is None or e is None:
                    continue

                s = float(s) + clip_start
                e = float(e) + clip_start

                if e <= s:
                    continue

                tokens.append({
                    "text":tk.text,
                    "start":s,
                    "end":e
                })
    except Exception as e:
        print("tail align error:",e)

    return tokens

def _longest_compact_prefix_overlap(prev_texts, tail_text):
    prev_compact = "".join(compact_text(x) for x in (prev_texts or []))
    tail_compact = compact_text(tail_text)
    max_k = min(len(prev_compact), len(tail_compact))
    for k in range(max_k, 5, -1):
        prefix = tail_compact[:k]
        if prev_compact.endswith(prefix) or prefix in prev_compact:
            return k
    return 0


def _recent_tail_context_items(clean_segments, last_end, window_sec=30.0, limit=16):
    start_cut = max(0.0, float(last_end) - float(window_sec))
    rows = [x for x in (clean_segments or []) if float(x.get("end", 0.0) or 0.0) >= start_cut]
    rows = sorted(rows, key=lambda x: (float(x.get("start", 0.0) or 0.0), float(x.get("end", 0.0) or 0.0)))
    return rows[-int(limit):]


def _merge_tail_token_text(tokens):
    return "".join(str(t.get("text") or "").strip() for t in (tokens or [])).strip()


def _slice_text_by_compact_prefix(text, prefix_len):
    if prefix_len <= 0:
        return (text or "").strip()
    count = 0
    out = []
    for ch in (text or ""):
        if ch.isspace():
            if count >= prefix_len:
                out.append(ch)
            continue
        count += 1
        if count > prefix_len:
            out.append(ch)
    return "".join(out).strip()


def supplement_tail_with_qwen(clean_segments, audio_len):
    if not TAIL_SUPPLEMENT_ENABLE or not clean_segments:
        return clean_segments

    last_end = float(clean_segments[-1]["end"])
    missing = float(audio_len) - last_end
    if missing <= TAIL_SUPPLEMENT_TRIGGER_SEC:
        return clean_segments
    if bool(globals().get("_tail_rescue_no_new_tail", False)):
        print("[tail supplement] skipped; whisper tail rescue reached last_end with no new tail")
        return clean_segments

    clip_l = max(0.0, min(last_end - TAIL_SUPPLEMENT_LEFT_CONTEXT, float(audio_len) - TAIL_SUPPLEMENT_LOOKBACK))
    clip_r = float(audio_len)
    if clip_r - clip_l < 1.0:
        return clean_segments

    seg_wav = wav[int(clip_l * SR):int(clip_r * SR)]
    tail_text = qwen_tail_transcribe(seg_wav)
    if not tail_text:
        print("[tail supplement] empty")
        return clean_segments

    recent_items = _recent_tail_context_items(clean_segments, last_end, window_sec=30.0, limit=16)
    recent = [x.get("text", "") for x in recent_items]
    overlap_k = _longest_compact_prefix_overlap(recent, tail_text)
    novel_text = _slice_text_by_compact_prefix(tail_text, overlap_k)
    novel_compact = compact_text(novel_text)

    print("[tail supplement] missing:", round(missing, 3), "tail_text:", tail_text)
    print("[tail supplement] overlap_k:", overlap_k, "novel:", novel_text)

    if len(novel_compact) < TAIL_SUPPLEMENT_MIN_NEW_CHARS:
        print("[tail supplement] no novel chars")
        return clean_segments
    if is_tail_hallucination(novel_text, recent):
        print("[tail supplement] hallucination rejected")
        return clean_segments
    if is_hotword_hallucination_text(novel_text, recent_texts=recent, strict=True):
        print("[tail supplement] hotword hallucination rejected")
        return clean_segments

    tokens = align_tail_text(seg_wav, novel_text, clip_l)
    tokens = [t for t in tokens if float(t["end"]) > last_end + 0.10]
    if not tokens:
        print("[tail supplement] align produced no new tail tokens")
        return clean_segments

    token_text = _merge_tail_token_text(tokens)
    token_compact = compact_text(token_text)
    if len(token_compact) < TAIL_SUPPLEMENT_MIN_NEW_CHARS:
        print("[tail supplement] no novel token chars")
        return clean_segments
    if is_tail_hallucination(token_text, recent):
        print("[tail supplement] token text hallucination rejected")
        return clean_segments
    if is_hotword_hallucination_text(token_text, recent_texts=recent, strict=True):
        print("[tail supplement] token text hotword hallucination rejected")
        return clean_segments

    token_start = max(last_end + 0.03, float(tokens[0]["start"]))
    token_end = float(tokens[-1]["end"])
    token_dur = max(0.0, token_end - token_start)
    token_cps = len(token_compact) / max(token_dur, 0.01)
    max_read_cps = float(globals().get("MAX_READ_CPS", 17.0))
    if token_dur <= 3.0 and len(token_compact) > 45:
        print("[tail supplement] short tail has too much text; rejected")
        return clean_segments
    if token_cps > max(max_read_cps * 1.8, 22.0):
        print("[tail supplement] token text too dense; rejected")
        return clean_segments

    if bool(globals().get("_tail_rescue_no_new_tail", False)):
        if any(text_similarity_loose(token_text, rt) >= 0.84 for rt in recent[-8:]):
            print("[tail supplement] rescue already covered tail; duplicate token text rejected")
            return clean_segments
        if token_dur <= 2.0 and len(token_compact) <= 8:
            print("[tail supplement] rescue already covered tail; tiny token text rejected")
            return clean_segments

    new_seg = {
        "id": 10**9,
        "orig_index": 10**9,
        "start": token_start,
        "end": token_end,
        "text": token_text,
        "whisper_words": tokens,
        "is_tail_rescue": True,
        "tail_rescue_source": "qwen_tail_supplement",
    }
    if new_seg["end"] - new_seg["start"] < MIN_SEG_DUR:
        print("[tail supplement] new seg too short")
        return clean_segments

    clean_segments.append(new_seg)
    clean_segments.sort(key=lambda x: (x["start"], x["end"]))
    for i, seg in enumerate(clean_segments):
        seg["id"] = i
    print("[tail supplement] appended")
    return clean_segments


def split_long_segment(seg):

    words = seg["whisper_words"]
    text = seg["text"]
    seg_meta = {k: v for k, v in seg.items() if k not in {"id", "start", "end", "text", "whisper_words"}}

    if not words:
        return [seg]

    dur = seg["end"] - seg["start"]

    if dur < MAX_SENT_DUR:
        return [seg]

    chunks = []
    cur_words = []

    for w in words:

        cur_words.append(w)

        chunk_start = cur_words[0]["start"]
        chunk_end = cur_words[-1]["end"]

        chunk_text = "".join(x["text"] for x in cur_words)

        dur = chunk_end - chunk_start

        if (
            dur > MAX_SENT_DUR
            or len(chunk_text) > MAX_CHARS_PER_CHUNK
            or (w["text"] in SPLIT_PUNCT and dur > MIN_SENT_DUR)
        ):

            chunks.append({
                "start":chunk_start,
                "end":chunk_end,
                "text":chunk_text,
                "whisper_words":cur_words.copy()
            } | seg_meta)

            cur_words = []

    if cur_words:

        chunks.append({
            "start":cur_words[0]["start"],
            "end":cur_words[-1]["end"],
            "text":"".join(x["text"] for x in cur_words),
            "whisper_words":cur_words
        } | seg_meta)

    return chunks



In [ ]:
#@title **clean_segments 构建与尾补执行**
clean_segments = []
audio_len = len(wav)/SR
globals()["_hotword_hallucination_dropped"] = 0
globals()["_hotword_rescue_no_hotwords_used"] = 0

for i, s in enumerate(raw_segments):
    start = float(s.start)
    end = float(s.end)
    text = (s.text or "").strip()
    dur = end - start

    if not text:
        continue
    if dur < MIN_SEG_DUR:
        continue

    final_seg = s
    hotword_suspect = is_hotword_hallucination_text(text, start, end, strict=True)

    # ===== 对疑似塌缩段做局部 rescue =====
    if is_suspect_whisper_segment(s):
        clip_l = max(0.0, start - 0.35)
        clip_r = min(len(wav) / sr, end + 0.35)
        clip_wav = wav[int(clip_l * sr):int(clip_r * sr)]

        try:
            rescue_segments = transcribe_clip(clip_wav, hotwords="" if hotword_suspect else None)
            if hotword_suspect:
                globals()["_hotword_rescue_no_hotwords_used"] += 1

            # 只在 rescue 明显更丰富时才替换；热词幻觉段如果救不回来就丢弃。
            raw_text_c = re.sub(r"\s+", "", text)
            valid_rescue = []
            for rs in rescue_segments:
                rs_start = float(rs.start) + clip_l
                rs_end = float(rs.end) + clip_l
                rs_text = (rs.text or "").strip()
                if not rs_text:
                    continue
                if rs_end - rs_start < MIN_SEG_DUR:
                    continue
                if is_hotword_hallucination_text(rs_text, rs_start, rs_end, strict=True):
                    continue
                valid_rescue.append((rs, rs_start, rs_end, rs_text))

            rescue_text = "".join(x[3] for x in valid_rescue)
            rescue_text_c = re.sub(r"\s+", "", rescue_text)

            rescue_is_better = len(valid_rescue) >= 2 and len(rescue_text_c) >= len(raw_text_c) + 3
            rescue_salvages_hotword = hotword_suspect and len(valid_rescue) >= 1 and len(rescue_text_c) >= 2
            if rescue_is_better or rescue_salvages_hotword:
                for rs, rs_start, rs_end, rs_text in valid_rescue:
                    clean_segments.append({
                        "id": len(clean_segments),
                        "orig_index": i,
                        "start": rs_start,
                        "end": rs_end,
                        "text": rs_text,
                        "whisper_words": [
                            {
                                "text": (getattr(w, "word", "") or "").strip(),
                                "start": float(getattr(w, "start")) + clip_l,
                                "end": float(getattr(w, "end")) + clip_l,
                            }
                            for w in (getattr(rs, "words", None) or [])
                            if getattr(w, "start", None) is not None
                            and getattr(w, "end", None) is not None
                            and float(getattr(w, "end")) > float(getattr(w, "start"))
                            and (getattr(w, "word", "") or "").strip()
                        ]
                    })
                continue

            if hotword_suspect:
                globals()["_hotword_hallucination_dropped"] += 1
                continue
        except Exception as e:
            print("local rescue failed:", e)
            if hotword_suspect:
                globals()["_hotword_hallucination_dropped"] += 1
                continue

    whisper_words = extract_words(final_seg)

    clean_segments.append({
        "id": len(clean_segments),
        "orig_index": i,
        "start": start,
        "end": end,
        "text": text,
        "whisper_words": whisper_words
    })

## 先常规排序
clean_segments.sort(key=lambda x: (x["start"], x["end"]))

# 末尾补救
clean_segments = append_tail_rescue(clean_segments)

if clean_segments:
    last_end = clean_segments[-1]["end"]
    missing = audio_len - last_end
    print("tail missing:",round(missing,3))
    clean_segments = supplement_tail_with_qwen(clean_segments, audio_len)

new_segments = []

for seg in clean_segments:
    parts = split_long_segment(seg)
    for p in parts:
        new_segments.append(p)

clean_segments = new_segments
clean_segments.sort(key=lambda x:(x["start"],x["end"]))

for i,s in enumerate(clean_segments):
    s["id"]=i

print("after split:",len(clean_segments))
print("clean segments:", len(clean_segments))

if clean_segments:
    audio_len = len(wav) / sr
    print("last seg end:", round(float(clean_segments[-1]["end"]), 3), "/", round(audio_len, 3))

with open("fw_clean_segments.json", "w", encoding="utf-8") as f:
    json.dump(clean_segments, f, ensure_ascii=False, indent=2)


### Qwen3 Forced Aligner 对齐


In [ ]:
#@title **Qwen3 Forced Aligner 主对齐**
# Cell 5: Qwen3ForcedAligner per-segment alignment
# Fallback order: qwen_tokens -> whisper_words_fallback -> segment_fallback
import soundfile as sf
import torch
import gc
import re
from collections import Counter

wav, sr = sf.read(AUDIO_FILE)

if wav.ndim > 1:
    wav = wav.mean(axis=1)

audio_len = len(wav) / sr

# ===== free VRAM first =====
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Before load VRAM free: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

alignments = []

def _sanitize_word_tokens(words):
    toks = []
    for w in words or []:
        t = (w.get("text", "") or "").strip()
        s = w.get("start", None)
        e = w.get("end", None)
        if s is None or e is None:
            continue
        s = float(s)
        e = float(e)
        if e <= s:
            continue
        if not t:
            continue
        toks.append({"text": t, "start": s, "end": e})
    return toks

def compact_for_match_local(s):
    return re.sub(r"\s+", "", (s or "").strip())

def token_text_coverage_ratio(orig_text, tokens):
    a = compact_for_match_local(orig_text)
    b = compact_for_match_local("".join(t.get("text", "") for t in (tokens or [])))
    if not a or not b:
        return 0.0
    return min(len(b), len(a)) / max(1, len(a))

def repeated_ngram_score(s, min_n=2, max_n=5):
    s = compact_for_match_local(s)
    if len(s) < min_n * 2:
        return 0.0

    best = 0.0
    upper = min(max_n, len(s) // 2)
    for n in range(min_n, upper + 1):
        grams = [s[i:i+n] for i in range(len(s) - n + 1)]
        if not grams:
            continue
        cnt = Counter(grams)
        top = cnt.most_common(1)[0][1]
        score = (top * n) / max(1, len(s))
        best = max(best, score)
    return best

def repetition_collapse_risk(text, tokens):
    src = compact_for_match_local(text)
    tok_list = [
        compact_for_match_local(t.get("text", ""))
        for t in (tokens or [])
        if compact_for_match_local(t.get("text", ""))
    ]

    if not src or not tok_list:
        return True

    src_rep = repeated_ngram_score(src, 2, 5)
    tok_rep = repeated_ngram_score("".join(tok_list), 2, 5)

    token_count = len(tok_list)
    uniq_ratio = len(set(tok_list)) / max(1, token_count)
    tok_text_len = len("".join(tok_list))
    src_len = len(src)

    if src_len >= 10 and src_rep >= 0.38 and token_count <= 2:
        return True

    if src_len >= 10 and src_rep >= 0.35 and uniq_ratio < 0.45 and token_count <= 4:
        return True

    if src_rep >= 0.40 and tok_text_len < max(4, int(src_len * 0.35)):
        return True

    if tok_rep >= 0.60 and token_count <= 3 and src_len >= 8:
        return True

    return False

def _alignment_early_start_tolerance(text, orig_start, orig_end, seg_index, seg_meta=None):
    txt = compact_for_match_local(text)
    dur = max(0.0, float(orig_end) - float(orig_start))
    is_tail = bool((seg_meta or {}).get("is_tail_rescue"))
    tol = 0.18
    if dur <= 1.2 or len(txt) <= 10:
        tol = min(tol, 0.12)
    if seg_index == 0:
        tol = min(tol, 0.10)
    if is_tail:
        tol = min(tol, 0.06)
    return tol

def is_alignment_sane(text, tokens, orig_start, orig_end, seg_index, seg_meta=None):
    if not tokens:
        return False

    txt = compact_for_match_local(text)
    if not txt:
        return False

    span = max(0.0, float(tokens[-1]["end"]) - float(tokens[0]["start"]))
    cps = len(txt) / max(span, 1e-6)

    if span <= 0.10:
        return False
    if cps > 22.0:
        return False

    early_tol = _alignment_early_start_tolerance(text, orig_start, orig_end, seg_index, seg_meta=seg_meta)
    if float(tokens[0]["start"]) < float(orig_start) - early_tol:
        return False
    if float(tokens[-1]["end"]) > float(orig_end) + 1.50:
        return False

    if seg_index == 0:
        if span < 0.45 and len(txt) >= 8:
            return False
        if cps > 18.0:
            return False

    # 通用重复塌缩拦截：防止重复喊话 / 叠词被压成极少 token
    if repetition_collapse_risk(text, tokens):
        return False

    return True


for seg_idx, seg in enumerate(clean_segments):

    text = seg["text"]

    start = seg["start"]
    end = seg["end"]

    clip_start = max(0, start - PAD_LEFT)
    clip_end = min(audio_len, end + pad_right(text))

    seg_wav = wav[int(clip_start * SR):int(clip_end * SR)]

    tokens = []
    timing_source = "segment_fallback"

    if len(seg_wav) >= int(SR * MIN_CLIP_SEC):

        try:

            result = aligner.align(
                audio=[(seg_wav, SR)],
                text=[text],
                language=[LANG_NAME]
            )

            if result and len(result) > 0:

                for tk in result[0]:

                    s = getattr(tk, "start_time", None) or getattr(tk, "start", None)
                    e = getattr(tk, "end_time", None) or getattr(tk, "end", None)

                    if s is None or e is None:
                        continue

                    s = float(s) + clip_start
                    e = float(e) + clip_start

                    if e <= s:
                        continue

                    tokens.append({
                        "text": tk.text,
                        "start": s,
                        "end": e
                    })

                if tokens:
                    cov = token_text_coverage_ratio(text, tokens)
                    sane = is_alignment_sane(text, tokens, start, end, seg_idx, seg_meta=seg)
                    if cov >= 0.80 and sane:
                        timing_source = "qwen_tokens"
                    else:
                        tokens = []

        except Exception as e:
            print("align fail:", e)

    if (not tokens) and bool(globals().get("ENABLE_WORD_LEVEL_FALLBACK", True)):
        fallback_tokens = _sanitize_word_tokens(seg.get("whisper_words", []))
        if fallback_tokens:
            tokens = fallback_tokens
            timing_source = "whisper_words_fallback"

    alignments.append({
        "text": text,
        "orig_start": start,
        "orig_end": end,
        "tokens": tokens,
        "whisper_words": seg.get("whisper_words", []) or [],
        "is_tail_rescue": bool(seg.get("is_tail_rescue", False)),
        "tail_rescue_source": seg.get("tail_rescue_source"),
        "timing_source": timing_source
    })

print(Counter(x["timing_source"] for x in alignments))

In [ ]:
#@title **`sentence_items` 生成与基础诊断**
sentence_items = []

for seg in alignments:

    tokens = seg["tokens"]

    if tokens:
        start = tokens[0]["start"]
        end = tokens[-1]["end"]
    else:
        start = seg["orig_start"]
        end = seg["orig_end"]

    sentence_items.append({
        "start": start,
        "end": end,
        "text": seg["text"],   # keep original Whisper text
        "tokens": tokens,
        "whisper_words": seg.get("whisper_words", []) or [],
        "speaker": None,
        "is_tail_rescue": bool(seg.get("is_tail_rescue", False)),
        "tail_rescue_source": seg.get("tail_rescue_source"),
        "timing_source": seg.get("timing_source", "segment_fallback")
    })

print("sentence_items:", len(sentence_items))


def _compact_sentence_hall_text(s):
    s = (s or "").strip()
    s = re.sub(r"\s+", "", s)
    s = s.replace("。", "").replace("！", "").replace("？", "")
    s = s.replace("!", "").replace("?", "")
    return s


def is_sentence_hallucination(text, start, end, recent_items, audio_len):
    t = _compact_sentence_hall_text(text)
    if not t:
        return True

    bad_phrases = [
        "ご視聴ありがとうございました",
        "ご覧いただきありがとうございました",
        "最後までご視聴ありがとうございました",
    ]
    if any(bp in t for bp in bad_phrases):
        if float(start) < float(audio_len) - 6.0:
            return True

    dur = max(0.0, float(end) - float(start))
    if is_hotword_hallucination_text(text, start, end, recent_texts=[x.get("text", "") for x in recent_items[-6:]], strict=True):
        return True
    if dur < 2.2:
        recent_texts = [it.get("text", "") for it in recent_items[-5:]]
        for rt in recent_texts:
            if text_similarity_loose(t, _compact_sentence_hall_text(rt)) >= 0.84:
                return True

    return False


def drop_hallucinated_sentence_items(items, audio_len):
    out = []
    dropped = 0
    for it in items:
        if is_sentence_hallucination(it.get("text", ""), it.get("start", 0.0), it.get("end", 0.0), out, audio_len):
            dropped += 1
            continue
        out.append(it)
    globals()["_dropped_sentence_hallucination"] = dropped
    return out


sentence_items = drop_hallucinated_sentence_items(sentence_items, audio_len)
print("sentence_items after hallucination filter:", len(sentence_items), "dropped:", int(globals().get("_dropped_sentence_hallucination", 0)))


### 说话人识别 / NeMo diarization


In [ ]:
#@title **根据说话人数自动选择字幕参数**
# ===== Auto subtitle profile by diarization UI =====

AUTO_MODE_BY_SPEAKERS = True

def _clamp_int(v, lo, hi):
    return max(lo, min(hi, int(v)))

def get_runtime_speaker_config():
    """
    直接读取 Cell 7 写入的全局变量，不依赖 Cell 12。
    这样不会因为 get_diar_runtime_config() 尚未定义而退回 2 人。
    """
    scenario = str(globals().get("DIAR_SCENARIO", "auto_route") or "auto_route").strip()
    speaker_mode = str(globals().get("DIAR_SPEAKER_MODE", "auto") or "auto").strip()

    fixed_spk = _clamp_int(globals().get("DIAR_FIXED_SPK", 2), 1, 10)
    min_spk = _clamp_int(globals().get("DIAR_MIN_SPK", 1), 1, 10)
    max_spk = _clamp_int(globals().get("DIAR_MAX_SPK", 4), 1, 10)

    valid_scenarios = {"single_low_overlap", "multi_overlap", "auto_route"}
    valid_modes = {"auto", "fixed", "range"}

    if scenario not in valid_scenarios:
        scenario = "auto_route"
    if speaker_mode not in valid_modes:
        speaker_mode = "auto"

    if min_spk > max_spk:
        min_spk, max_spk = max_spk, min_spk

    if speaker_mode == "fixed":
        min_spk = fixed_spk
        max_spk = fixed_spk

    return {
        "scenario": scenario,
        "speaker_mode": speaker_mode,
        "fixed_spk": fixed_spk if speaker_mode == "fixed" else None,
        "min_spk": min_spk,
        "max_spk": max_spk,
    }


def get_runtime_speaker_count():
    cfg = get_runtime_speaker_config()
    scenario = cfg["scenario"]
    mode = cfg["speaker_mode"]

    # 用户手动固定人数 -> 必须优先
    if mode == "fixed":
        return int(cfg["fixed_spk"] or 2)

    # 用户手动范围 -> 用上限决定 profile
    if mode == "range":
        if scenario == "single_low_overlap":
            return min(int(cfg["max_spk"]), 2)
        return int(cfg["max_spk"])

    # auto 模式才根据场景给默认值
    if scenario == "single_low_overlap":
        return 2
    if scenario == "multi_overlap":
        return max(3, int(cfg["max_spk"] or 4))

    return int(cfg["max_spk"] or 2)


def get_subtitle_profile(num_speakers: int):
    n = max(1, int(num_speakers or 1))

    # 1-2 人：普通访谈 / 对谈
    if n <= 2:
        return {
            "PROFILE_NAME": "duo",
            "ALLOW_CROSS_SPEAKER_OVERLAP": False,
            "ENABLE_MULTI_TRACK_ASS": False,
            "ENABLE_GLOBAL_CONFLICT_RESOLVE": True,

            "MAX_CHARS_PER_CHUNK": 34,
            "MIN_CHARS_PER_CHUNK": 8,
            "LONG_PAUSE_SEC": 0.42,
            "SHORT_PAUSE_SEC": 0.22,
            "MIN_SPLIT_DUR": 1.00,

            "MIN_SENT_DUR": 0.82,
            "MAX_SENT_DUR": 8.50,

            "MIN_MERGE_DUR": 1.65,
            "MAX_GAP_MERGE": 0.22,
            "MAX_MERGE_DUR": 8.80,

            "MICRO_SEG_SEC": 0.92,
            "MIN_FINAL_DUR": 0.82,
            "MIN_FINAL_DUR_STRICT": 0.84,

            "START_PAD": 0.18,
            "MAX_START_ADVANCE": 0.30,
            "END_PAD": 0.22,
            "MAX_END_EXTEND": 1.35,

            "ENABLE_SPEAKER_AWARE_SPLIT": False,
            "POSTHOC_SPEAKER_ASSIGN_ONLY": True,
        }

    # 3-4 人：多人对话 / 抢话 / 游戏场景
    if n <= 4:
        return {
            "PROFILE_NAME": "multi4",
            "ALLOW_CROSS_SPEAKER_OVERLAP": True,
            "ENABLE_MULTI_TRACK_ASS": True,
            "ENABLE_GLOBAL_CONFLICT_RESOLVE": False,

            "MAX_CHARS_PER_CHUNK": 16,
            "MIN_CHARS_PER_CHUNK": 3,
            "LONG_PAUSE_SEC": 0.22,
            "SHORT_PAUSE_SEC": 0.10,
            "MIN_SPLIT_DUR": 0.28,

            "MIN_SENT_DUR": 0.40,
            "MAX_SENT_DUR": 3.00,

            "MIN_MERGE_DUR": 0.60,
            "MAX_GAP_MERGE": 0.08,
            "MAX_MERGE_DUR": 2.20,

            "MICRO_SEG_SEC": 0.45,
            "MIN_FINAL_DUR": 0.40,
            "MIN_FINAL_DUR_STRICT": 0.40,

            "START_PAD": 0.08,
            "MAX_START_ADVANCE": 0.14,
            "END_PAD": 0.08,
            "MAX_END_EXTEND": 0.35,

            "ENABLE_SPEAKER_AWARE_SPLIT": False,
            "POSTHOC_SPEAKER_ASSIGN_ONLY": True,
        }

    # 5 人以上：高并发混战
    return {
        "PROFILE_NAME": "crowded",
        "ALLOW_CROSS_SPEAKER_OVERLAP": True,
        "ENABLE_MULTI_TRACK_ASS": True,
        "ENABLE_GLOBAL_CONFLICT_RESOLVE": False,

        "MAX_CHARS_PER_CHUNK": 12,
        "MIN_CHARS_PER_CHUNK": 2,
        "LONG_PAUSE_SEC": 0.16,
        "SHORT_PAUSE_SEC": 0.06,
        "MIN_SPLIT_DUR": 0.20,

        "MIN_SENT_DUR": 0.32,
        "MAX_SENT_DUR": 2.20,

        "MIN_MERGE_DUR": 0.40,
        "MAX_GAP_MERGE": 0.05,
        "MAX_MERGE_DUR": 1.60,

        "MICRO_SEG_SEC": 0.36,
        "MIN_FINAL_DUR": 0.34,
        "MIN_FINAL_DUR_STRICT": 0.34,

        "START_PAD": 0.06,
        "MAX_START_ADVANCE": 0.10,
        "END_PAD": 0.06,
        "MAX_END_EXTEND": 0.22,

        "ENABLE_SPEAKER_AWARE_SPLIT": False,
        "POSTHOC_SPEAKER_ASSIGN_ONLY": True,
    }


def apply_subtitle_profile(profile: dict):
    globals().update(profile)
    print(f"[SubtitleProfile] {profile['PROFILE_NAME']}")
    for k in [
        "ALLOW_CROSS_SPEAKER_OVERLAP",
        "ENABLE_MULTI_TRACK_ASS",
        "ENABLE_GLOBAL_CONFLICT_RESOLVE",
        "MAX_CHARS_PER_CHUNK",
        "LONG_PAUSE_SEC",
        "SHORT_PAUSE_SEC",
        "MIN_SENT_DUR",
        "MAX_SENT_DUR",
        "MIN_MERGE_DUR",
        "MAX_GAP_MERGE",
        "START_PAD",
        "END_PAD",
    ]:
        print(f"  {k} = {globals()[k]}")


if AUTO_MODE_BY_SPEAKERS:
    runtime_cfg = get_runtime_speaker_config()
    print("[AutoSpeakerProfile] runtime config =", runtime_cfg)

    RUNTIME_NUM_SPEAKERS = get_runtime_speaker_count()
    print(f"[AutoSpeakerProfile] runtime speakers = {RUNTIME_NUM_SPEAKERS}")

    SUB_PROFILE = get_subtitle_profile(RUNTIME_NUM_SPEAKERS)
    apply_subtitle_profile(SUB_PROFILE)

In [ ]:
#@title **diarization 运行时配置与前置检查**
import json

if "_clamp_int" not in globals():
    def _clamp_int(v, lo, hi):
        return max(lo, min(hi, int(v)))

if "get_diar_runtime_config" not in globals():
    def get_diar_runtime_config() -> dict:
        scenario = globals().get("DIAR_SCENARIO", "auto_route")
        speaker_mode = globals().get("DIAR_SPEAKER_MODE", "auto")
        fixed_spk = _clamp_int(globals().get("DIAR_FIXED_SPK", 2), 1, 10)
        min_spk = _clamp_int(globals().get("DIAR_MIN_SPK", 1), 1, 10)
        max_spk = _clamp_int(globals().get("DIAR_MAX_SPK", 4), 1, 10)

        valid_scenarios = {"single_low_overlap", "multi_overlap", "auto_route"}
        valid_modes = {"auto", "fixed", "range"}

        if scenario not in valid_scenarios:
            scenario = "auto_route"
        if speaker_mode not in valid_modes:
            speaker_mode = "auto"

        if min_spk > max_spk:
            min_spk, max_spk = max_spk, min_spk

        if speaker_mode == "fixed":
            min_spk = fixed_spk
            max_spk = fixed_spk

        return {
            "scenario": scenario,
            "speaker_mode": speaker_mode,
            "fixed_spk": fixed_spk if speaker_mode == "fixed" else None,
            "min_spk": min_spk,
            "max_spk": max_spk,
            "enable_diagnostics": bool(globals().get("ENABLE_DIAR_DIAGNOSTICS", True)),
        }


ENABLE_DIARIZATION = bool(globals().get("ENABLE_DIARIZATION", True))
if not ENABLE_DIARIZATION:
    print("已关闭 diarization，跳过说话人识别。")
    with open("sentence_items_with_speaker.json", "w", encoding="utf-8") as f:
        json.dump(sentence_items, f, ensure_ascii=False, indent=2)

if ENABLE_DIARIZATION:
    !pip install -q -c {COLAB_CONSTRAINTS} "nemo_toolkit[asr]" soundfile ffmpeg-python wget omegaconf

    import gc
    import glob
    import math
    import os
    import shutil
    import subprocess
    from collections import Counter, defaultdict
    from typing import Any, Dict, List, Tuple

    import soundfile as sf
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print("GPU:", torch.cuda.get_device_name(0))
        print(f"Before load VRAM free: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

    assert "AUDIO_FILE" in globals(), "AUDIO_FILE is missing"
    assert "sentence_items" in globals(), "sentence_items is missing; run Qwen alignment cell first"

    ENABLE_DIARIZATION = bool(globals().get("ENABLE_DIARIZATION", True))
    ENABLE_DIAR_DIAGNOSTICS = bool(globals().get("ENABLE_DIAR_DIAGNOSTICS", True))

    DIAR_LONG_AUDIO_SEC = max(60.0, float(globals().get("DIAR_LONG_AUDIO_SEC", 1080)))
    DIAR_CHUNK_SEC = max(120.0, float(globals().get("DIAR_CHUNK_SEC", 720)))
    DIAR_CHUNK_OVERLAP_SEC = max(5.0, float(globals().get("DIAR_CHUNK_OVERLAP_SEC", 25)))
    DIAR_BOUNDARY_SEARCH_SEC = max(0.0, float(globals().get("DIAR_BOUNDARY_SEARCH_SEC", 20)))
    DIAR_CHUNK_MIN_SEC = max(60.0, float(globals().get("DIAR_CHUNK_MIN_SEC", 180)))

    DIAR_WORKDIR = "nemo_diar_work"
    DIAR_OUTDIR = os.path.join(DIAR_WORKDIR, "out")
    MANIFEST_PATH = os.path.join(DIAR_WORKDIR, "manifest.json")

    os.makedirs(DIAR_WORKDIR, exist_ok=True)
    os.makedirs(DIAR_OUTDIR, exist_ok=True)

    runtime_cfg = get_diar_runtime_config()
    print("[Diarization] Runtime config:", json.dumps(runtime_cfg, ensure_ascii=False))

    shared_audio_source = globals().get("SHARED_AUDIO_SOURCE")
    if shared_audio_source and os.path.exists(shared_audio_source):
        diar_wav = shared_audio_source
        print("[Diarization] using shared audio source:", diar_wav)
    else:
        diar_wav = os.path.join(DIAR_WORKDIR, "audio_16k_mono.wav")
        subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", AUDIO_FILE,
                "-ac", "1",
                "-ar", "16000",
                diar_wav,
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

    audio_info = sf.info(diar_wav)
    audio_duration = float(audio_info.frames) / float(audio_info.samplerate)
    globals()["AUDIO_LEN"] = max(float(globals().get("AUDIO_LEN", 0.0) or 0.0), audio_duration)
    globals()["audio_len"] = globals()["AUDIO_LEN"]
    globals()["DIAR_AUDIO_DURATION"] = audio_duration
    globals()["USE_CHUNKED_DIARIZATION"] = bool(audio_duration > DIAR_LONG_AUDIO_SEC)
    shared_audio_chunks = globals().get("SHARED_AUDIO_CHUNKS", []) or []

    diar_debug_payload = {
        "audio_duration": round(audio_duration, 3),
        "chunked": bool(audio_duration > DIAR_LONG_AUDIO_SEC),
        "shared_chunk_plan_used": bool(shared_audio_chunks),
        "shared_chunk_count": len(shared_audio_chunks),
        "runtime_cfg": dict(runtime_cfg),
        "params": {
            "long_audio_sec": DIAR_LONG_AUDIO_SEC,
            "chunk_sec": DIAR_CHUNK_SEC,
            "chunk_overlap_sec": DIAR_CHUNK_OVERLAP_SEC,
            "boundary_search_sec": DIAR_BOUNDARY_SEARCH_SEC,
            "chunk_min_sec": DIAR_CHUNK_MIN_SEC,
        },
        "warnings": [],
        "chunks": [],
    }
    globals()["diar_debug_payload"] = diar_debug_payload

    def _append_diar_warning(msg: str):
        print(f"[Diar][WARN] {msg}")
        diar_debug_payload.setdefault("warnings", []).append(str(msg))


    def _write_manifest(path: str, audio_filepath: str, duration: float, num_speakers: int = None) -> Dict[str, Any]:
        manifest_item = {
            "audio_filepath": os.path.abspath(audio_filepath),
            "offset": 0,
            "duration": max(0.0, float(duration)),
            "label": "infer",
            "text": "-",
            "num_speakers": num_speakers,
            "rttm_filepath": None,
            "uem_filepath": None,
        }
        with open(path, "w", encoding="utf-8") as f:
            f.write(json.dumps(manifest_item, ensure_ascii=False) + "\n")
        return manifest_item


    _write_manifest(MANIFEST_PATH, diar_wav, audio_duration, runtime_cfg["fixed_spk"])

    import wget
    from omegaconf import OmegaConf, open_dict
    from nemo.collections.asr.models import ClusteringDiarizer

    device_str = "cuda" if torch.cuda.is_available() else "cpu"

    cfg_path = os.path.join(DIAR_WORKDIR, "diar_infer_meeting.yaml")
    if not os.path.exists(cfg_path):
        wget.download(
            "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/speaker_tasks/diarization/conf/inference/diar_infer_meeting.yaml",
            cfg_path,
        )


    def _load_neural_diarizer_cls():
        try:
            from nemo.collections.asr.models.msdd_models import NeuralDiarizer
            return NeuralDiarizer
        except Exception:
            try:
                from nemo.collections.asr.models import NeuralDiarizer
                return NeuralDiarizer
            except Exception:
                return None


    def _safe_set(cfg_node, key, value):
        try:
            setattr(cfg_node, key, value)
        except Exception:
            try:
                cfg_node[key] = value
            except Exception:
                pass


    def _ensure_msdd_cfg(cfg):
        if "msdd_model" in cfg.diarizer:
            msdd_cfg = cfg.diarizer.msdd_model
        else:
            with open_dict(cfg.diarizer):
                cfg.diarizer.msdd_model = OmegaConf.create({})
            msdd_cfg = cfg.diarizer.msdd_model
            _append_diar_warning("NeMo config missing diarizer.msdd_model; injected MSDD compatibility defaults")

        if "parameters" not in msdd_cfg or msdd_cfg.parameters is None:
            with open_dict(msdd_cfg):
                msdd_cfg.parameters = OmegaConf.create({})

        defaults = {
            "infer_batch_size": 25,
            "sigmoid_threshold": [0.7],
            "seq_eval_mode": False,
            "split_infer": True,
            "diar_window_length": 50,
            "overlap_infer_spk_limit": 5,
        }
        with open_dict(msdd_cfg.parameters):
            for key, value in defaults.items():
                if key not in msdd_cfg.parameters:
                    msdd_cfg.parameters[key] = value
        return msdd_cfg


    def _reset_dir(path: str):
        if os.path.isdir(path):
            shutil.rmtree(path)
        os.makedirs(path, exist_ok=True)


    def _apply_common_cfg(cfg, out_dir: str, runtime_cfg: Dict[str, Any], use_msdd: bool, manifest_path: str):
        _safe_set(cfg, "device", device_str)

        cfg.diarizer.manifest_filepath = os.path.abspath(manifest_path)
        cfg.diarizer.out_dir = os.path.abspath(out_dir)
        cfg.diarizer.oracle_vad = False

        cfg.diarizer.vad.model_path = "vad_multilingual_marblenet"
        cfg.diarizer.vad.parameters.onset = 0.55
        cfg.diarizer.vad.parameters.offset = 0.45
        cfg.diarizer.vad.parameters.pad_offset = -0.05
        cfg.diarizer.vad.parameters.pad_onset = 0.05
        cfg.diarizer.vad.parameters.min_duration_on = 0.2
        cfg.diarizer.vad.parameters.min_duration_off = 0.15
        cfg.diarizer.vad.parameters.filter_speech_first = True

        cfg.diarizer.speaker_embeddings.model_path = "titanet_large"
        cfg.diarizer.speaker_embeddings.parameters.window_length_in_sec = [1.5, 1.25, 1.0, 0.75]
        cfg.diarizer.speaker_embeddings.parameters.shift_length_in_sec = [0.75, 0.625, 0.5, 0.375]
        cfg.diarizer.speaker_embeddings.parameters.multiscale_weights = [1.0, 1.0, 1.0, 1.0]
        cfg.diarizer.speaker_embeddings.parameters.save_embeddings = bool(use_msdd)

        clustering_params = cfg.diarizer.clustering.parameters
        fixed_spk = runtime_cfg["fixed_spk"]
        clustering_params.oracle_num_speakers = fixed_spk is not None
        clustering_params.max_num_speakers = int(fixed_spk if fixed_spk is not None else runtime_cfg["max_spk"])

        if "min_num_speakers" in clustering_params:
            clustering_params.min_num_speakers = int(fixed_spk if fixed_spk is not None else runtime_cfg["min_spk"])

        if "enhanced_count_thres" in clustering_params:
            clustering_params.enhanced_count_thres = 120.0

        if use_msdd:
            msdd_cfg = _ensure_msdd_cfg(cfg)
            msdd_cfg.model_path = "diar_msdd_telephonic"
            msdd_cfg.parameters.sigmoid_threshold = [0.7]


In [ ]:
#@title **## diarization 执行与 speaker assignment**

if ENABLE_DIARIZATION:
    def _pick_rttm_file(out_dir: str, wav_path: str) -> str:
        pred_dir = os.path.join(out_dir, "pred_rttms")
        rttm_files = glob.glob(os.path.join(pred_dir, "*.rttm"))
        if not rttm_files:
            rttm_files = glob.glob(os.path.join(out_dir, "**", "*.rttm"), recursive=True)
        assert rttm_files, f"No RTTM output found in: {out_dir}"

        wav_base = os.path.splitext(os.path.basename(wav_path))[0]
        exact = [p for p in rttm_files if os.path.splitext(os.path.basename(p))[0] == wav_base]
        if exact:
            return sorted(exact, key=lambda p: os.path.getmtime(p), reverse=True)[0]
        return sorted(rttm_files, key=lambda p: os.path.getmtime(p), reverse=True)[0]


    def _read_rttm_segments(
        rttm_path: str,
        offset_sec: float = 0.0,
        chunk_index: int = None,
        pass_name: str = None,
    ) -> List[Dict[str, Any]]:
        segs = []
        with open(rttm_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 9 or parts[0] != "SPEAKER":
                    continue
                start = float(parts[3]) + float(offset_sec)
                dur = float(parts[4])
                spk = parts[7]
                segs.append(
                    {
                        "start": start,
                        "end": start + dur,
                        "speaker": spk,
                        "chunk_index": chunk_index,
                        "pass_name": pass_name,
                    }
                )
        segs.sort(key=lambda x: (x["start"], x["end"]))
        return segs


    def _calc_rttm_risk(segments: List[Dict[str, Any]]) -> Dict[str, Any]:
        if not segments:
            return {
                "high_risk": False,
                "switch_rate": 0.0,
                "short_ratio": 0.0,
                "dominant_ratio": 0.0,
                "reason": "empty_rttm",
            }

        switch_n = 0
        for i in range(1, len(segments)):
            if segments[i]["speaker"] != segments[i - 1]["speaker"]:
                switch_n += 1

        switch_rate = switch_n / max(1, len(segments) - 1)
        short_ratio = sum(1 for x in segments if (x["end"] - x["start"]) < 1.0) / len(segments)
        spk_cnt = Counter(x["speaker"] for x in segments)
        dominant_ratio = (max(spk_cnt.values()) / len(segments)) if spk_cnt else 0.0

        high_risk = (
            switch_rate >= 0.55
            or (switch_rate >= 0.42 and short_ratio >= 0.38)
            or (short_ratio >= 0.62 and dominant_ratio <= 0.85)
        )

        return {
            "high_risk": bool(high_risk),
            "switch_rate": round(switch_rate, 4),
            "short_ratio": round(short_ratio, 4),
            "dominant_ratio": round(dominant_ratio, 4),
            "reason": "frequent_switch_or_short_segments" if high_risk else "ok",
        }


    def _collect_gap_candidates(items: List[Dict[str, Any]], audio_len: float) -> List[Dict[str, Any]]:
        points = []
        prev_end = 0.0
        for item in sorted(items or [], key=lambda x: (float(x.get("start", 0.0)), float(x.get("end", 0.0)))):
            start = float(item.get("start", 0.0))
            end = float(item.get("end", start))
            gap = start - prev_end
            if gap >= 0.45:
                points.append({"time": prev_end + gap / 2.0, "gap": gap})
            prev_end = max(prev_end, end)
        tail_gap = max(0.0, float(audio_len) - prev_end)
        if tail_gap >= 0.45:
            points.append({"time": prev_end + tail_gap / 2.0, "gap": tail_gap})
        return points


    def _snap_chunk_boundary(base_start: float, target_end: float, audio_len: float, gap_candidates: List[Dict[str, Any]]) -> float:
        if target_end >= audio_len:
            return float(audio_len)

        lo = max(base_start + DIAR_CHUNK_MIN_SEC, target_end - DIAR_BOUNDARY_SEARCH_SEC)
        hi = min(audio_len, target_end + DIAR_BOUNDARY_SEARCH_SEC)
        choices = []
        for cand in gap_candidates:
            t = float(cand["time"])
            if lo <= t <= hi:
                choices.append((abs(t - target_end), -float(cand["gap"]), t))
        if choices:
            return float(sorted(choices)[0][2])
        return float(min(audio_len, max(base_start + DIAR_CHUNK_MIN_SEC, target_end)))


    def _plan_diarization_chunks(audio_len: float, items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        shared_chunks = globals().get("SHARED_AUDIO_CHUNKS", []) or []
        if audio_len > DIAR_LONG_AUDIO_SEC and len(shared_chunks) > 1:
            planned = []
            for chunk in shared_chunks:
                planned.append({
                    "index": int(chunk["index"]),
                    "base_start": float(chunk["base_start"]),
                    "base_end": float(chunk["base_end"]),
                    "extract_start": float(chunk["extract_start"]),
                    "extract_end": float(chunk["extract_end"]),
                    "overlap_start": float(chunk.get("overlap_start", 0.0)),
                    "overlap_end": float(chunk.get("overlap_end", 0.0)),
                    "wav_path": chunk.get("wav_path"),
                })
            diar_debug_payload["shared_chunk_plan_used"] = True
            return planned

        if audio_len <= DIAR_LONG_AUDIO_SEC:
            return [
                {
                    "index": 0,
                    "base_start": 0.0,
                    "base_end": float(audio_len),
                    "extract_start": 0.0,
                    "extract_end": float(audio_len),
                    "overlap_start": 0.0,
                    "overlap_end": 0.0,
                }
            ]

        gap_candidates = _collect_gap_candidates(items, audio_len)
        chunks = []
        base_start = 0.0
        idx = 0
        while base_start < audio_len - 1e-6:
            target_end = min(audio_len, base_start + DIAR_CHUNK_SEC)
            base_end = _snap_chunk_boundary(base_start, target_end, audio_len, gap_candidates)
            if audio_len - base_end < DIAR_CHUNK_MIN_SEC:
                base_end = float(audio_len)
            if base_end <= base_start:
                base_end = float(min(audio_len, base_start + DIAR_CHUNK_SEC))
            extract_start = 0.0 if idx == 0 else max(0.0, base_start - DIAR_CHUNK_OVERLAP_SEC)
            chunks.append(
                {
                    "index": idx,
                    "base_start": round(base_start, 3),
                    "base_end": round(base_end, 3),
                    "extract_start": round(extract_start, 3),
                    "extract_end": round(base_end, 3),
                    "overlap_start": round(extract_start if idx > 0 else 0.0, 3),
                    "overlap_end": round(base_start if idx > 0 else 0.0, 3),
                }
            )
            base_start = base_end
            idx += 1

        return chunks


    def _extract_chunk_wav(source_wav: str, chunk: Dict[str, Any]) -> str:
        chunk_wav = os.path.join(DIAR_WORKDIR, f"chunk_{int(chunk['index']):03d}.wav")
        dur = max(0.0, float(chunk["extract_end"]) - float(chunk["extract_start"]))
        subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", source_wav,
                "-ss", f"{float(chunk['extract_start']):.3f}",
                "-t", f"{dur:.3f}",
                "-ac", "1",
                "-ar", "16000",
                chunk_wav,
            ],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        return chunk_wav


    def _run_diarization_pass(
        pass_name: str,
        runtime_cfg: Dict[str, Any],
        use_msdd: bool,
        wav_path: str,
        manifest_path: str,
        offset_sec: float = 0.0,
        chunk_index: int = None,
    ) -> Dict[str, Any]:
        pass_out = os.path.join(DIAR_OUTDIR, pass_name)
        _reset_dir(pass_out)

        cfg = OmegaConf.load(cfg_path)
        _apply_common_cfg(cfg, pass_out, runtime_cfg, use_msdd=use_msdd, manifest_path=manifest_path)

        print(f"[Diar] pass={pass_name}, use_msdd={use_msdd}, out={pass_out}")

        if use_msdd:
            NeuralDiarizer = _load_neural_diarizer_cls()
            if NeuralDiarizer is not None:
                diarizer_obj = NeuralDiarizer(cfg=cfg)
            else:
                _append_diar_warning("NeuralDiarizer unavailable, fallback to ClusteringDiarizer")
                diarizer_obj = ClusteringDiarizer(cfg=cfg)
        else:
            diarizer_obj = ClusteringDiarizer(cfg=cfg)

        if hasattr(diarizer_obj, "to"):
            try:
                diarizer_obj = diarizer_obj.to(device_str)
            except Exception:
                pass

        diarizer_obj.diarize()

        rttm_path = _pick_rttm_file(pass_out, wav_path)
        segs = _read_rttm_segments(
            rttm_path,
            offset_sec=offset_sec,
            chunk_index=chunk_index,
            pass_name=pass_name,
        )

        print("RTTM:", rttm_path)
        print(f"speaker_segments: {len(segs)}")

        return {
            "pass_name": pass_name,
            "use_msdd": use_msdd,
            "out_dir": pass_out,
            "rttm_path": rttm_path,
            "segments": segs,
        }


    def _run_msdd_with_fallback(
        pass_name: str,
        runtime_cfg: Dict[str, Any],
        fallback_name: str,
        wav_path: str,
        manifest_path: str,
        offset_sec: float = 0.0,
        chunk_index: int = None,
    ):
        try:
            return _run_diarization_pass(
                pass_name,
                runtime_cfg,
                use_msdd=True,
                wav_path=wav_path,
                manifest_path=manifest_path,
                offset_sec=offset_sec,
                chunk_index=chunk_index,
            ), {
                "used_msdd": True,
                "fallback": False,
                "error": None,
            }
        except Exception as e:
            print(f"[Diar][WARN] MSDD failed, fallback to clustering. reason: {type(e).__name__}: {e}")
            fallback = _run_diarization_pass(
                fallback_name,
                runtime_cfg,
                use_msdd=False,
                wav_path=wav_path,
                manifest_path=manifest_path,
                offset_sec=offset_sec,
                chunk_index=chunk_index,
            )
            return fallback, {
                "used_msdd": False,
                "fallback": True,
                "error": f"{type(e).__name__}: {e}",
            }


    def _run_clip_diarization(
        clip_tag: str,
        runtime_cfg: Dict[str, Any],
        wav_path: str,
        manifest_path: str,
        offset_sec: float = 0.0,
        chunk_index: int = None,
    ):
        if runtime_cfg["scenario"] == "single_low_overlap":
            diar_result = _run_diarization_pass(
                f"{clip_tag}_single",
                runtime_cfg,
                use_msdd=False,
                wav_path=wav_path,
                manifest_path=manifest_path,
                offset_sec=offset_sec,
                chunk_index=chunk_index,
            )
            diar_meta = {"routing": "single_low_overlap", "rerun": False, "used_msdd": False}
            return diar_result, diar_meta

        if runtime_cfg["scenario"] == "multi_overlap":
            diar_result, msdd_meta = _run_msdd_with_fallback(
                f"{clip_tag}_multi_msdd",
                runtime_cfg,
                f"{clip_tag}_multi_clustering_fallback",
                wav_path=wav_path,
                manifest_path=manifest_path,
                offset_sec=offset_sec,
                chunk_index=chunk_index,
            )
            diar_meta = {"routing": "multi_overlap", "rerun": False, **msdd_meta}
            return diar_result, diar_meta

        first = _run_diarization_pass(
            f"{clip_tag}_auto_clustering",
            runtime_cfg,
            use_msdd=False,
            wav_path=wav_path,
            manifest_path=manifest_path,
            offset_sec=offset_sec,
            chunk_index=chunk_index,
        )
        risk = _calc_rttm_risk(first["segments"])
        print("[Diar][AutoRisk]", json.dumps(risk, ensure_ascii=False))

        if risk["high_risk"]:
            second, msdd_meta = _run_msdd_with_fallback(
                f"{clip_tag}_auto_msdd",
                runtime_cfg,
                f"{clip_tag}_auto_msdd_fallback_clustering",
                wav_path=wav_path,
                manifest_path=manifest_path,
                offset_sec=offset_sec,
                chunk_index=chunk_index,
            )
            diar_meta = {
                "routing": "auto_route",
                "rerun": True,
                "reason": risk,
                "first_pass": first["rttm_path"],
                "final_pass": second["rttm_path"],
                **msdd_meta,
            }
            return second, diar_meta

        diar_meta = {
            "routing": "auto_route",
            "rerun": False,
            "reason": risk,
            "final_pass": first["rttm_path"],
            "used_msdd": False,
            "fallback": False,
            "error": None,
        }
        return first, diar_meta


    def _max_global_speakers(runtime_cfg: Dict[str, Any]) -> int:
        fixed = runtime_cfg.get("fixed_spk")
        if fixed is not None:
            return int(fixed)
        return int(runtime_cfg.get("max_spk") or 4)


    def _chunk_local_speaker_stats(chunk_segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        totals = defaultdict(float)
        counts = Counter()
        for seg in chunk_segments or []:
            spk = seg.get("speaker")
            if not spk:
                continue
            dur = max(0.0, float(seg.get("end", 0.0)) - float(seg.get("start", 0.0)))
            if dur <= 0:
                continue
            totals[spk] += dur
            counts[spk] += 1
        total = sum(totals.values())
        rows = []
        for spk, dur in sorted(totals.items(), key=lambda kv: (-kv[1], kv[0])):
            rows.append({
                "speaker": spk,
                "duration": round(dur, 3),
                "share": round(dur / max(total, 1e-6), 4),
                "segments": int(counts[spk]),
            })
        return rows


    def _effective_local_speakers(local_stats: List[Dict[str, Any]]) -> List[str]:
        if not local_stats:
            return []
        min_dur = float(globals().get("DIAR_STITCH_MIN_LOCAL_DUR_SEC", 2.0))
        min_share = float(globals().get("DIAR_STITCH_MIN_LOCAL_SHARE", 0.003))
        keep_dur = float(globals().get("DIAR_STITCH_KEEP_LOCAL_DUR_SEC", 8.0))
        labels = [
            row["speaker"]
            for row in local_stats
            if (float(row.get("duration", 0.0)) >= min_dur and float(row.get("share", 0.0)) >= min_share)
            or float(row.get("duration", 0.0)) >= keep_dur
        ]
        if not labels:
            labels = [local_stats[0]["speaker"]]
        return labels


    def _map_chunk_speakers(
        existing_segments: List[Dict[str, Any]],
        chunk_segments: List[Dict[str, Any]],
        chunk: Dict[str, Any],
        runtime_cfg: Dict[str, Any],
        next_global_idx: int,
    ):
        if not chunk_segments:
            return [], next_global_idx, []

        local_stats = _chunk_local_speaker_stats(chunk_segments)
        local_labels = [row["speaker"] for row in local_stats]
        effective_labels = _effective_local_speakers(local_stats)
        effective_label_set = set(effective_labels)
        mapping = {}
        stitch_records = []
        min_overlap_sec = float(globals().get("DIAR_STITCH_MIN_OVERLAP_SEC", 0.75))
        min_overlap_share = float(globals().get("DIAR_STITCH_MIN_OVERLAP_SHARE", 0.30))

        if not existing_segments:
            for local in effective_labels:
                mapping[local] = (f"speaker_{next_global_idx}", 1.0, "initial_chunk")
                next_global_idx += 1
        else:
            ov_start = float(chunk.get("overlap_start", 0.0))
            ov_end = float(chunk.get("overlap_end", 0.0))
            existing_recent = [seg for seg in existing_segments if float(seg["end"]) > ov_start and float(seg["start"]) < ov_end]
            current_recent = [seg for seg in chunk_segments if seg.get("speaker") in effective_label_set and float(seg["end"]) > ov_start and float(seg["start"]) < ov_end]

            pair_scores = []
            local_totals = defaultdict(float)
            scores_by_local = defaultdict(dict)
            for cur in current_recent:
                local = cur["speaker"]
                cur_ov = max(0.0, min(float(cur["end"]), ov_end) - max(float(cur["start"]), ov_start))
                if cur_ov <= 0:
                    continue
                local_totals[local] += cur_ov
                for prev in existing_recent:
                    inter = max(
                        0.0,
                        min(float(cur["end"]), float(prev["end"]), ov_end) - max(float(cur["start"]), float(prev["start"]), ov_start),
                    )
                    if inter <= 0:
                        continue
                    global_spk = prev["speaker"]
                    scores_by_local[local][global_spk] = scores_by_local[local].get(global_spk, 0.0) + inter

            for local, bucket in scores_by_local.items():
                for global_spk, inter in bucket.items():
                    pair_scores.append((inter, local, global_spk))

            used_globals = set()
            for inter, local, global_spk in sorted(pair_scores, reverse=True):
                if local in mapping or global_spk in used_globals:
                    continue
                local_total = local_totals.get(local, 0.0)
                share = inter / max(local_total, 1e-6)
                if inter >= min_overlap_sec and share >= min_overlap_share:
                    mapping[local] = (global_spk, round(share, 4), "overlap_match")
                    used_globals.add(global_spk)

            global_cap = _max_global_speakers(runtime_cfg)
            for local in effective_labels:
                if local in mapping:
                    continue
                local_bucket = scores_by_local.get(local, {})
                if local_bucket:
                    best_global, best_inter = max(local_bucket.items(), key=lambda kv: kv[1])
                    local_total = local_totals.get(local, 0.0)
                    share = best_inter / max(local_total, 1e-6)
                    if best_inter >= min_overlap_sec and share >= min_overlap_share:
                        mapping[local] = (best_global, round(share, 4), "reuse_best_existing_confident")
                        continue
                source = "new_speaker" if next_global_idx < global_cap else "new_speaker_overflow_no_confident_reuse"
                mapping[local] = (f"speaker_{next_global_idx}", 0.0, source)
                next_global_idx += 1

        dominant_effective = next((local for local in local_labels if local in mapping), None)
        if dominant_effective is not None:
            dominant_label = mapping[dominant_effective][0]
            for local in local_labels:
                if local not in mapping:
                    mapping[local] = (dominant_label, 0.0, "local_noise_absorbed")

        mapped_segments = []
        boundary_cut = float(chunk["base_start"])
        for seg in chunk_segments:
            if float(chunk["index"]) > 0 and float(seg["end"]) <= boundary_cut:
                continue
            new_seg = dict(seg)
            label, conf, source = mapping.get(seg["speaker"], (seg["speaker"], 0.0, "identity"))
            new_seg["speaker"] = label
            new_seg["chunk_index"] = int(chunk["index"])
            new_seg["stitch_confidence"] = float(conf)
            new_seg["stitch_source"] = source
            if float(chunk["index"]) > 0 and float(new_seg["start"]) < boundary_cut:
                new_seg["start"] = boundary_cut
                if float(new_seg["end"]) <= float(new_seg["start"]):
                    continue
            mapped_segments.append(new_seg)

        stats_by_local = {row["speaker"]: row for row in local_stats}
        for local, (label, conf, source) in sorted(mapping.items()):
            stat = stats_by_local.get(local, {})
            stitch_records.append(
                {
                    "local_speaker": local,
                    "global_speaker": label,
                    "confidence": conf,
                    "source": source,
                    "duration": stat.get("duration", 0.0),
                    "share": stat.get("share", 0.0),
                    "segments": stat.get("segments", 0),
                    "effective": bool(local in effective_label_set),
                }
            )

        return mapped_segments, next_global_idx, stitch_records


    def _dedupe_and_merge_speaker_segments(segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        out = []
        for seg in sorted(segments or [], key=lambda x: (float(x["start"]), float(x["end"]))):
            start = float(seg["start"])
            end = float(seg["end"])
            if end <= start:
                continue
            cur = dict(seg)
            cur["start"] = start
            cur["end"] = end
            if out:
                prev = out[-1]
                gap = float(cur["start"]) - float(prev["end"])
                if prev.get("speaker") == cur.get("speaker") and gap <= 0.08:
                    prev["end"] = max(float(prev["end"]), float(cur["end"]))
                    prev["stitch_confidence"] = max(
                        float(prev.get("stitch_confidence", 0.0) or 0.0),
                        float(cur.get("stitch_confidence", 0.0) or 0.0),
                    )
                    prev["stitch_source"] = prev.get("stitch_source") or cur.get("stitch_source")
                    continue
            out.append(cur)
        return out


    def _run_chunked_diarization(runtime_cfg: Dict[str, Any]):
        chunk_plan = _plan_diarization_chunks(audio_duration, sentence_items)
        diar_debug_payload["chunked"] = True
        diar_debug_payload["chunks"] = []
        diar_debug_payload["chunk_plan"] = chunk_plan

        combined_segments = []
        next_global_idx = 0
        success_n = 0
        last_rttm_path = None

        for chunk in chunk_plan:
            record = dict(chunk)
            try:
                chunk_wav = chunk.get("wav_path")
                used_shared_wav = bool(chunk_wav and os.path.exists(chunk_wav))
                if not used_shared_wav:
                    chunk_wav = _extract_chunk_wav(diar_wav, chunk)
                chunk_manifest = os.path.join(DIAR_WORKDIR, f"manifest_chunk_{int(chunk['index']):03d}.json")
                _write_manifest(
                    chunk_manifest,
                    chunk_wav,
                    float(chunk["extract_end"]) - float(chunk["extract_start"]),
                    runtime_cfg["fixed_spk"],
                )
                chunk_result, clip_meta = _run_clip_diarization(
                    f"chunk_{int(chunk['index']):03d}",
                    runtime_cfg,
                    chunk_wav,
                    chunk_manifest,
                    offset_sec=float(chunk["extract_start"]),
                    chunk_index=int(chunk["index"]),
                )
                mapped_segments, next_global_idx, stitch_records = _map_chunk_speakers(
                    combined_segments,
                    chunk_result["segments"],
                    chunk,
                    runtime_cfg,
                    next_global_idx,
                )
                combined_segments = _dedupe_and_merge_speaker_segments(combined_segments + mapped_segments)
                success_n += 1
                last_rttm_path = chunk_result.get("rttm_path")
                record.update(
                    {
                        "status": "ok",
                        "manifest_path": chunk_manifest,
                        "wav_path": chunk_wav,
                        "shared_wav": bool(used_shared_wav),
                        "segment_count": len(chunk_result["segments"]),
                        "mapped_segment_count": len(mapped_segments),
                        "used_msdd": bool(clip_meta.get("used_msdd")),
                        "fallback": bool(clip_meta.get("fallback")),
                        "risk": clip_meta.get("reason"),
                        "rttm_path": chunk_result.get("rttm_path"),
                        "stitch_records": stitch_records,
                        "local_speaker_stats": [
                            {
                                "local_speaker": x.get("local_speaker"),
                                "duration": x.get("duration", 0.0),
                                "share": x.get("share", 0.0),
                                "segments": x.get("segments", 0),
                                "effective": bool(x.get("effective")),
                            }
                            for x in stitch_records
                        ],
                        "effective_local_speakers": [x.get("local_speaker") for x in stitch_records if x.get("effective")],
                        "low_conf_stitch_count": sum(
                            1
                            for x in stitch_records
                            if x.get("source") in {"local_noise_absorbed"}
                            or (str(x.get("source", "")).startswith("reuse_") and float(x.get("confidence", 0.0) or 0.0) < 0.30)
                        ),
                    }
                )
            except Exception as e:
                msg = f"chunk {int(chunk['index'])} failed: {type(e).__name__}: {e}"
                _append_diar_warning(msg)
                record.update({"status": "error", "error": f"{type(e).__name__}: {e}"})
            diar_debug_payload["chunks"].append(record)

        if success_n == 0:
            raise RuntimeError("all diarization chunks failed")

        diar_meta = {
            "routing": "chunked_auto_route",
            "chunked": True,
            "chunks_total": len(chunk_plan),
            "chunks_succeeded": success_n,
            "chunks_failed": len(chunk_plan) - success_n,
            "final_pass": last_rttm_path,
        }
        return {
            "pass_name": "chunked_auto_route",
            "use_msdd": any(bool(x.get("used_msdd")) for x in diar_debug_payload["chunks"]),
            "out_dir": DIAR_OUTDIR,
            "rttm_path": last_rttm_path,
            "segments": combined_segments,
        }, diar_meta


    try:
        if audio_duration > DIAR_LONG_AUDIO_SEC:
            diar_result, diar_meta = _run_chunked_diarization(runtime_cfg)
        else:
            diar_result, diar_meta = _run_clip_diarization(
                "full",
                runtime_cfg,
                diar_wav,
                MANIFEST_PATH,
                offset_sec=0.0,
                chunk_index=0,
            )
            diar_meta["chunked"] = False
    except Exception as e:
        _append_diar_warning(f"diarization failed, fallback to no-speaker mode. {type(e).__name__}: {e}")
        diar_result = {
            "pass_name": "failed_no_speaker_fallback",
            "use_msdd": False,
            "out_dir": None,
            "rttm_path": None,
            "segments": [],
        }
        diar_meta = {
            "routing": "failed_no_speaker_fallback",
            "chunked": bool(audio_duration > DIAR_LONG_AUDIO_SEC),
            "error": f"{type(e).__name__}: {e}",
        }

    diar_debug_payload["final_meta"] = diar_meta
    speaker_segments = diar_result["segments"]

    def clean_speaker_segments_for_assignment(speaker_segments: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        cleaned = []
        for seg in sorted(speaker_segments or [], key=lambda x: (float(x["start"]), float(x["end"]))):
            start = float(seg["start"])
            end = float(seg["end"])
            spk = seg.get("speaker")
            if spk is None or end <= start:
                continue
            cleaned.append(
                {
                    "start": start,
                    "end": end,
                    "speaker": spk,
                    "unstable": bool(seg.get("unstable", False)),
                    "chunk_index": seg.get("chunk_index"),
                    "stitch_confidence": float(seg.get("stitch_confidence", 0.0) or 0.0),
                    "stitch_source": seg.get("stitch_source"),
                    "pass_name": seg.get("pass_name"),
                }
            )

        if not cleaned:
            return []

        merged = [dict(cleaned[0])]
        for seg in cleaned[1:]:
            prev = merged[-1]
            gap = seg["start"] - prev["end"]
            if prev["speaker"] == seg["speaker"] and gap <= 0.08:
                prev["end"] = max(prev["end"], seg["end"])
                prev["unstable"] = bool(prev.get("unstable", False) or seg.get("unstable", False))
                prev["stitch_confidence"] = max(
                    float(prev.get("stitch_confidence", 0.0) or 0.0),
                    float(seg.get("stitch_confidence", 0.0) or 0.0),
                )
            else:
                merged.append(dict(seg))

        absorbed = []
        for i, seg in enumerate(merged):
            dur = seg["end"] - seg["start"]
            prev_seg = merged[i - 1] if i > 0 else None
            next_seg = merged[i + 1] if i + 1 < len(merged) else None
            if (
                dur < 0.20
                and prev_seg is not None
                and next_seg is not None
                and prev_seg["speaker"] == next_seg["speaker"]
            ):
                prev_gap = max(0.0, seg["start"] - prev_seg["end"])
                next_gap = max(0.0, next_seg["start"] - seg["end"])
                if prev_gap <= 0.12 and next_gap <= 0.12:
                    continue
            absorbed.append(dict(seg))

        for i, seg in enumerate(absorbed):
            dur = seg["end"] - seg["start"]
            prev_seg = absorbed[i - 1] if i > 0 else None
            next_seg = absorbed[i + 1] if i + 1 < len(absorbed) else None
            unstable = bool(seg.get("unstable", False))
            if dur < 0.35 and prev_seg is not None and next_seg is not None:
                prev_gap = max(0.0, seg["start"] - prev_seg["end"])
                next_gap = max(0.0, next_seg["start"] - seg["end"])
                if prev_gap <= 0.18 and next_gap <= 0.18 and prev_seg["speaker"] == next_seg["speaker"] != seg["speaker"]:
                    unstable = True
            seg["unstable"] = unstable

        return absorbed

    speaker_segments = clean_speaker_segments_for_assignment(speaker_segments)
    globals()["speaker_segments"] = speaker_segments

    # =========================
    # 4) Sentence-level assignment
    # =========================

    def overlap_len(a0, a1, b0, b1):
        return max(0.0, min(a1, b1) - max(a0, b0))


    def interval_iou(a0, a1, b0, b1):
        inter = overlap_len(a0, a1, b0, b1)
        union = max(a1, b1) - min(a0, b0)
        if union <= 0:
            return 0.0
        return inter / union


    def _center_bonus(seg_start, seg_end, sent_start, sent_end):
        sent_mid = 0.5 * (sent_start + sent_end)
        seg_mid = 0.5 * (seg_start + seg_end)
        sent_len = max(1e-6, sent_end - sent_start)
        ratio = abs(seg_mid - sent_mid) / sent_len
        if ratio <= 0.18:
            return 1.25
        if ratio <= 0.35:
            return 1.08
        return 1.0


    def compute_speaker_candidates(item: Dict[str, Any], speaker_segments: List[Dict[str, Any]], prev_speaker: str = None) -> List[Dict[str, Any]]:
        s0 = float(item["start"])
        s1 = float(item["end"])
        dur = max(1e-6, s1 - s0)

        bucket = defaultdict(
            lambda: {
                "overlap": 0.0,
                "weighted_overlap": 0.0,
                "weighted_evidence": 0.0,
                "max_iou": 0.0,
                "unstable_overlap": 0.0,
                "count": 0,
                "short_single_overlap": 0.0,
            }
        )

        for seg in speaker_segments:
            g0 = float(seg["start"])
            g1 = float(seg["end"])
            spk = seg["speaker"]
            inter = overlap_len(s0, s1, g0, g1)
            if inter <= 0:
                continue

            unstable = bool(seg.get("unstable", False))
            seg_dur = max(0.0, g1 - g0)
            seg_weight = 0.5 if unstable else 1.0
            iou = interval_iou(s0, s1, g0, g1)
            center_w = _center_bonus(g0, g1, s0, s1)

            rec = bucket[spk]
            rec["overlap"] += inter
            rec["weighted_overlap"] += inter * center_w * seg_weight
            rec["weighted_evidence"] += inter * seg_weight
            rec["max_iou"] = max(rec["max_iou"], iou)
            rec["count"] += 1
            if unstable:
                rec["unstable_overlap"] += inter
            if seg_dur < 0.35:
                rec["short_single_overlap"] += inter

        if not bucket:
            return []

        total_weighted = sum(v["weighted_evidence"] for v in bucket.values())
        cands = []
        for spk, rec in bucket.items():
            overlap_ratio = min(1.0, rec["weighted_evidence"] / dur)
            overlap_share = rec["weighted_evidence"] / max(total_weighted, 1e-6)
            center_gain = rec["weighted_overlap"] / max(rec["weighted_evidence"], 1e-6)
            unstable_ratio = rec["unstable_overlap"] / max(rec["overlap"], 1e-6)
            short_frag_ratio = rec["short_single_overlap"] / max(rec["overlap"], 1e-6)
            continuity_bonus = 0.0
            if prev_speaker and spk == prev_speaker:
                continuity_bonus = 0.05 if dur <= 1.4 else 0.02
            single_short_penalty = 0.0
            if rec["count"] == 1 and rec["overlap"] < 0.35:
                single_short_penalty = 0.08
            elif short_frag_ratio > 0.75 and dur <= 1.4:
                single_short_penalty = 0.05

            score = (
                0.58 * overlap_ratio
                + 0.18 * rec["max_iou"]
                + 0.10 * overlap_share
                + 0.05 * min(1.35, center_gain) / 1.35
                + continuity_bonus
                - 0.06 * min(1.0, unstable_ratio)
                - single_short_penalty
            )

            cands.append(
                {
                    "speaker": spk,
                    "score": float(score),
                    "overlap_ratio": float(overlap_ratio),
                    "overlap_share": float(overlap_share),
                    "max_iou": float(rec["max_iou"]),
                    "unstable_ratio": float(unstable_ratio),
                    "continuity_bonus": float(continuity_bonus),
                    "single_short_penalty": float(single_short_penalty),
                    "raw_overlap": float(rec["overlap"]),
                    "segment_count": int(rec["count"]),
                }
            )

        cands.sort(key=lambda x: x["score"], reverse=True)
        return cands


    def assign_speaker_to_item(item: Dict[str, Any], speaker_segments: List[Dict[str, Any]], idx: int, prev_stable: str = None):
        out = dict(item)
        dur = max(1e-6, float(out["end"]) - float(out["start"]))
        cands = compute_speaker_candidates(out, speaker_segments, prev_speaker=prev_stable)

        if not cands:
            out["speaker"] = None
            out["speaker_confidence"] = 0.0
            out["speaker_uncertain"] = True
            out["speaker_reason"] = "no_overlap_evidence"
            out["_speaker_candidates"] = []
            diag = {
                "idx": idx,
                "start": float(out["start"]),
                "end": float(out["end"]),
                "text": out.get("text", ""),
                "speaker": None,
                "speaker_confidence": 0.0,
                "speaker_uncertain": True,
                "speaker_reason": "no_overlap_evidence",
                "top_candidates": [],
            }
            return out, diag

        best = dict(cands[0])
        second = cands[1] if len(cands) > 1 else None
        margin = best["score"] - (second["score"] if second else 0.0)
        has_fast_competing_turn = bool(
            dur <= 1.8
            and prev_stable
            and best["speaker"] != prev_stable
            and second is not None
            and second["speaker"] == prev_stable
            and margin >= 0.03
            and best["overlap_ratio"] >= 0.20
        )

        if dur <= 1.6 and prev_stable and not has_fast_competing_turn:
            prev_cand = next((c for c in cands if c["speaker"] == prev_stable), None)
            if prev_cand and (best["score"] - prev_cand["score"]) <= 0.10:
                best = dict(prev_cand)
                best["score"] = max(best["score"], cands[0]["score"] - 0.01)
                margin = best["score"] - (second["score"] if second else 0.0)
                out["speaker_reason"] = "short_inherit_prev_stable"

        confidence = 0.68 * best["score"] + 0.32 * max(0.0, margin)
        confidence = max(0.0, min(1.0, confidence))

        uncertain = False
        reason = out.get("speaker_reason", "direct_best_score")
        if best["overlap_ratio"] < 0.30:
            uncertain = True
            reason = "low_overlap_ratio"
        elif second and margin < 0.14 and best["overlap_ratio"] < 0.42:
            uncertain = True
            reason = "weak_margin_low_overlap"
        elif confidence < 0.55:
            uncertain = True
            reason = "low_confidence"

        out["speaker"] = best["speaker"]
        out["speaker_confidence"] = round(float(confidence), 4)
        out["speaker_uncertain"] = bool(uncertain)
        out["speaker_reason"] = reason
        out["_speaker_candidates"] = cands[:5]

        diag = {
            "idx": idx,
            "start": float(out["start"]),
            "end": float(out["end"]),
            "text": out.get("text", ""),
            "speaker": out["speaker"],
            "speaker_confidence": out["speaker_confidence"],
            "speaker_uncertain": out["speaker_uncertain"],
            "speaker_reason": out["speaker_reason"],
            "margin": round(float(margin), 4),
            "prev_stable": prev_stable,
            "top_candidates": cands[:3],
        }
        return out, diag


    def _neighbor_stable_speaker(items: List[Dict[str, Any]], idx: int, direction: int, max_steps: int = 4):
        n = len(items)
        steps = 0
        j = idx + direction

        while 0 <= j < n and steps < max_steps:
            spk = items[j].get("speaker")
            uncertain = bool(items[j].get("speaker_uncertain", False))
            if spk and not uncertain:
                return spk
            j += direction
            steps += 1

        return None


    def smooth_uncertain_speakers(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        out = [dict(x) for x in items]

        for i in range(len(out)):
            if not out[i].get("speaker_uncertain", False):
                continue

            prev_stable = _neighbor_stable_speaker(out, i, direction=-1, max_steps=4)
            next_stable = _neighbor_stable_speaker(out, i, direction=1, max_steps=4)
            cand_scores = {x["speaker"]: x["score"] for x in out[i].get("_speaker_candidates", [])}
            cur_spk = out[i].get("speaker")
            dur = float(out[i]["end"]) - float(out[i]["start"])

            if prev_stable and next_stable and prev_stable == next_stable:
                out[i]["speaker"] = prev_stable
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "uncertain_bridge_prev_next_same"
                continue

            if dur <= 1.4 and prev_stable and prev_stable in cand_scores:
                if next_stable and next_stable != prev_stable and cand_scores.get(next_stable, -1e9) >= (cand_scores[prev_stable] - 0.04):
                    continue
                cur_score = cand_scores.get(cur_spk, -1e9)
                if cand_scores[prev_stable] >= (cur_score - 0.10):
                    out[i]["speaker"] = prev_stable
                    out[i]["speaker_uncertain"] = False
                    out[i]["speaker_reason"] = "uncertain_short_smoothed_to_prev"
                    continue

            if cur_spk is None and prev_stable and next_stable and prev_stable == next_stable:
                out[i]["speaker"] = prev_stable
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "uncertain_fill_prev_next_same"
                continue

            if cur_spk is None:
                out[i]["speaker"] = None
                out[i]["speaker_uncertain"] = True
                out[i]["speaker_reason"] = "keep_none_no_strong_evidence"

        return out


    def _speaker_item_is_weak(it: Dict[str, Any]) -> bool:
        return bool(it.get("speaker_uncertain", False)) or float(it.get("speaker_confidence", 0.0)) < 0.72


    def _candidate_score(it: Dict[str, Any], speaker: str) -> float:
        for cand in it.get("_speaker_candidates", []) or []:
            if cand.get("speaker") == speaker:
                return float(cand.get("score", 0.0))
        return -1e9


    def speaker_run_smoother(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        out = [dict(x) for x in items]
        n = len(out)

        for i in range(1, n - 1):
            prev_it = out[i - 1]
            cur_it = out[i]
            next_it = out[i + 1]
            prev_spk = prev_it.get("speaker")
            next_spk = next_it.get("speaker")
            cur_spk = cur_it.get("speaker")
            cur_dur = float(cur_it["end"]) - float(cur_it["start"])
            cur_conf = float(cur_it.get("speaker_confidence", 0.0))
            if not prev_spk or prev_spk != next_spk:
                continue
            if cur_dur >= 1.8:
                continue
            if next_spk and prev_spk != cur_spk and _candidate_score(cur_it, next_spk) >= (_candidate_score(cur_it, prev_spk) - 0.04):
                continue
            if cur_conf < 0.72 or _candidate_score(cur_it, prev_spk) >= (_candidate_score(cur_it, cur_spk) - 0.12):
                out[i]["speaker"] = prev_spk
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "speaker_run_smoothed_aba"

        for i in range(1, n - 2):
            left = out[i - 1]
            mid1 = out[i]
            mid2 = out[i + 1]
            right = out[i + 2]
            left_spk = left.get("speaker")
            right_spk = right.get("speaker")
            if not left_spk or left_spk != right_spk:
                continue
            mid_dur = (float(mid1["end"]) - float(mid1["start"])) + (float(mid2["end"]) - float(mid2["start"]))
            if mid_dur >= 1.5:
                continue
            if not (_speaker_item_is_weak(mid1) and _speaker_item_is_weak(mid2)):
                continue
            out[i]["speaker"] = left_spk
            out[i]["speaker_uncertain"] = False
            out[i]["speaker_reason"] = "speaker_run_smoothed_abba"
            out[i + 1]["speaker"] = left_spk
            out[i + 1]["speaker_uncertain"] = False
            out[i + 1]["speaker_reason"] = "speaker_run_smoothed_abba"

        return out


    def window_majority_speaker_smoother(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        out = [dict(x) for x in items]
        rewrites = 0

        for i, cur in enumerate(out):
            if bool(cur.get("is_secondary_overlap")) or bool(cur.get("secondary_speaker")):
                continue
            dur = float(cur["end"]) - float(cur["start"])
            conf = float(cur.get("speaker_confidence", 0.0))
            if not (dur <= 2.0 or conf < 0.76 or bool(cur.get("speaker_uncertain", False))):
                continue

            lo = max(0, i - 2)
            hi = min(len(out), i + 3)
            bucket = {}
            for j in range(lo, hi):
                spk = out[j].get("speaker")
                if not spk:
                    continue
                stable = not bool(out[j].get("speaker_uncertain", False))
                seg_dur = max(0.0, float(out[j]["end"]) - float(out[j]["start"]))
                bucket[spk] = bucket.get(spk, 0.0) + (seg_dur if stable else seg_dur * 0.5)

            if not bucket:
                continue

            total = sum(bucket.values())
            major_spk, major_dur = max(bucket.items(), key=lambda kv: kv[1])
            if total <= 0 or major_dur / total < 0.58:
                continue

            cur_spk = cur.get("speaker")
            if cur_spk == major_spk:
                continue

            if _candidate_score(cur, major_spk) >= (_candidate_score(cur, cur_spk) - 0.18):
                out[i]["speaker"] = major_spk
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "window_majority_smoother"
                rewrites += 1

        globals()["_window_speaker_rewrite"] = rewrites
        return out


In [ ]:
#@title **diarization 结果导出与诊断**
if ENABLE_DIARIZATION:
    def save_assignment_diagnostics(path: str, rows: List[Dict[str, Any]]):
        with open(path, "w", encoding="utf-8") as f:
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")


    def save_timeline_plot(path: str, rttm_segments: List[Dict[str, Any]], items: List[Dict[str, Any]]):
        try:
            import matplotlib.pyplot as plt
            import matplotlib.patches as mpatches
        except Exception as e:
            print(f"[Diag] Skip timeline plot (matplotlib unavailable): {e}")
            return False

        speakers = sorted({x["speaker"] for x in rttm_segments if x.get("speaker")} | {x.get("speaker") for x in items if x.get("speaker")})
        if not speakers:
            print("[Diag] Skip timeline plot (no speaker data)")
            return False

        cmap = plt.get_cmap("tab20")
        colors = {spk: cmap(i % 20) for i, spk in enumerate(speakers)}

        fig, axes = plt.subplots(2, 1, figsize=(15, 4.8), sharex=True, constrained_layout=True)

        for seg in rttm_segments:
            dur = max(0.0, float(seg["end"]) - float(seg["start"]))
            if dur <= 0:
                continue
            axes[0].broken_barh([(float(seg["start"]), dur)], (0.1, 0.8), facecolors=colors.get(seg["speaker"], "gray"))

        for it in items:
            spk = it.get("speaker")
            if not spk:
                continue
            dur = max(0.0, float(it["end"]) - float(it["start"]))
            if dur <= 0:
                continue
            uncertain = bool(it.get("speaker_uncertain", False))
            axes[1].broken_barh(
                [(float(it["start"]), dur)],
                (0.1, 0.8),
                facecolors=colors.get(spk, "gray"),
                alpha=0.35 if uncertain else 0.9,
                hatch="//" if uncertain else None,
            )

        axes[0].set_title("RTTM Speaker Segments")
        axes[1].set_title("Sentence-level Assigned Speakers")
        axes[0].set_yticks([])
        axes[1].set_yticks([])
        axes[1].set_xlabel("Time (s)")

        xmax = 0.0
        for seg in rttm_segments:
            xmax = max(xmax, float(seg["end"]))
        for it in items:
            xmax = max(xmax, float(it["end"]))
        axes[0].set_xlim(0.0, max(1.0, xmax))

        legend_handles = [mpatches.Patch(color=colors[s], label=s) for s in speakers]
        axes[0].legend(handles=legend_handles, loc="upper right", ncol=min(4, len(legend_handles)), fontsize=8)

        fig.savefig(path, dpi=160)
        plt.close(fig)
        return True


    # =========================
    # 5) Assign + diagnostics
    # =========================

    diagnostics_rows = []
    diar_debug_payload = dict(globals().get("diar_debug_payload", {}) or {})

    if ENABLE_DIARIZATION and speaker_segments:
        assigned = []
        prev_stable = None
        for i, item in enumerate(sentence_items):
            x, diag = assign_speaker_to_item(dict(item), speaker_segments, idx=i, prev_stable=prev_stable)
            assigned.append(x)
            diagnostics_rows.append(diag)
            if x.get("speaker") and not x.get("speaker_uncertain", False):
                prev_stable = x.get("speaker")

        assigned = smooth_uncertain_speakers(assigned)
        assigned = speaker_run_smoother(assigned)
        assigned = window_majority_speaker_smoother(assigned)

        for i, x in enumerate(assigned):
            if i < len(diagnostics_rows):
                diagnostics_rows[i]["final_speaker"] = x.get("speaker")
                diagnostics_rows[i]["final_confidence"] = x.get("speaker_confidence")
                diagnostics_rows[i]["final_uncertain"] = bool(x.get("speaker_uncertain", False))
                diagnostics_rows[i]["final_reason"] = x.get("speaker_reason")

        for x in assigned:
            x.pop("_speaker_candidates", None)

        sentence_items = assigned

    # =========================
    # 6) Save output
    # =========================

    with open("sentence_items_with_speaker.json", "w", encoding="utf-8") as f:
        json.dump(sentence_items, f, ensure_ascii=False, indent=2)

    print("saved: sentence_items_with_speaker.json")

    diar_debug_payload["speaker_segment_count"] = len(speaker_segments or [])
    diar_debug_payload["assigned_item_count"] = len(sentence_items or [])
    diar_debug_payload["diar_meta"] = diar_meta
    with open("diarization.debug.json", "w", encoding="utf-8") as f:
        json.dump(diar_debug_payload, f, ensure_ascii=False, indent=2)
    print("saved: diarization.debug.json")

    if runtime_cfg["enable_diagnostics"]:
        diag_path = "speaker_assignment_diagnostics.jsonl"
        save_assignment_diagnostics(diag_path, diagnostics_rows)
        print("saved:", diag_path)

        timeline_path = "speaker_timeline.png"
        if save_timeline_plot(timeline_path, speaker_segments, sentence_items):
            print("saved:", timeline_path)

    print("diarization_meta:", json.dumps(diar_meta, ensure_ascii=False))
    print("done diarization + speaker assignment")


### 字幕后处理与导出


In [ ]:
#@title **finalize 参数与基础工具**
# Cell 14 最终版：speaker-aware finalize + overlap preserve + de-fragment merge + export
import os
import re
import json
import math
from copy import deepcopy
from collections import defaultdict, Counter

print("sentence_items in:", len(sentence_items))
if "speaker_segments" in globals():
    print("speaker_segments:", len(speaker_segments))
for k in [
    "MAX_GAP_MERGE_SAME",
    "MAX_GAP_MERGE_TAIL",
    "TAIL_FRAGMENT_MAX_DUR",
    "TAIL_FRAGMENT_MAX_CHARS",
    "SHORT_UTT_MAX_DUR",
    "SHORT_UTT_MAX_CHARS",
    "MERGED_MAX_DUR",
    "MERGED_MAX_CHARS",
    "OVERLAP_STRONG_SEC",
    "OVERLAP_RATIO_TH",
    "SECONDARY_KEEP_RATIO",
    "SECONDARY_KEEP_SEC",
    "SPEAKER_STABLE_MARGIN",
    "SHORT_REACTION_MAX_DUR",
    "SHORT_REACTION_LOOKAROUND_SEC",
    "SHORT_REACTION_SCORE_MARGIN",
]:
    if k in globals():
        print(k, "=", globals()[k])

# =========================================================
# 开关 / 参数
# =========================================================
ENABLE_DIARIZATION = bool(globals().get("ENABLE_DIARIZATION", True))

ALLOW_CROSS_SPEAKER_OVERLAP = bool(globals().get("ALLOW_CROSS_SPEAKER_OVERLAP", True))
ENABLE_MULTI_TRACK_ASS = bool(globals().get("ENABLE_MULTI_TRACK_ASS", True))
ENABLE_GLOBAL_CONFLICT_RESOLVE = bool(globals().get("ENABLE_GLOBAL_CONFLICT_RESOLVE", True))

ASS_FORCE_SINGLE_LINE = bool(globals().get("ASS_FORCE_SINGLE_LINE", True))

MAX_LINE_LEN = int(globals().get("MAX_LINE_LEN", 18))
MAX_LINES = int(globals().get("MAX_LINES", 2))

MIN_SENT_DUR = float(globals().get("MIN_SENT_DUR", 0.82))
MAX_SENT_DUR = min(float(globals().get("MAX_SENT_DUR", 3.0)), 3.0)
MAX_CHARS_PER_CHUNK = min(int(globals().get("MAX_CHARS_PER_CHUNK", 20)), 20)
PRE_ASSIGN_SPLIT_MAX_DUR = float(globals().get("PRE_ASSIGN_SPLIT_MAX_DUR", 3.6))
PRE_ASSIGN_SPLIT_MAX_CHARS = int(globals().get("PRE_ASSIGN_SPLIT_MAX_CHARS", 24))
POST_ASSIGN_SPLIT_MAX_DUR = float(globals().get("POST_ASSIGN_SPLIT_MAX_DUR", 2.8))
POST_ASSIGN_SPLIT_MAX_CHARS = int(globals().get("POST_ASSIGN_SPLIT_MAX_CHARS", 18))
MIN_SPLIT_CHILD_DUR = float(globals().get("MIN_SPLIT_CHILD_DUR", 0.75))
MIN_SPLIT_CHILD_CHARS = int(globals().get("MIN_SPLIT_CHILD_CHARS", 6))
MIN_INTER_SENT_GAP = float(globals().get("MIN_INTER_SENT_GAP", 0.03))
COLLAPSE_MICRO_GAP_BELOW = float(globals().get("COLLAPSE_MICRO_GAP_BELOW", 0.03))
MIN_MEANINGFUL_OVERLAP = float(globals().get("MIN_MEANINGFUL_OVERLAP", 0.10))
TEXT_DURATION_TARGET_CPS = float(globals().get("TEXT_DURATION_TARGET_CPS", 16.0))
TEXT_DURATION_MISMATCH_MAX_DUR = float(globals().get("TEXT_DURATION_MISMATCH_MAX_DUR", 0.70))
TEXT_DURATION_MISMATCH_MIN_CHARS = int(globals().get("TEXT_DURATION_MISMATCH_MIN_CHARS", 8))

START_PAD = float(globals().get("START_PAD", 0.16))
END_PAD = float(globals().get("END_PAD", 0.26))
SAFE_GAP_TO_NEXT = float(globals().get("SAFE_GAP_TO_NEXT", 0.03))
LAST_ITEM_EXTRA_PAD = float(globals().get("LAST_ITEM_EXTRA_PAD", 0.60))
MIN_TAIL_HOLD = float(globals().get("MIN_TAIL_HOLD", 0.12))
MAX_CONFLICT_TRIM_CUR = float(globals().get("MAX_CONFLICT_TRIM_CUR", 0.05))
MAX_CONFLICT_SHIFT_NEXT = float(globals().get("MAX_CONFLICT_SHIFT_NEXT", 0.18))
SPLIT_GAP_HARD = float(globals().get("SPLIT_GAP_HARD", 0.28))
SPLIT_GAP_SOFT = float(globals().get("SPLIT_GAP_SOFT", 0.12))
SAFE_SPLIT_TOKEN_GAP = float(globals().get("SAFE_SPLIT_TOKEN_GAP", 0.08))
SAFE_SPLIT_PAUSE = float(globals().get("SAFE_SPLIT_PAUSE", 0.16))
MIN_FINAL_PRIMARY_DUR = float(globals().get("MIN_FINAL_PRIMARY_DUR", 0.35))
MIN_FINAL_PRIMARY_CHARS = int(globals().get("MIN_FINAL_PRIMARY_CHARS", 4))

# overlap / speaker
OVERLAP_STRONG_SEC = float(globals().get("OVERLAP_STRONG_SEC", 0.16))
OVERLAP_RATIO_TH = float(globals().get("OVERLAP_RATIO_TH", 0.10))
SECONDARY_KEEP_RATIO = float(globals().get("SECONDARY_KEEP_RATIO", 0.48))
SECONDARY_KEEP_SEC = float(globals().get("SECONDARY_KEEP_SEC", 0.12))
SPEAKER_STABLE_MARGIN = float(globals().get("SPEAKER_STABLE_MARGIN", 0.06))
SHORT_REACTION_MAX_DUR = float(globals().get("SHORT_REACTION_MAX_DUR", 1.8))
SHORT_REACTION_LOOKAROUND_SEC = float(globals().get("SHORT_REACTION_LOOKAROUND_SEC", 0.5))
SHORT_REACTION_SCORE_MARGIN = float(globals().get("SHORT_REACTION_SCORE_MARGIN", 0.04))

# de-fragment merge
MAX_GAP_MERGE_SAME = min(float(globals().get("MAX_GAP_MERGE_SAME", 0.10)), 0.10)
MAX_GAP_MERGE_TAIL = min(float(globals().get("MAX_GAP_MERGE_TAIL", 0.12)), 0.12)
TAIL_FRAGMENT_MAX_DUR = min(float(globals().get("TAIL_FRAGMENT_MAX_DUR", 0.55)), 0.55)
TAIL_FRAGMENT_MAX_CHARS = min(int(globals().get("TAIL_FRAGMENT_MAX_CHARS", 4)), 4)
SHORT_UTT_MAX_DUR = min(float(globals().get("SHORT_UTT_MAX_DUR", 0.90)), 0.90)
SHORT_UTT_MAX_CHARS = min(int(globals().get("SHORT_UTT_MAX_CHARS", 8)), 8)
MERGED_MAX_DUR = min(float(globals().get("MERGED_MAX_DUR", 3.2)), 3.2)
MERGED_MAX_CHARS = min(int(globals().get("MERGED_MAX_CHARS", 20)), 20)
MAX_READ_CPS = float(globals().get("MAX_READ_CPS", 17.0))

# tail rescue
TAIL_RESCUE_MAX_EXTEND = min(float(globals().get("TAIL_RESCUE_MAX_EXTEND", 0.24)), 0.24)
TAIL_RESCUE_SINGLE_SPK = min(float(globals().get("TAIL_RESCUE_SINGLE_SPK", 0.18)), 0.18)
TAIL_RESCUE_SAME_SPK = min(float(globals().get("TAIL_RESCUE_SAME_SPK", 0.14)), 0.14)
TAIL_RESCUE_DIFF_SPK = min(float(globals().get("TAIL_RESCUE_DIFF_SPK", 0.06)), 0.06)

# ass
ASS_MARGIN_V_MAIN = int(globals().get("ASS_MARGIN_V_MAIN", 0))
ASS_MARGIN_V_SUB = int(globals().get("ASS_MARGIN_V_SUB", 0))
ASS_LAYER_MAIN = int(globals().get("ASS_LAYER_MAIN", 0))
ASS_LAYER_SUB = int(globals().get("ASS_LAYER_SUB", 1))

PRINT_FINAL_COUNTS_ONLY = bool(globals().get("PRINT_FINAL_COUNTS_ONLY", True))
SUBTITLE_TEXT_ENCODING = str(globals().get("SUBTITLE_TEXT_ENCODING", "utf-8-sig"))

TOKEN_SPLIT_HARD_PUNCT = {"。", "！", "？", "!", "?"}
TOKEN_SPLIT_SOFT_PUNCT = {"、", "，", ",", "…", "～", "~", "；", ";", "：", ":"}


# =========================================================
# 基础工具
# =========================================================
def _f(x, d=0.0):
    try:
        return float(x)
    except:
        return float(d)

def _s(x):
    return "" if x is None else str(x)

def clean_compact_text(s):
    s = _s(s)
    s = s.replace("\u3000", " ")
    s = re.sub(r"[ \t\r\f\v]+", " ", s)
    s = re.sub(r"\n+", " ", s)
    return s.strip()

def compact_for_match(s):
    return re.sub(r"\s+", "", clean_compact_text(s))

def visible_text(s):
    return clean_compact_text(s)

def text_len(s):
    return len(compact_for_match(s))

def is_empty_text(s):
    return text_len(s) == 0

def item_dur(x):
    return max(0.0, _f(x.get("end")) - _f(x.get("start")))

def overlap_sec(a0, a1, b0, b1):
    return max(0.0, min(a1, b1) - max(a0, b0))

def safe_round(x, n=3):
    return round(float(x), n)

def copy_item(x):
    return dict(x)

def sort_items(items):
    return sorted(items, key=lambda z: (_f(z.get("start")), _f(z.get("end"))))

def normalize_speaker_name(spk):
    if spk is None:
        return "Default"
    s = str(spk).strip()
    if not s:
        return "Default"
    m = re.search(r'(\d+)$', s)
    if m:
        return f"Speaker{int(m.group(1))}"
    s = re.sub(r"\s+", "_", s)
    return s

def norm_speaker_key(spk):
    s = _s(spk).strip()
    return s if s else "speaker_unknown"

def is_short_text(text):
    return text_len(text) <= SHORT_UTT_MAX_CHARS

def looks_like_independent_short_utt(text):
    t = clean_compact_text(text)
    if not t:
        return False
    keep_set = {
        "嗯", "啊", "诶", "欸", "哦", "喔", "哈", "唉",
        "嗯？", "啊？", "诶？", "欸？", "哦？", "真的", "真的假的", "不是", "对", "对啊",
        "等一下", "等会儿", "没有", "有", "行", "可以", "好",
        "うん", "え", "えっ", "あ", "あっ", "へえ", "マジ", "ほんと", "本当", "そう", "いや", "はい", "いいよ",
    }
    if t in keep_set:
        return True
    if len(t) <= 3 and re.fullmatch(r"[嗯啊诶欸哦喔哈唉えあうんはい]+[？!?？！…]*", t):
        return True
    return False

def merge_text(a, b):
    ta = clean_compact_text(a)
    tb = clean_compact_text(b)
    if not ta:
        return tb
    if not tb:
        return ta
    if re.search(r"[\u4e00-\u9fffぁ-んァ-ン]$", ta) or re.search(r"^[\u4e00-\u9fffぁ-んァ-ン]", tb):
        return ta + tb
    if re.search(r"[A-Za-z0-9]$", ta) and re.search(r"^[A-Za-z0-9]", tb):
        return ta + " " + tb
    return ta + tb


# =========================================================
# 时间格式
# =========================================================
def format_ass_time(sec: float) -> str:
    sec = max(0.0, float(sec))
    h = int(sec // 3600)
    sec -= h * 3600
    m = int(sec // 60)
    sec -= m * 60
    s = int(sec)
    cs = int(round((sec - s) * 100))
    if cs >= 100:
        s += 1
        cs -= 100
    if s >= 60:
        m += 1
        s -= 60
    if m >= 60:
        h += 1
        m -= 60
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

def format_srt_time(sec: float) -> str:
    sec = max(0.0, float(sec))
    h = int(sec // 3600)
    sec -= h * 3600
    m = int(sec // 60)
    sec -= m * 60
    s = int(sec)
    ms = int(round((sec - s) * 1000))
    if ms >= 1000:
        s += 1
        ms -= 1000
    if s >= 60:
        m += 1
        s -= 60
    if m >= 60:
        h += 1
        m -= 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


In [ ]:
#@title **分句与文本切分工具**
def split_lines(text, max_len=18, max_lines=2):
    text = visible_text(text)
    compact = compact_for_match(text)

    if len(compact) <= max_len:
        return text

    best = None
    for i, ch in enumerate(text):
        pos = i + 1
        left = compact_for_match(text[:pos])
        right = compact_for_match(text[pos:])
        if ch in TOKEN_SPLIT_HARD_PUNCT or ch in TOKEN_SPLIT_SOFT_PUNCT:
            if len(left) >= 4 and len(right) >= 4:
                score = abs(len(left) - len(compact) / 2)
                if best is None or score < best[0]:
                    best = (score, pos)

    if best is not None:
        pos = best[1]
        a = text[:pos].rstrip()
        b = text[pos:].lstrip()
        if len(compact_for_match(a)) <= max_len and len(compact_for_match(b)) <= max_len:
            return a + r"\N" + b

    if max_lines == 2:
        target = len(compact) // 2
        best_pos = None
        best_score = 10**9
        for i in range(1, len(text)):
            left = compact_for_match(text[:i])
            right = compact_for_match(text[i:])
            if not left or not right:
                continue
            if len(left) <= max_len and len(right) <= max_len:
                score = abs(len(left) - target)
                if score < best_score:
                    best_score = score
                    best_pos = i
        if best_pos is not None:
            a = text[:best_pos].rstrip()
            b = text[best_pos:].lstrip()
            return a + r"\N" + b

    return text

def to_ass_text(text):
    txt = visible_text(text)
    txt = txt.replace("\\r", " ").replace("\\n", " ").replace(r"\\N", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    if ASS_FORCE_SINGLE_LINE:
        return txt
    return split_lines(txt, max_len=MAX_LINE_LEN, max_lines=MAX_LINES)

def ass_escape(text: str) -> str:
    text = str(text)
    text = text.replace("\\", r"\\")
    text = text.replace("{", r"\{")
    text = text.replace("}", r"\}")
    return text


def _timed_tokens_for_item(item):
    out = []
    for key in ("tokens", "whisper_words"):
        for tk in item.get(key, []) or []:
            if not isinstance(tk, dict):
                continue
            txt = clean_compact_text(tk.get("text", ""))
            start = tk.get("start", None)
            end = tk.get("end", None)
            if start is None or end is None:
                continue
            start = float(start)
            end = float(end)
            if not txt or end <= start:
                continue
            out.append({"text": txt, "start": start, "end": end})
        if out:
            break
    out.sort(key=lambda z: (z["start"], z["end"]))
    return out


def _merge_token_texts(tokens):
    txt = ""
    for tk in tokens:
        txt = merge_text(txt, tk.get("text", ""))
    return clean_compact_text(txt)


def _build_split_item(base_item, tokens, start, end, text):
    out = copy_item(base_item)
    out["start"] = float(start)
    out["end"] = float(end)
    out["text"] = clean_compact_text(text)
    out["tokens"] = [dict(t) for t in tokens]
    if base_item.get("whisper_words"):
        out["whisper_words"] = [dict(t) for t in tokens]
    out["secondary_speaker"] = None
    out["is_secondary_overlap"] = False
    out["_split_parent_key"] = base_item.get("_split_parent_key") or f"{_f(base_item.get('start')):.3f}-{_f(base_item.get('end')):.3f}:{compact_for_match(base_item.get('text', ''))}"
    return out

def _split_boundary_bonus(left_text, gap):
    last = left_text[-1:] if left_text else ""
    if last in TOKEN_SPLIT_HARD_PUNCT:
        return 0.24
    if last in TOKEN_SPLIT_SOFT_PUNCT:
        return 0.14
    if gap >= SPLIT_GAP_HARD:
        return 0.18
    if gap >= SPLIT_GAP_SOFT:
        return 0.10
    return 0.0


SAFE_SPLIT_SOFT_CONNECTORS = {'\u3066', '\u3067', '\u3068', '\u306b', '\u3092', '\u304c', '\u306f', '\u306e', '\u3082', '\u3078', '\u3084', '\u304b', '\u3089', '\u3088\u308a', '\u4e86', '\u7684', '\u5730', '\u5f97', '\u7740', '\u8fc7'}
SAFE_SPLIT_CONTINUATION_PREFIXES = ('\u306e', '\u3066', '\u3067', '\u3068', '\u306b', '\u3092', '\u304c', '\u306f', '\u3082', '\u3078', '\u3063\u3066', '\u3044\u3046', '\u3088\u308a', '\u307b\u3057\u3044', '\u591a\u6570', '\u8aad\u3093\u3067', '\u3057\u3066', '\u3055\u308c\u305f', '\u3055\u305b', '\u305f\u3060', '\u307e\u3060')
SAFE_SPLIT_OPEN_QUOTES = ('\u300c', '\u300e')
SAFE_SPLIT_CLOSE_QUOTES = ('\u300d', '\u300f')


def _char_family(ch):
    if not ch:
        return "other"
    code = ord(ch)
    if 0x3040 <= code <= 0x30FF:
        return "kana"
    if 0x4E00 <= code <= 0x9FFF:
        return "han"
    if ch.isascii() and ch.isalpha():
        return "latin"
    if ch.isdigit():
        return "digit"
    return "other"


def _safe_split_text_piece(text):
    return compact_for_match(clean_compact_text(text))



def looks_like_numeric_chain_token(text):
    t = compact_for_match(text)
    if not t:
        return False
    return bool(re.fullmatch(r"[0-9０-９A-Za-z]+(?:[年月日号版回期])?", t))


def is_continuous_numeric_or_ascii_chain(left_tok, right_tok):
    lt = compact_for_match(left_tok)
    rt = compact_for_match(right_tok)
    if not lt or not rt:
        return False
    if not (looks_like_numeric_chain_token(lt) and looks_like_numeric_chain_token(rt)):
        return False
    merged = lt + rt
    return bool(re.fullmatch(r"[0-9０-９A-Za-z]+(?:[年月日号版回期])?", merged))


def split_preserves_compact_text(item, left_text, right_text):
    original = compact_for_match(item.get("text", ""))
    merged = compact_for_match(merge_text(left_text, right_text))
    if original != merged:
        return False
    original_digits = re.sub(r"\s+", "", clean_compact_text(item.get("text", "")))
    merged_digits = re.sub(r"\s+", "", clean_compact_text(merge_text(left_text, right_text)))
    if original_digits != merged_digits:
        return False
    return True


def reject_split_if_right_piece_looks_truncated(item, pieces):
    if not pieces or len(pieces) != 2:
        return False
    left, right = pieces
    left_text = clean_compact_text(left.get("text", ""))
    right_text = clean_compact_text(right.get("text", ""))
    if not left_text or not right_text:
        return True

    tokens = _timed_tokens_for_item(item)
    if len(tokens) >= 2:
        split_idx = len(left.get("tokens", []) or [])
        if 0 < split_idx < len(tokens):
            left_tok = clean_compact_text(tokens[split_idx - 1].get("text", ""))
            right_tok = clean_compact_text(tokens[split_idx].get("text", ""))
            gap = max(0.0, _f(tokens[split_idx].get("start")) - _f(tokens[split_idx - 1].get("end")))
            if is_continuous_numeric_or_ascii_chain(left_tok, right_tok):
                return True
            if gap < 0.20 and looks_like_numeric_chain_token(right_tok):
                return True

    if right_text.startswith(SAFE_SPLIT_CONTINUATION_PREFIXES) and not ends_with_hard_sentence_punct(left_text):
        return True
    if compact_for_match(right_text)[:6].startswith(tuple(compact_for_match(x) for x in SAFE_SPLIT_CONTINUATION_PREFIXES)):
        return True
    if re.match(r"^[0-9０-９](?:\s*[0-9０-９])+", right_text):
        return True
    if re.match(r"^[0-9０-９A-Za-z]+[年月日号版回期]?$", compact_for_match(right_text[:6])):
        left_compact = compact_for_match(left_text[-6:])
        right_compact = compact_for_match(right_text[:6])
        if left_compact and right_compact and re.fullmatch(r"[0-9０-９A-Za-z]+", left_compact + right_compact):
            return True
    return False


def _looks_like_unfinished_clause(text):
    t = clean_compact_text(text)
    if not t or ends_with_hard_sentence_punct(t):
        return False
    unfinished_suffixes = (
        '\u306e', '\u3092', '\u306b', '\u304c', '\u3068', '\u3066', '\u3067', '\u3063\u3066',
        '\u3051\u308c\u3069', '\u306a\u3093\u304b\u3092', '\u8aad\u3093\u3067', '\u307b\u3057\u3044', '\u591a\u6570'
    )
    return t.endswith(unfinished_suffixes)

def is_safe_split_boundary(left_tokens, right_tokens, left_text, right_text, gap):
    left_text = clean_compact_text(left_text)
    right_text = clean_compact_text(right_text)
    if not left_text or not right_text:
        return False

    left_last = left_text[-1]
    right_first = right_text[0]
    if left_text.endswith(SAFE_SPLIT_OPEN_QUOTES) or right_text.startswith(SAFE_SPLIT_CLOSE_QUOTES):
        return False
    if left_last in TOKEN_SPLIT_HARD_PUNCT or left_last in TOKEN_SPLIT_SOFT_PUNCT:
        return True
    if gap >= SAFE_SPLIT_PAUSE:
        return True

    left_tok = _safe_split_text_piece(left_tokens[-1].get("text", "")) if left_tokens else _safe_split_text_piece(left_text[-1:])
    right_tok = _safe_split_text_piece(right_tokens[0].get("text", "")) if right_tokens else _safe_split_text_piece(right_text[:1])
    if left_tok in SAFE_SPLIT_SOFT_CONNECTORS or right_tok in SAFE_SPLIT_SOFT_CONNECTORS:
        return False
    if right_text.startswith(SAFE_SPLIT_CONTINUATION_PREFIXES) and gap < max(SAFE_SPLIT_PAUSE, 0.24):
        return False
    if is_continuous_numeric_or_ascii_chain(left_tok, right_tok):
        return False

    left_family = _char_family(left_tok[-1] if left_tok else left_last)
    right_family = _char_family(right_tok[0] if right_tok else right_first)
    if gap < SAFE_SPLIT_TOKEN_GAP and left_family == right_family and left_family in {"kana", "han", "latin", "digit"}:
        return False
    if gap < SAFE_SPLIT_PAUSE and left_last not in TOKEN_SPLIT_HARD_PUNCT and left_last not in TOKEN_SPLIT_SOFT_PUNCT:
        return False
    return True

def _choose_token_split(item, max_dur, max_chars, min_child_dur, min_child_chars, min_gap, require_strong_boundary):
    tokens = _timed_tokens_for_item(item)
    if len(tokens) < 2:
        return None

    best = None
    for i in range(1, len(tokens)):
        left_tokens = tokens[:i]
        right_tokens = tokens[i:]
        left_text = _merge_token_texts(left_tokens)
        right_text = _merge_token_texts(right_tokens)
        if is_empty_text(left_text) or is_empty_text(right_text):
            continue

        left_len = text_len(left_text)
        right_len = text_len(right_text)
        left_dur = max(0.0, left_tokens[-1]["end"] - left_tokens[0]["start"])
        right_dur = max(0.0, right_tokens[-1]["end"] - right_tokens[0]["start"])
        gap = max(0.0, right_tokens[0]["start"] - left_tokens[-1]["end"])
        boundary_bonus = _split_boundary_bonus(left_text, gap)

        if left_len < min_child_chars or right_len < min_child_chars:
            continue
        if left_dur < min_child_dur or right_dur < min_child_dur:
            continue
        if not is_safe_split_boundary(left_tokens, right_tokens, left_text, right_text, gap):
            continue
        if require_strong_boundary and boundary_bonus <= 0.0 and gap < min_gap:
            continue

        overflow = max(0, left_len - max_chars) + max(0, right_len - max_chars)
        dur_over = max(0.0, left_dur - max_dur) + max(0.0, right_dur - max_dur)
        balance = abs(left_len - right_len)
        score = overflow * 5.0 + dur_over * 3.0 + balance * 0.08 - boundary_bonus
        if best is None or score < best[0]:
            best = (score, i, left_text, right_text)

    if best is None:
        return None

    _, i, left_text, right_text = best
    left_tokens = tokens[:i]
    right_tokens = tokens[i:]
    if not split_preserves_compact_text(item, left_text, right_text):
        return None
    left = _build_split_item(item, left_tokens, left_tokens[0]["start"], left_tokens[-1]["end"], left_text)
    right = _build_split_item(item, right_tokens, right_tokens[0]["start"], right_tokens[-1]["end"], right_text)
    pieces = [left, right]
    if reject_split_if_right_piece_looks_truncated(item, pieces):
        return None
    return pieces

def _choose_text_split_pos(text, max_chars):
    compact = compact_for_match(text)
    if len(compact) <= max_chars:
        return None

    target = max(6, min(len(compact) - 4, len(compact) // 2))
    best = None
    for i, ch in enumerate(text[:-1], 1):
        left = compact_for_match(text[:i])
        right = compact_for_match(text[i:])
        if not left or not right:
            continue
        raw_left = clean_compact_text(text[:i])
        raw_right = clean_compact_text(text[i:])
        if not is_safe_split_boundary([], [], raw_left, raw_right, 0.0):
            continue
        if len(left) > max_chars or len(right) > max_chars + 2:
            continue
        bonus = 0
        if ch in TOKEN_SPLIT_HARD_PUNCT:
            bonus = 4
        elif ch in TOKEN_SPLIT_SOFT_PUNCT:
            bonus = 2
        elif ch.isspace():
            bonus = 1
        score = abs(len(left) - target) - bonus
        if best is None or score < best[0]:
            best = (score, i)
    return None if best is None else best[1]


def _split_item_text_fallback(item, max_chars, min_child_dur, min_child_chars):
    text = clean_compact_text(item.get("text", ""))
    pos = _choose_text_split_pos(text, max_chars)
    if pos is None:
        return [copy_item(item)]

    left_text = clean_compact_text(text[:pos])
    right_text = clean_compact_text(text[pos:])
    if text_len(left_text) < min_child_chars or text_len(right_text) < min_child_chars:
        return [copy_item(item)]
    if not split_preserves_compact_text(item, left_text, right_text):
        return [copy_item(item)]

    start = _f(item.get("start"))
    end = _f(item.get("end"))
    dur = max(0.0, end - start)
    total_len = max(1, len(compact_for_match(text)))
    left_len = max(1, len(compact_for_match(left_text)))
    split_at = start + dur * (left_len / total_len)
    split_at = max(start + min_child_dur, min(end - min_child_dur, split_at))
    if split_at <= start or split_at >= end:
        return [copy_item(item)]
    left = _build_split_item(item, [], start, split_at, left_text)
    right = _build_split_item(item, [], split_at, end, right_text)
    pieces = [left, right]
    if reject_split_if_right_piece_looks_truncated(item, pieces):
        return [copy_item(item)]
    return pieces

def _reject_overfine_split(pieces):
    if not pieces or len(pieces) < 2:
        return False
    for left, right in zip(pieces, pieces[1:]):
        gap = _f(right.get("start")) - _f(left.get("end"))
        left_text = clean_compact_text(left.get("text", ""))
        strong_boundary = left_text[-1:] in TOKEN_SPLIT_HARD_PUNCT or gap >= 0.30
        if min(item_dur(left), item_dur(right)) < 0.90 and not strong_boundary:
            return True
    if len(pieces) >= 3:
        same_spk = {norm_speaker_key(x.get("speaker")) for x in pieces}
        total_dur = sum(item_dur(x) for x in pieces)
        if len(same_spk) <= 1 and total_dur < 2.2:
            return True
    return False


def pre_assign_split_item(item):
    text = clean_compact_text(item.get("text", ""))
    dur = item_dur(item)
    tokens = _timed_tokens_for_item(item)
    has_long_gap = any((tokens[i]["start"] - tokens[i - 1]["end"]) >= 0.28 for i in range(1, len(tokens)))
    if dur <= PRE_ASSIGN_SPLIT_MAX_DUR and text_len(text) <= PRE_ASSIGN_SPLIT_MAX_CHARS and not has_long_gap:
        return [copy_item(item)]

    pieces = _choose_token_split(
        item,
        max_dur=PRE_ASSIGN_SPLIT_MAX_DUR,
        max_chars=PRE_ASSIGN_SPLIT_MAX_CHARS,
        min_child_dur=MIN_SPLIT_CHILD_DUR,
        min_child_chars=MIN_SPLIT_CHILD_CHARS,
        min_gap=0.28,
        require_strong_boundary=True,
    )
    if pieces is not None:
        if _reject_overfine_split(pieces):
            globals()["_unsafe_split_rejected"] = int(globals().get("_unsafe_split_rejected", 0)) + 1
            return [copy_item(item)]
        return pieces
    return _split_item_text_fallback(item, PRE_ASSIGN_SPLIT_MAX_CHARS, MIN_SPLIT_CHILD_DUR, MIN_SPLIT_CHILD_CHARS)


def post_assign_split_item(item):
    text = clean_compact_text(item.get("text", ""))
    dur = item_dur(item)
    if 0.6 <= dur <= 1.2:
        return [copy_item(item)]
    if dur <= POST_ASSIGN_SPLIT_MAX_DUR and text_len(text) <= POST_ASSIGN_SPLIT_MAX_CHARS:
        return [copy_item(item)]

    pieces = _choose_token_split(
        item,
        max_dur=POST_ASSIGN_SPLIT_MAX_DUR,
        max_chars=POST_ASSIGN_SPLIT_MAX_CHARS,
        min_child_dur=MIN_SPLIT_CHILD_DUR,
        min_child_chars=MIN_SPLIT_CHILD_CHARS,
        min_gap=SPLIT_GAP_SOFT,
        require_strong_boundary=True,
    )
    if pieces is not None:
        if _reject_overfine_split(pieces):
            globals()["_unsafe_split_rejected"] = int(globals().get("_unsafe_split_rejected", 0)) + 1
            return [copy_item(item)]
        if all(norm_speaker_key(x.get("speaker")) == norm_speaker_key(item.get("speaker")) for x in pieces):
            for a, b in zip(pieces, pieces[1:]):
                if _f(b.get("start")) - _f(a.get("end")) < MIN_INTER_SENT_GAP:
                    return [copy_item(item)]
        return pieces
    return _split_item_text_fallback(item, POST_ASSIGN_SPLIT_MAX_CHARS, MIN_SPLIT_CHILD_DUR, MIN_SPLIT_CHILD_CHARS)


def _split_items_with(items, splitter, counter_name):
    pending = [copy_item(it) for it in sort_items(items)]
    out = []
    split_count = 0
    while pending:
        cur = pending.pop(0)
        pieces = splitter(cur)
        if len(pieces) == 1:
            out.append(pieces[0])
            continue
        split_count += len(pieces) - 1
        pending = pieces + pending
    globals()[counter_name] = split_count
    return sort_items(out)


def pre_assign_split_items(items):
    return _split_items_with(items, pre_assign_split_item, "_pre_assign_split_items")


def post_assign_split_items(items):
    return _split_items_with(items, post_assign_split_item, "_post_assign_split_items")

In [ ]:
#@title **speaker 分配与 overlap 处理**
def speaker_overlap_map(start, end, speaker_segments):
    scores = defaultdict(float)
    mids = defaultdict(float)
    dur = max(1e-6, end - start)
    mid0 = start + dur * 0.2
    mid1 = end - dur * 0.2
    if mid1 <= mid0:
        mid0, mid1 = start, end

    for seg in speaker_segments or []:
        ss = _f(seg.get("start"))
        ee = _f(seg.get("end"))
        spk = norm_speaker_key(seg.get("speaker"))
        ov = overlap_sec(start, end, ss, ee)
        if ov > 0:
            scores[spk] += ov
        mov = overlap_sec(mid0, mid1, ss, ee)
        if mov > 0:
            mids[spk] += mov
    return scores, mids


def region_has_overlap(start, end, speaker_segments):
    by_spk = defaultdict(list)
    for seg in speaker_segments or []:
        ss = _f(seg.get("start"))
        ee = _f(seg.get("end"))
        spk = norm_speaker_key(seg.get("speaker"))
        ov0 = max(start, ss)
        ov1 = min(end, ee)
        if ov1 > ov0:
            by_spk[spk].append((ov0, ov1))

    speakers = list(by_spk.keys())
    if len(speakers) < 2:
        return False

    for i in range(len(speakers)):
        for j in range(i + 1, len(speakers)):
            for a0, a1 in by_spk[speakers[i]]:
                for b0, b1 in by_spk[speakers[j]]:
                    if overlap_sec(a0, a1, b0, b1) >= OVERLAP_STRONG_SEC:
                        return True
    return False


def _has_fast_handoff_window(start, end, primary, secondary, speaker_segments):
    if not primary or not secondary or primary == secondary:
        return False
    lo = max(0.0, start - SHORT_REACTION_LOOKAROUND_SEC)
    hi = end + SHORT_REACTION_LOOKAROUND_SEC
    pri_hits = 0
    sec_hits = 0
    for seg in speaker_segments or []:
        ss = _f(seg.get("start"))
        ee = _f(seg.get("end"))
        if ee <= lo or ss >= hi:
            continue
        spk = norm_speaker_key(seg.get("speaker"))
        if spk == primary:
            pri_hits += 1
        elif spk == secondary:
            sec_hits += 1
    return pri_hits > 0 and sec_hits > 0


def assign_speaker_for_item(item, speaker_segments, prev_speaker=None):
    start = _f(item.get("start"))
    end = _f(item.get("end"))
    dur = max(1e-6, end - start)
    base_speaker = norm_speaker_key(item.get("speaker")) if item.get("speaker") else None
    is_tail = bool(item.get("is_tail_rescue", False))
    is_weak_text = text_len(item.get("text", "")) <= 6

    scores, mids = speaker_overlap_map(start, end, speaker_segments)
    out = copy_item(item)

    if not scores:
        out["speaker"] = base_speaker or prev_speaker or "speaker_unknown"
        out["secondary_speaker"] = None
        out["secondary_overlap_sec"] = 0.0
        out["speaker_confidence"] = 0.0
        out["has_overlap_region"] = False
        out["speaker_uncertain"] = True
        out["speaker_reason"] = "no_overlap_evidence"
        out["_speaker_candidates"] = []
        out["track"] = 0
        return out

    final_scores = {}
    for spk in set(scores) | set(mids):
        final_scores[spk] = scores.get(spk, 0.0) + mids.get(spk, 0.0) * 0.55

    if base_speaker and base_speaker in final_scores:
        final_scores[base_speaker] += 0.04 if dur <= 1.4 else 0.02

    ranked = sorted(final_scores.items(), key=lambda kv: kv[1], reverse=True)
    primary, pscore = ranked[0]
    secondary, sscore = (ranked[1] if len(ranked) > 1 else (None, 0.0))
    weak_item = dur <= 1.6 or is_weak_text or is_tail
    fast_handoff = bool(
        weak_item
        and dur <= SHORT_REACTION_MAX_DUR
        and secondary is not None
        and _has_fast_handoff_window(start, end, primary, secondary, speaker_segments)
        and mids.get(primary, 0.0) >= max(0.02, mids.get(secondary, 0.0) - SHORT_REACTION_SCORE_MARGIN)
    )

    if prev_speaker and prev_speaker in final_scores and not fast_handoff:
        prev_score = final_scores[prev_speaker]
        stable_margin = SPEAKER_STABLE_MARGIN + (0.03 if weak_item else 0.0)
        if (pscore - prev_score) <= stable_margin:
            primary = prev_speaker
            pscore = prev_score
            other = [(k, v) for k, v in ranked if k != primary]
            secondary, sscore = (other[0] if other else (None, 0.0))

    confidence = min(1.0, pscore / dur)
    has_overlap = region_has_overlap(start, end, speaker_segments)
    if not has_overlap and fast_handoff and secondary is not None:
        has_overlap = bool(scores.get(secondary, 0.0) >= max(MIN_MEANINGFUL_OVERLAP * 0.8, 0.08))

    primary_overlap = scores.get(primary, 0.0)
    primary_overlap_ratio = primary_overlap / dur if dur > 0 else 0.0
    margin = pscore - sscore if secondary is not None else pscore
    speaker_candidates = [
        {
            "speaker": spk,
            "score": safe_round(score, 4),
            "overlap_sec": safe_round(scores.get(spk, 0.0), 4),
            "mid_score": safe_round(mids.get(spk, 0.0), 4),
        }
        for spk, score in ranked[:5]
    ]

    keep_secondary = False
    secondary_overlap_sec = 0.0
    if secondary is not None and has_overlap:
        raw2 = scores.get(secondary, 0.0)
        if weak_item:
            sec_need = max(MIN_MEANINGFUL_OVERLAP * 0.8, 0.08)
            sec_ratio_need = max(OVERLAP_RATIO_TH * 0.8, 0.08)
            sec_keep_ratio = min(0.60, max(SECONDARY_KEEP_RATIO, 0.55))
        else:
            sec_need = max(SECONDARY_KEEP_SEC, MIN_MEANINGFUL_OVERLAP)
            sec_ratio_need = OVERLAP_RATIO_TH
            sec_keep_ratio = SECONDARY_KEEP_RATIO
        if raw2 >= sec_need and raw2 / dur >= sec_ratio_need and sscore >= pscore * sec_keep_ratio:
            keep_secondary = True
            secondary_overlap_sec = raw2

    out["speaker"] = primary
    out["secondary_speaker"] = secondary if keep_secondary else None
    out["secondary_overlap_sec"] = safe_round(secondary_overlap_sec, 4)
    out["speaker_confidence"] = safe_round(confidence, 4)
    out["has_overlap_region"] = bool(has_overlap)
    out["speaker_uncertain"] = bool(
        primary_overlap_ratio < (0.22 if weak_item else 0.18)
        or confidence < 0.58
        or (secondary is not None and margin < (0.14 if weak_item else 0.10))
    )
    if fast_handoff:
        out["speaker_reason"] = "short_fast_handoff_preserved"
    else:
        out["speaker_reason"] = "tail_or_short_balanced" if weak_item else "direct_best_score"
    out["_speaker_candidates"] = speaker_candidates
    out["track"] = 0
    return out


def assign_speakers(items, speaker_segments):
    out = []
    prev = None
    for it in sort_items(items):
        cur = assign_speaker_for_item(it, speaker_segments, prev_speaker=prev)
        prev = cur.get("speaker")
        out.append(cur)
    return out


def _speaker_item_is_weak(it):
    return (
        bool(it.get("speaker_uncertain", False))
        or float(it.get("speaker_confidence", 0.0) or 0.0) < 0.72
        or bool(it.get("is_tail_rescue", False))
        or item_dur(it) <= 1.6
        or text_len(it.get("text", "")) <= 6
    )


def _candidate_score(it, speaker):
    for cand in it.get("_speaker_candidates", []) or []:
        if cand.get("speaker") == speaker:
            return float(cand.get("score", 0.0) or 0.0)
    return -1e9


def _should_preserve_interrupt(items, idx, prev_spk, next_spk):
    cur = items[idx]
    if item_dur(cur) > SHORT_REACTION_MAX_DUR:
        return False
    if cur.get("secondary_speaker") or cur.get("has_overlap_region"):
        return True
    cur_spk = cur.get("speaker")
    prev_score = _candidate_score(cur, prev_spk)
    next_score = _candidate_score(cur, next_spk) if next_spk else -1e9
    cur_score = _candidate_score(cur, cur_spk)
    if next_spk and next_spk != prev_spk and next_score >= (prev_score - SHORT_REACTION_SCORE_MARGIN):
        return True
    if cur_spk and cur_spk != prev_spk and cur_score >= (prev_score - SHORT_REACTION_SCORE_MARGIN):
        return True
    return False


def _neighbor_stable_speaker(items, idx, direction, max_steps=4):
    j = idx + direction
    steps = 0
    while 0 <= j < len(items) and steps < max_steps:
        spk = items[j].get("speaker")
        if spk and not _speaker_item_is_weak(items[j]):
            return spk
        j += direction
        steps += 1
    return None


def smooth_balanced_speakers(items):
    out = [copy_item(x) for x in sort_items(items)]
    rewrites = 0

    for i in range(len(out)):
        cur = out[i]
        if not _speaker_item_is_weak(cur):
            continue
        prev_stable = _neighbor_stable_speaker(out, i, -1, max_steps=4)
        next_stable = _neighbor_stable_speaker(out, i, 1, max_steps=4)
        cur_spk = cur.get("speaker")
        dur = item_dur(cur)

        if prev_stable and next_stable and prev_stable != next_stable and _should_preserve_interrupt(out, i, prev_stable, next_stable):
            continue

        if prev_stable and next_stable and prev_stable == next_stable:
            if cur_spk != prev_stable:
                out[i]["speaker"] = prev_stable
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "balanced_bridge_prev_next_same"
                rewrites += 1
            continue

        if dur <= 1.5 and prev_stable:
            if next_stable and next_stable != prev_stable and _should_preserve_interrupt(out, i, prev_stable, next_stable):
                continue
            cur_score = _candidate_score(cur, cur_spk)
            prev_score = _candidate_score(cur, prev_stable)
            if prev_score >= (cur_score - 0.10) and cur_spk != prev_stable:
                out[i]["speaker"] = prev_stable
                out[i]["speaker_uncertain"] = False
                out[i]["speaker_reason"] = "balanced_short_to_prev"
                rewrites += 1

    for i in range(1, len(out) - 1):
        prev_it = out[i - 1]
        cur_it = out[i]
        next_it = out[i + 1]
        prev_spk = prev_it.get("speaker")
        next_spk = next_it.get("speaker")
        if not prev_spk or prev_spk != next_spk:
            continue
        if not _speaker_item_is_weak(cur_it):
            continue
        if item_dur(cur_it) >= 1.8:
            continue
        if _should_preserve_interrupt(out, i, prev_spk, next_spk):
            continue
        cur_spk = cur_it.get("speaker")
        if _candidate_score(cur_it, prev_spk) >= (_candidate_score(cur_it, cur_spk) - 0.12) and cur_spk != prev_spk:
            out[i]["speaker"] = prev_spk
            out[i]["speaker_uncertain"] = False
            out[i]["speaker_reason"] = "balanced_aba_smoother"
            rewrites += 1

    globals()["_speaker_smoothed"] = rewrites
    return out


def make_secondary_overlap_items(items):
    if not ALLOW_CROSS_SPEAKER_OVERLAP:
        return items

    out = []
    made = 0
    for it in items:
        out.append(copy_item(it))
        sec = it.get("secondary_speaker")
        if not sec:
            continue
        if float(it.get("secondary_overlap_sec", 0.0) or 0.0) < MIN_MEANINGFUL_OVERLAP * 0.75:
            continue
        dup = copy_item(it)
        dup["speaker"] = sec
        dup["primary_speaker"] = it.get("speaker")
        dup["is_secondary_overlap"] = True
        dup["track"] = 1 if ENABLE_MULTI_TRACK_ASS else 0
        out.append(dup)
        made += 1

    globals()["_overlap_secondary_made"] = made
    return sort_items(out)


In [ ]:
#@title **merge / cleanup 工具**
# =========================================================
# merge
# =========================================================
def merged_text_len(a, b):
    return text_len(merge_text(a.get("text", ""), b.get("text", "")))

def same_speaker(a, b):
    return norm_speaker_key(a.get("speaker")) == norm_speaker_key(b.get("speaker"))

def gap_between(a, b):
    return _f(b.get("start")) - _f(a.get("end"))

def item_has_strong_overlap_flag(it):
    return bool(it.get("has_overlap_region")) or bool(it.get("secondary_speaker")) or bool(it.get("is_secondary_overlap"))

def final_merge_char_cap():
    return max(1, min(int(MERGED_MAX_CHARS), int(MAX_CHARS_PER_CHUNK), int(MAX_LINE_LEN * MAX_LINES)))

def dependent_merge_char_cap():
    return max(1, min(18, final_merge_char_cap()))

def ends_with_hard_sentence_punct(text):
    return clean_compact_text(text).endswith(("。", "！", "？", "!", "?"))

def is_question_sentence(text):
    t = clean_compact_text(text)
    if not t:
        return False
    if t.endswith(("？", "?")):
        return True
    return bool(re.search(r"(ですか|ますか|でしたか|ませんか|ないですか|でしょうか|だろうか|かな|かね)$", t))

def merged_total_dur(a, b):
    return max(0.0, max(_f(a.get("end")), _f(b.get("end"))) - min(_f(a.get("start")), _f(b.get("start"))))

def merged_cps(a, b):
    dur = max(0.01, merged_total_dur(a, b))
    return merged_text_len(a, b) / dur

def can_merge_same_speaker(a, b):
    return False

def can_merge_dependent_short_fragment(prev_it, cur_it):
    if not same_speaker(prev_it, cur_it):
        return False
    g = gap_between(prev_it, cur_it)
    if g < -0.05 or g > min(MAX_GAP_MERGE_SAME, 0.10):
        return False
    if item_has_strong_overlap_flag(prev_it) or item_has_strong_overlap_flag(cur_it):
        return False

    prev_text = prev_it.get("text", "")
    cur_text = cur_it.get("text", "")
    if is_empty_text(prev_text) or is_empty_text(cur_text):
        return False
    if looks_like_independent_short_utt(cur_text):
        return False
    if ends_with_hard_sentence_punct(prev_text):
        return False
    if is_question_sentence(prev_text):
        return False
    if text_len(prev_text) >= 16:
        return False
    if item_dur(cur_it) > 0.9:
        return False
    if text_len(cur_text) <= 4:
        return False
    if text_len(cur_text) > 8:
        return False
    if merged_text_len(prev_it, cur_it) > dependent_merge_char_cap():
        return False
    if merged_total_dur(prev_it, cur_it) > min(MERGED_MAX_DUR, 3.2):
        return False
    if merged_cps(prev_it, cur_it) > MAX_READ_CPS:
        return False

    return True

def can_merge_tail_fragment(prev_it, cur_it):
    if not same_speaker(prev_it, cur_it):
        return False
    g = gap_between(prev_it, cur_it)
    if g < -0.05 or g > min(MAX_GAP_MERGE_TAIL, 0.12):
        return False
    if item_has_strong_overlap_flag(prev_it) or item_has_strong_overlap_flag(cur_it):
        return False

    prev_text = prev_it.get("text", "")
    cur_text = cur_it.get("text", "")
    if is_empty_text(prev_text) or is_empty_text(cur_text):
        return False
    if item_dur(cur_it) > min(TAIL_FRAGMENT_MAX_DUR, 0.55):
        return False
    if text_len(cur_text) > min(TAIL_FRAGMENT_MAX_CHARS, 4):
        return False
    if looks_like_independent_short_utt(cur_text):
        return False
    if ends_with_hard_sentence_punct(prev_text):
        return False
    if item_dur(prev_it) >= 2.4:
        return False
    if merged_text_len(prev_it, cur_it) > final_merge_char_cap():
        return False
    if merged_total_dur(prev_it, cur_it) > min(MERGED_MAX_DUR, MAX_SENT_DUR, 4.0):
        return False
    if merged_cps(prev_it, cur_it) > MAX_READ_CPS:
        return False

    return True

def merge_two_items(a, b):
    out = copy_item(a)
    out["start"] = min(_f(a.get("start")), _f(b.get("start")))
    out["end"] = max(_f(a.get("end")), _f(b.get("end")))
    out["text"] = merge_text(a.get("text", ""), b.get("text", ""))
    out["speaker"] = a.get("speaker") or b.get("speaker")
    out["secondary_speaker"] = None
    out["has_overlap_region"] = bool(a.get("has_overlap_region")) or bool(b.get("has_overlap_region"))
    out["speaker_confidence"] = max(_f(a.get("speaker_confidence", 0.0)), _f(b.get("speaker_confidence", 0.0)))
    out["speaker_uncertain"] = bool(a.get("speaker_uncertain", False)) and bool(b.get("speaker_uncertain", False))
    out["is_tail_rescue"] = bool(a.get("is_tail_rescue", False)) or bool(b.get("is_tail_rescue", False))
    out["tail_rescue_source"] = a.get("tail_rescue_source") or b.get("tail_rescue_source")
    out["track"] = int(a.get("track", 0))
    out["merged_count"] = int(a.get("merged_count", 1)) + int(b.get("merged_count", 1))
    return out

def merge_same_speaker_fragments(items):
    globals()["_same_speaker_merged"] = 0
    return sort_items(items) if items else items

def merge_dependent_short_fragments(items):
    if not items:
        globals()["_dependent_short_merged"] = 0
        return items
    items = sort_items(items)
    out = []
    merged_n = 0

    for cur in items:
        if out and can_merge_dependent_short_fragment(out[-1], cur):
            out[-1] = merge_two_items(out[-1], cur)
            merged_n += 1
        else:
            out.append(copy_item(cur))

    globals()["_dependent_short_merged"] = merged_n
    globals()["_same_speaker_merged"] = 0
    return out

def merge_tail_fragments(items):
    if not items:
        return items
    items = sort_items(items)
    out = []
    merged_n = 0

    for cur in items:
        if out and can_merge_tail_fragment(out[-1], cur):
            out[-1] = merge_two_items(out[-1], cur)
            merged_n += 1
        else:
            out.append(copy_item(cur))

    globals()["_tail_fragment_merged"] = merged_n
    return out


# =========================================================
# cleanup / boundary / tail rescue
# =========================================================
def cleanup_items(items, audio_len):
    out = []
    dropped_empty = 0
    dropped_bad = 0
    dropped_hotword = 0
    dropped_duplicate = 0

    for it in sort_items(items):
        x = copy_item(it)
        x["text"] = clean_compact_text(x.get("text", ""))

        if is_empty_text(x["text"]):
            dropped_empty += 1
            continue

        x["start"] = max(0.0, min(_f(audio_len), _f(x.get("start"))))
        x["end"] = max(x["start"], min(_f(audio_len), _f(x.get("end"))))

        if x["end"] - x["start"] < 0.06:
            dropped_bad += 1
            continue

        recent_texts = [y.get("text", "") for y in out[-8:]]
        if is_hotword_hallucination_text(x["text"], x["start"], x["end"], recent_texts=recent_texts, strict=True):
            dropped_hotword += 1
            continue
        if recent_texts and hotword_cover_ratio(x["text"]) >= 0.55:
            if any(text_similarity_loose(x["text"], rt) >= 0.90 for rt in recent_texts[-4:]):
                dropped_duplicate += 1
                continue

        out.append(x)

    globals()["_dropped_empty"] = dropped_empty
    globals()["_dropped_bad"] = dropped_bad
    globals()["_dropped_hotword_hallucination"] = dropped_hotword
    globals()["_dropped_hotword_duplicate"] = dropped_duplicate
    return out

def clamp_and_fix_order(items, audio_len):
    out = []
    for it in sort_items(items):
        x = copy_item(it)
        x["start"] = max(0.0, min(_f(audio_len), _f(x.get("start"))))
        x["end"] = max(x["start"], min(_f(audio_len), _f(x.get("end"))))
        if x["end"] - x["start"] < 0.06:
            x["end"] = min(_f(audio_len), x["start"] + 0.06)
        out.append(x)

    if not ENABLE_GLOBAL_CONFLICT_RESOLVE:
        return out

    for i in range(len(out) - 1):
        a = out[i]
        b = out[i + 1]
        must_separate = same_speaker(a, b) or not ALLOW_CROSS_SPEAKER_OVERLAP
        if not must_separate:
            ov = _f(a["end"]) - _f(b["start"])
            if ov > 0.90:
                shrink = min(0.10, ov / 2.0)
                a["end"] = max(_f(a["start"]) + 0.06, _f(a["end"]) - shrink)
                b["start"] = min(_f(b["end"]) - 0.06, _f(b["start"]) + shrink)
            continue

        limit = _f(b["start"]) - SAFE_GAP_TO_NEXT
        if _f(a["end"]) <= limit:
            continue

        overlap = _f(a["end"]) - limit
        movable = max(0.0, _f(b["end"]) - (_f(b["start"]) + 0.06))
        shift_b = min(MAX_CONFLICT_SHIFT_NEXT, movable, overlap)
        if shift_b > 0:
            b["start"] = min(_f(b["end"]) - 0.06, _f(b["start"]) + shift_b)

        limit = _f(b["start"]) - SAFE_GAP_TO_NEXT
        if _f(a["end"]) <= limit:
            continue

        remain = _f(a["end"]) - limit
        cur_dur = max(0.0, _f(a["end"]) - _f(a["start"]))
        if not ends_with_hard_sentence_punct(a.get("text", "")) and cur_dur >= 1.2:
            trim_cap = min(MAX_CONFLICT_TRIM_CUR, 0.05)
        else:
            trim_cap = MAX_CONFLICT_TRIM_CUR
        trim_a = min(trim_cap, remain)
        if trim_a > 0:
            a["end"] = max(_f(a["start"]) + 0.06, _f(a["end"]) - trim_a)

        limit = _f(b["start"]) - SAFE_GAP_TO_NEXT
        if _f(a["end"]) > limit:
            a["end"] = max(_f(a["start"]) + 0.06, limit)

    return sort_items(out)

def find_next_same_track(items, idx):
    cur_track = int(items[idx].get("track", 0))
    for j in range(idx + 1, len(items)):
        if int(items[j].get("track", 0)) == cur_track:
            return items[j]
    return None

def extend_tail_safely(items, speaker_segments, audio_len):
    if not items:
        return items
    out = [copy_item(x) for x in sort_items(items)]

    for i in range(len(out)):
        cur = out[i]
        nxt = find_next_same_track(out, i)

        cur_start = _f(cur["start"])
        cur_end = _f(cur["end"])
        cur_spk = norm_speaker_key(cur.get("speaker"))
        cur_dur = max(0.0, cur_end - cur_start)

        if nxt is None:
            want = min(_f(audio_len), cur_end + LAST_ITEM_EXTRA_PAD)
            cur["end"] = max(cur_end, want)
            continue

        nxt_start = _f(nxt["start"])
        nxt_spk = norm_speaker_key(nxt.get("speaker"))
        max_stop = max(cur_end, nxt_start - SAFE_GAP_TO_NEXT)
        if max_stop <= cur_end:
            continue

        if cur_spk == nxt_spk:
            ext = min(TAIL_RESCUE_SAME_SPK, max_stop - cur_end)
        else:
            if ALLOW_CROSS_SPEAKER_OVERLAP and cur.get("has_overlap_region"):
                ext = min(TAIL_RESCUE_SINGLE_SPK, max_stop - cur_end)
            elif ALLOW_CROSS_SPEAKER_OVERLAP and nxt_start - cur_end > 0.12:
                ext = min(TAIL_RESCUE_DIFF_SPK, max_stop - cur_end)
            else:
                ext = min(0.04, max_stop - cur_end)

        if (nxt_start - cur_end) < 0.50 and not ends_with_hard_sentence_punct(cur.get("text", "")):
            ext = max(ext, min(max(MIN_TAIL_HOLD, 0.14), max_stop - cur_end))

        if cur_dur >= 1.2 and not ends_with_hard_sentence_punct(cur.get("text", "")):
            ext = max(ext, min(0.14, max_stop - cur_end))

        ext = min(ext, TAIL_RESCUE_MAX_EXTEND)
        if ext > 0:
            cur["end"] = cur_end + ext

        if (cur["end"] - cur_start) < MIN_SENT_DUR:
            need = MIN_SENT_DUR - (cur["end"] - cur_start)
            cur["end"] = min(max_stop, cur["end"] + max(0.0, need))

    return sort_items(out)


# =========================================================
# 主流程
# =========================================================
def reset_final_merge_counters():
    globals()["_same_speaker_merged"] = 0
    globals()["_dependent_short_merged"] = 0
    globals()["_tail_fragment_merged"] = 0
    globals()["_pre_assign_split_items"] = 0
    globals()["_post_assign_split_items"] = 0
    globals()["_same_speaker_overlap_fixed"] = 0
    globals()["_unsafe_split_rejected"] = 0
    globals()["_tiny_final_items_merged"] = 0
    globals()["_tiny_final_items_dropped"] = 0
    globals()["_boundary_rebalance_count"] = 0
    globals()["_early_start_fixed"] = 0
    globals()["_speaker_smoothed"] = 0
    globals()["_tail_fragments_merged_or_dropped"] = 0
    globals()["_post_split_speaker_repair"] = 0
    globals()["_early_end_protected"] = 0
    globals()["_text_duration_mismatch_fixed"] = 0
    globals()["_dropped_hotword_hallucination"] = 0
    globals()["_dropped_hotword_duplicate"] = 0

def enforce_same_speaker_non_overlap(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    for i in range(len(out) - 1):
        a = out[i]
        b = out[i + 1]
        if int(a.get("track", 0)) != int(b.get("track", 0)):
            continue
        if norm_speaker_key(a.get("speaker")) != norm_speaker_key(b.get("speaker")):
            continue

        a_start = _f(a.get("start"))
        a_end = _f(a.get("end"))
        b_start = _f(b.get("start"))
        b_end = _f(b.get("end"))
        need_start = a_end
        if b_start >= need_start:
            continue

        fixed += 1
        movable_b = max(0.0, b_end - (b_start + 0.06))
        shift_b = min(need_start - b_start, movable_b)
        if shift_b > 0:
            b_start += shift_b

        need_start = a_end
        if b_start < need_start:
            a_end = max(a_start + 0.06, min(a_end, b_start))

        need_start = a_end
        if b_start < need_start:
            b_start = min(b_end - 0.06, need_start)

        a["end"] = min(_f(audio_len), max(a_start + 0.06, a_end))
        b["start"] = max(0.0, min(b_end - 0.06, b_start))

    globals()["_same_speaker_overlap_fixed"] = fixed
    return sort_items(out)


def normalize_micro_gaps_and_overlaps(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    if not out:
        return out

    for i in range(len(out) - 1):
        a = out[i]
        b = out[i + 1]
        if int(a.get("track", 0)) != int(b.get("track", 0)):
            continue

        a_start = _f(a.get("start"))
        a_end = _f(a.get("end"))
        b_start = _f(b.get("start"))
        b_end = _f(b.get("end"))
        gap = b_start - a_end

        if 0.0 < gap < COLLAPSE_MICRO_GAP_BELOW:
            b_start = max(0.0, min(b_end - 0.06, a_end))

        elif norm_speaker_key(a.get("speaker")) != norm_speaker_key(b.get("speaker")) and gap < 0.0 and abs(gap) < MIN_MEANINGFUL_OVERLAP:
            target_start = a_end
            movable_b = max(0.0, b_end - (b_start + 0.06))
            shift_b = min(target_start - b_start, movable_b)
            if shift_b > 0:
                b_start += shift_b
            if b_start < a_end:
                a_end = max(a_start + 0.06, min(a_end, b_start))
            if b_start < a_end:
                b_start = min(b_end - 0.06, a_end)

        a["end"] = min(_f(audio_len), max(a_start + 0.06, a_end))
        b["start"] = max(0.0, min(b_end - 0.06, b_start))

    return sort_items(out)


def fix_obviously_early_starts(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    for i in range(len(out) - 1):
        a = out[i]
        b = out[i + 1]
        if int(a.get("track", 0)) != int(b.get("track", 0)):
            continue
        if bool(a.get("is_overlap")) or bool(b.get("is_overlap")) or bool(a.get("is_secondary_overlap")) or bool(b.get("is_secondary_overlap")):
            continue
        a_end = _f(a.get("end"))
        b_start = _f(b.get("start"))
        b_end = _f(b.get("end"))
        gap = b_start - a_end
        weak_b = bool(b.get("is_tail_rescue", False)) or bool(b.get("speaker_uncertain", False)) or item_dur(b) <= 1.6 or text_len(b.get("text", "")) <= 6
        if gap < -0.06 and (weak_b or not ends_with_hard_sentence_punct(a.get("text", ""))):
            target_start = a_end
        elif bool(b.get("is_tail_rescue", False)) and 0.0 <= gap < COLLAPSE_MICRO_GAP_BELOW and not ends_with_hard_sentence_punct(a.get("text", "")):
            target_start = a_end
        else:
            continue
        target_start = min(b_end - 0.06, target_start)
        if target_start <= b_start + 1e-6:
            continue
        b["start"] = max(0.0, min(_f(audio_len), target_start))
        fixed += 1
    globals()["_early_start_fixed"] = fixed
    return sort_items(out)


def rebalance_sentence_boundaries(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    for i in range(len(out) - 1):
        a = out[i]
        b = out[i + 1]
        if int(a.get("track", 0)) != int(b.get("track", 0)):
            continue
        gap = _f(b.get("start")) - _f(a.get("end"))
        if 0.0 < gap < COLLAPSE_MICRO_GAP_BELOW:
            b["start"] = _f(a.get("end"))
            fixed += 1
            continue
        if gap < MIN_INTER_SENT_GAP or gap > 0.20:
            continue
        if ends_with_hard_sentence_punct(a.get("text", "")):
            continue
        give_back = min(0.10, gap - MIN_INTER_SENT_GAP)
        if give_back <= 0:
            continue
        a["end"] = min(_f(audio_len), _f(a.get("end")) + give_back)
        fixed += 1
    globals()["_boundary_rebalance_count"] = fixed
    return sort_items(out)


In [ ]:
#@title **finalize 函数**
def readable_min_duration_for_item(item):
    if bool(item.get("is_overlap")) or bool(item.get("is_secondary_overlap")):
        return max(0.34, float(globals().get("MIN_FINAL_OVERLAP_DUR", 0.34)))
    chars = text_len(item.get("text", ""))
    if chars <= 0:
        return max(MIN_FINAL_PRIMARY_DUR, 0.60)
    if chars <= 2:
        return max(MIN_FINAL_PRIMARY_DUR, float(globals().get("MIN_READABLE_TINY_DUR", 0.65)))
    if chars <= 6:
        return max(MIN_FINAL_PRIMARY_DUR, float(globals().get("MIN_READABLE_SHORT_DUR", 0.78)))
    cps = float(globals().get("READABLE_MIN_CPS", 12.0))
    return min(float(globals().get("READABLE_MAX_MIN_DUR", 2.8)), max(0.80, chars / max(cps, 1.0)))


def cleanup_tiny_final_items(items, audio_len):
    out = []
    merged = 0
    dropped = 0
    pending = [copy_item(x) for x in sort_items(items)]

    def _can_absorb(dst, src):
        if norm_speaker_key(dst.get("speaker")) != norm_speaker_key(src.get("speaker")):
            return False
        if int(dst.get("track", 0)) != int(src.get("track", 0)):
            return False
        merged_dur = max(_f(dst.get("end")), _f(src.get("end"))) - min(_f(dst.get("start")), _f(src.get("start")))
        merged_chars = text_len(merge_text(dst.get("text", ""), src.get("text", "")))
        return merged_dur <= 2.8 and merged_chars <= 18

    i = 0
    while i < len(pending):
        cur = pending[i]
        cur_dur = item_dur(cur)
        cur_chars = text_len(cur.get("text", ""))
        min_readable_dur = readable_min_duration_for_item(cur)
        if cur.get("is_secondary_overlap"):
            out.append(cur)
            i += 1
            continue
        if cur_dur >= min_readable_dur and cur_chars >= MIN_FINAL_PRIMARY_CHARS:
            out.append(cur)
            i += 1
            continue
        if out and _can_absorb(out[-1], cur) and (_f(cur.get("start")) - _f(out[-1].get("end"))) <= 0.18:
            out[-1] = merge_two_items(out[-1], cur)
            merged += 1
            i += 1
            continue
        if i + 1 < len(pending) and _can_absorb(cur, pending[i + 1]) and (_f(pending[i + 1].get("start")) - _f(cur.get("end"))) <= 0.18:
            pending[i + 1] = merge_two_items(cur, pending[i + 1])
            merged += 1
            i += 1
            continue
        next_start = _f(audio_len) if i + 1 >= len(pending) else _f(pending[i + 1].get("start"))
        right_room = max(0.0, next_start - _f(cur.get("end")) - MIN_INTER_SENT_GAP)
        cur["end"] = min(_f(audio_len), _f(cur.get("end")) + min(right_room, max(0.0, min_readable_dur - cur_dur)))
        if item_dur(cur) < min_readable_dur and cur_chars <= 2 and float(cur.get("speaker_confidence", 0.0)) < 0.65:
            dropped += 1
            i += 1
            continue
        out.append(cur)
        i += 1

    globals()["_tiny_final_items_merged"] = merged
    globals()["_tiny_final_items_dropped"] = dropped
    globals()["_tail_fragments_merged_or_dropped"] = merged + dropped
    return sort_items(out)


def enforce_readable_item_durations(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    merged = 0
    for i, cur in enumerate(out):
        if bool(cur.get("is_secondary_overlap")):
            continue
        min_dur = readable_min_duration_for_item(cur)
        if item_dur(cur) >= min_dur:
            continue
        prev_it = out[i - 1] if i > 0 and int(out[i - 1].get("track", 0)) == int(cur.get("track", 0)) else None
        next_it = out[i + 1] if i + 1 < len(out) and int(out[i + 1].get("track", 0)) == int(cur.get("track", 0)) else None
        prev_end = _f(prev_it.get("end")) if prev_it else 0.0
        next_start = _f(next_it.get("start")) if next_it else _f(audio_len)
        need = max(0.0, min_dur - item_dur(cur))
        right_room = max(0.0, next_start - _f(cur.get("end")) - MIN_INTER_SENT_GAP)
        add_right = min(right_room, need)
        if add_right > 0:
            cur["end"] = min(_f(audio_len), _f(cur.get("end")) + add_right)
            need -= add_right
        left_room = max(0.0, _f(cur.get("start")) - prev_end - MIN_INTER_SENT_GAP)
        add_left = min(left_room, need, float(globals().get("READABLE_MAX_LEFT_EXTEND", 0.25)))
        if add_left > 0:
            cur["start"] = max(0.0, _f(cur.get("start")) - add_left)
            need -= add_left
        if add_right > 0 or add_left > 0:
            fixed += 1
        if item_dur(cur) >= min_dur:
            continue
        if prev_it and norm_speaker_key(prev_it.get("speaker")) == norm_speaker_key(cur.get("speaker")):
            merged_dur = max(_f(prev_it.get("end")), _f(cur.get("end"))) - min(_f(prev_it.get("start")), _f(cur.get("start")))
            merged_chars = text_len(merge_text(prev_it.get("text", ""), cur.get("text", "")))
            if merged_dur <= 3.0 and merged_chars <= 22:
                out[i - 1] = merge_two_items(prev_it, cur)
                cur["_drop_after_readable_repair"] = True
                merged += 1
                continue
        if next_it and norm_speaker_key(next_it.get("speaker")) == norm_speaker_key(cur.get("speaker")):
            merged_dur = max(_f(next_it.get("end")), _f(cur.get("end"))) - min(_f(next_it.get("start")), _f(cur.get("start")))
            merged_chars = text_len(merge_text(cur.get("text", ""), next_it.get("text", "")))
            if merged_dur <= 3.0 and merged_chars <= 22:
                out[i + 1] = merge_two_items(cur, next_it)
                cur["_drop_after_readable_repair"] = True
                merged += 1
    out = [x for x in out if not x.get("_drop_after_readable_repair")]
    globals()["_readable_duration_fixed"] = fixed
    globals()["_readable_duration_merged"] = merged
    return sort_items(out)


def _audio_clip_activity_stats(start, end):
    if "wav" not in globals() or "sr" not in globals():
        return {"audio_active": None, "rms_db": None, "peak_db": None, "duration": safe_round(max(0.0, end - start), 3)}
    start = max(0.0, float(start))
    end = max(start, float(end))
    a = max(0, int(start * sr))
    b = min(len(wav), int(end * sr))
    if b <= a:
        return {"audio_active": False, "rms_db": -120.0, "peak_db": -120.0, "duration": safe_round(end - start, 3)}
    clip = wav[a:b]
    rms = float((clip * clip).mean() ** 0.5)
    peak = float(abs(clip).max())
    def _db(v):
        return -120.0 if v <= 1e-9 else 20.0 * math.log10(v)
    rms_db = _db(rms)
    peak_db = _db(peak)
    rms_th = float(globals().get("SUBTITLE_GAP_RESCUE_RMS_DB", -45.0))
    peak_th = float(globals().get("SUBTITLE_GAP_RESCUE_PEAK_DB", -30.0))
    return {
        "audio_active": bool(rms_db >= rms_th or peak_db >= peak_th),
        "rms_db": safe_round(rms_db, 2),
        "peak_db": safe_round(peak_db, 2),
        "duration": safe_round(end - start, 3),
    }


def _item_has_timed_support(item):
    try:
        return bool(_timed_tokens_for_item(item))
    except Exception:
        return bool(item.get("tokens") or item.get("whisper_words"))


def _clip_transcription_matches_text(start, end, text):
    if "transcribe_clip" not in globals() or "wav" not in globals() or "sr" not in globals():
        return True, "transcribe_unavailable"
    audio_cap = globals().get("AUDIO_LEN", globals().get("audio_len", end)) or end
    clip_l = max(0.0, float(start) - 0.35)
    clip_r = min(float(audio_cap), float(end) + 0.35)
    if clip_r - clip_l < 0.25:
        return False, "clip_too_short"
    try:
        segs = transcribe_clip(wav[int(clip_l * sr):int(clip_r * sr)], hotwords="")
    except Exception as e:
        return True, f"transcribe_error:{type(e).__name__}"
    joined = "".join((getattr(x, "text", "") or "").strip() for x in (segs or []))
    if not compact_text(joined):
        return False, "isolated_empty"
    sim = text_similarity_loose(text, joined)
    if sim >= float(globals().get("GAP_EDGE_HALLUCINATION_MIN_SIM", 0.42)) or compact_text(text) in compact_text(joined):
        return True, "isolated_match"
    return False, f"isolated_mismatch:{safe_round(sim, 3)}"


def drop_unverified_gap_edge_hallucinations(items, audio_len):
    if not bool(globals().get("GAP_EDGE_HALLUCINATION_FILTER_ENABLE", True)):
        globals()["_gap_edge_hallucination_dropped"] = 0
        return sort_items(items)
    ordered = [copy_item(x) for x in sort_items(items)]
    out = []
    dropped = 0
    checked = 0
    threshold = float(globals().get("SUBTITLE_GAP_DIAG_THRESHOLD_SEC", 5.0))
    max_checks = int(globals().get("GAP_EDGE_HALLUCINATION_MAX_CHECKS", 24))
    for i, cur in enumerate(ordered):
        prev_it = ordered[i - 1] if i > 0 and int(ordered[i - 1].get("track", 0)) == int(cur.get("track", 0)) else None
        next_it = ordered[i + 1] if i + 1 < len(ordered) and int(ordered[i + 1].get("track", 0)) == int(cur.get("track", 0)) else None
        prev_gap = _f(cur.get("start")) - _f(prev_it.get("end")) if prev_it else 0.0
        next_gap = _f(next_it.get("start")) - _f(cur.get("end")) if next_it else 0.0
        near_large_gap = max(prev_gap, next_gap) >= threshold
        weak = item_dur(cur) <= 1.25 or text_len(cur.get("text", "")) <= 8 or not _item_has_timed_support(cur)
        if not (near_large_gap and weak and not cur.get("is_gap_rescue")):
            out.append(cur)
            continue
        if not _item_has_timed_support(cur):
            cur["_drop_reason"] = "no_timed_support_near_large_gap"
            dropped += 1
            continue
        if checked < max_checks:
            checked += 1
            ok, reason = _clip_transcription_matches_text(cur.get("start"), cur.get("end"), cur.get("text", ""))
            if not ok:
                cur["_drop_reason"] = reason
                dropped += 1
                continue
        out.append(cur)
    globals()["_gap_edge_hallucination_dropped"] = dropped
    return sort_items(out)


def _whisper_words_from_segment(seg, clip_l):
    words = []
    for w in (getattr(seg, "words", None) or []):
        if getattr(w, "start", None) is None or getattr(w, "end", None) is None:
            continue
        txt = (getattr(w, "word", "") or "").strip()
        if not txt:
            continue
        start = float(getattr(w, "start")) + clip_l
        end = float(getattr(w, "end")) + clip_l
        if end <= start:
            continue
        words.append({"text": txt, "start": start, "end": end})
    return words


def _gap_rescue_segments_hotword_suspect(segs, clip_l, recent_texts):
    return any(
        is_hotword_hallucination_text(
            (getattr(x, "text", "") or "").strip(),
            float(getattr(x, "start", 0.0)) + clip_l,
            float(getattr(x, "end", 0.0)) + clip_l,
            recent_texts=recent_texts,
            strict=True,
        )
        for x in (segs or [])
    )


def rescue_internal_speech_gaps(final_items, source_sentence_items, audio_len):
    diagnostics = []
    globals()["_gap_rescue_added"] = 0
    if not bool(globals().get("INTERNAL_GAP_RESCUE_ENABLE", True)):
        return [copy_item(x) for x in source_sentence_items], diagnostics
    if "transcribe_clip" not in globals() or "wav" not in globals() or "sr" not in globals():
        return [copy_item(x) for x in source_sentence_items], diagnostics

    threshold = float(globals().get("SUBTITLE_GAP_RESCUE_THRESHOLD_SEC", globals().get("SUBTITLE_GAP_DIAG_THRESHOLD_SEC", 5.0)))
    context = float(globals().get("SUBTITLE_GAP_RESCUE_CONTEXT_SEC", 0.35))
    min_clip = float(globals().get("SUBTITLE_GAP_RESCUE_MIN_CLIP_SEC", 1.0))
    min_seg_dur = float(globals().get("SUBTITLE_GAP_RESCUE_MIN_SEG_DUR", 0.45))
    max_gaps = int(globals().get("SUBTITLE_GAP_RESCUE_MAX_GAPS", 0))
    ordered = [copy_item(x) for x in sort_items(final_items) if int(x.get("track", 0)) == 0]
    augmented = [copy_item(x) for x in source_sentence_items]
    attempted = 0
    added = 0

    for idx in range(len(ordered) - 1):
        prev_it = ordered[idx]
        next_it = ordered[idx + 1]
        gap_start = _f(prev_it.get("end"))
        gap_end = _f(next_it.get("start"))
        gap = gap_end - gap_start
        if gap < threshold:
            continue
        if max_gaps > 0 and attempted >= max_gaps:
            break
        safe_start = max(0.0, gap_start + 0.05)
        safe_end = min(_f(audio_len), gap_end - 0.05)
        row = {
            "index": len(diagnostics),
            "prev_item_index": idx,
            "next_item_index": idx + 1,
            "gap_sec": safe_round(gap, 3),
            "gap_start": safe_round(gap_start, 3),
            "gap_end": safe_round(gap_end, 3),
            "prev_text": visible_text(prev_it.get("text", ""))[:80],
            "next_text": visible_text(next_it.get("text", ""))[:80],
            "rescue_attempted": False,
            "rescued_items": 0,
            "rescue_reject_reason": None,
        }
        row.update(_audio_clip_activity_stats(safe_start, safe_end))
        if safe_end - safe_start < min_clip:
            row["rescue_reject_reason"] = "gap_too_short_after_padding"
            diagnostics.append(row)
            continue
        if row.get("audio_active") is False:
            row["rescue_reject_reason"] = "inactive_audio"
            diagnostics.append(row)
            continue

        attempted += 1
        row["rescue_attempted"] = True
        clip_l = max(0.0, safe_start - context)
        clip_r = min(_f(audio_len), safe_end + context)
        clip_wav = wav[int(clip_l * sr):int(clip_r * sr)]
        recent_texts = [
            x.get("text", "")
            for x in ordered[max(0, idx - 3):min(len(ordered), idx + 5)]
        ]

        try:
            rescue_segments = transcribe_clip(clip_wav)
        except Exception as e:
            row["rescue_reject_reason"] = f"transcribe_error:{type(e).__name__}"
            diagnostics.append(row)
            continue

        if hotwords_compact and (not rescue_segments or _gap_rescue_segments_hotword_suspect(rescue_segments, clip_l, recent_texts)):
            try:
                no_hotword_segments = transcribe_clip(clip_wav, hotwords="")
                if no_hotword_segments:
                    rescue_segments = no_hotword_segments
                    row["retried_without_hotwords"] = True
            except Exception as e:
                row["no_hotwords_error"] = f"{type(e).__name__}: {e}"

        accepted = []
        rejected_reasons = Counter()
        for rs in rescue_segments or []:
            rs_text = (getattr(rs, "text", "") or "").strip()
            rs_start = float(getattr(rs, "start", 0.0)) + clip_l
            rs_end = float(getattr(rs, "end", 0.0)) + clip_l
            start = max(safe_start, rs_start)
            end = min(safe_end, rs_end)
            if not rs_text:
                rejected_reasons["empty_text"] += 1
                continue
            if end - start < min_seg_dur:
                rejected_reasons["too_short"] += 1
                continue
            rs_compact = compact_text(rs_text)
            if rs_compact and (rs_compact in compact_text(prev_it.get("text", "")) or rs_compact in compact_text(next_it.get("text", ""))):
                rejected_reasons["adjacent_duplicate"] += 1
                continue
            if is_tail_hallucination(rs_text, recent_texts, min_sim=0.88):
                rejected_reasons["tail_like_hallucination"] += 1
                continue
            if is_hotword_hallucination_text(rs_text, start, end, recent_texts=recent_texts, strict=True):
                rejected_reasons["hotword_hallucination"] += 1
                continue
            words = [w for w in _whisper_words_from_segment(rs, clip_l) if w["end"] > safe_start and w["start"] < safe_end]
            if words:
                start = max(safe_start, float(words[0]["start"]))
                end = min(safe_end, float(words[-1]["end"]))
                if end - start < min_seg_dur:
                    rejected_reasons["words_too_short"] += 1
                    continue
            accepted.append({
                "id": -1,
                "orig_index": 10**9 + added + len(accepted),
                "start": start,
                "end": end,
                "text": rs_text,
                "whisper_words": words,
                "is_gap_rescue": True,
                "gap_rescue_source": "whisper_internal_gap_rescue",
            })

        if not accepted:
            row["rescue_reject_reason"] = "no_accepted_segments"
            row["rejected_reasons"] = dict(rejected_reasons)
            diagnostics.append(row)
            continue
        for item in accepted:
            augmented.append(item)
        added += len(accepted)
        row["rescued_items"] = len(accepted)
        row["rescued_texts"] = [visible_text(x.get("text", ""))[:80] for x in accepted]
        row["rejected_reasons"] = dict(rejected_reasons)
        diagnostics.append(row)

    augmented = sort_items(augmented)
    for i, item in enumerate(augmented):
        item["id"] = i
    globals()["_gap_rescue_added"] = added
    return augmented, diagnostics


def post_split_speaker_repair(items):
    out = [copy_item(x) for x in sort_items(items)]
    repaired = 0
    for i in range(1, len(out) - 1):
        cur = out[i]
        prev_it = out[i - 1]
        next_it = out[i + 1]
        if int(cur.get("track", 0)) != int(prev_it.get("track", 0)) or int(cur.get("track", 0)) != int(next_it.get("track", 0)):
            continue
        cur_dur = item_dur(cur)
        cur_conf = float(cur.get("speaker_confidence", 0.0))
        if not (cur_dur <= 2.2 or cur_conf < 0.78 or bool(cur.get("speaker_uncertain", False))):
            continue
        prev_spk = norm_speaker_key(prev_it.get("speaker"))
        next_spk = norm_speaker_key(next_it.get("speaker"))
        if prev_spk == next_spk and prev_spk != norm_speaker_key(cur.get("speaker")):
            cur["speaker"] = prev_spk
            cur["speaker_uncertain"] = False
            cur["speaker_reason"] = "post_split_bridge_prev_next"
            repaired += 1
            continue
        parent_key = cur.get("_split_parent_key")
        if parent_key and prev_it.get("_split_parent_key") == parent_key and norm_speaker_key(prev_it.get("speaker")) != norm_speaker_key(cur.get("speaker")):
            cur["speaker"] = prev_it.get("speaker")
            cur["speaker_uncertain"] = False
            cur["speaker_reason"] = "post_split_inherit_parent_prev"
            repaired += 1
    globals()["_post_split_speaker_repair"] = repaired
    return sort_items(out)


def protect_split_runs_from_early_end(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    for i in range(len(out) - 1):
        cur = out[i]
        nxt = out[i + 1]
        if int(cur.get("track", 0)) != int(nxt.get("track", 0)):
            continue
        same_spk = norm_speaker_key(cur.get("speaker")) == norm_speaker_key(nxt.get("speaker"))
        same_parent = cur.get("_split_parent_key") and cur.get("_split_parent_key") == nxt.get("_split_parent_key")
        if not (same_spk or same_parent):
            continue
        gap = _f(nxt.get("start")) - _f(cur.get("end"))
        if gap < 0.0 or gap > 0.30:
            continue
        if ends_with_hard_sentence_punct(cur.get("text", "")) and not _looks_like_unfinished_clause(cur.get("text", "")):
            continue
        hold = 0.16 if _looks_like_unfinished_clause(cur.get("text", "")) else 0.12
        target_end = min(_f(audio_len), _f(cur.get("end")) + min(0.18, gap))
        target_end = max(_f(cur.get("end")), min(_f(nxt.get("start")), max(_f(cur.get("end")) + hold, target_end)))
        if target_end > _f(cur.get("end")) + 1e-6:
            cur["end"] = target_end
            fixed += 1
    globals()["_early_end_protected"] = fixed
    return sort_items(out)


def repair_text_duration_mismatch(items, audio_len):
    out = [copy_item(x) for x in sort_items(items)]
    fixed = 0
    for i, cur in enumerate(out):
        if bool(cur.get("is_overlap")) or bool(cur.get("is_secondary_overlap")):
            continue
        chars = text_len(cur.get("text", ""))
        dur = item_dur(cur)
        cps = chars / max(dur, 0.01)
        if not ((dur <= TEXT_DURATION_MISMATCH_MAX_DUR and chars >= TEXT_DURATION_MISMATCH_MIN_CHARS) or cps > max(MAX_READ_CPS * 1.6, 20.0)):
            continue

        prev_end = 0.0 if i == 0 else _f(out[i - 1].get("end"))
        next_start = _f(audio_len) if i == len(out) - 1 else _f(out[i + 1].get("start"))
        left_room = max(0.0, _f(cur.get("start")) - prev_end)
        right_room = max(0.0, next_start - _f(cur.get("end")))
        need_dur = max(dur, readable_min_duration_for_item(cur), chars / max(TEXT_DURATION_TARGET_CPS, 1.0))
        extra = max(0.0, need_dur - dur)
        if extra > 0:
            add_left = min(left_room, extra * 0.4, 0.18)
            add_right = min(right_room, extra - add_left, 0.24)
            cur["start"] = max(0.0, _f(cur.get("start")) - add_left)
            cur["end"] = min(_f(audio_len), _f(cur.get("end")) + add_right)
            fixed += 1

        chars = text_len(cur.get("text", ""))
        dur = item_dur(cur)
        cps = chars / max(dur, 0.01)
        if cps <= 16.0:
            continue

        prev_it = out[i - 1] if i > 0 else None
        if prev_it and norm_speaker_key(prev_it.get("speaker")) == norm_speaker_key(cur.get("speaker")):
            merged_dur = max(_f(prev_it.get("end")), _f(cur.get("end"))) - min(_f(prev_it.get("start")), _f(cur.get("start")))
            merged_chars = text_len(merge_text(prev_it.get("text", ""), cur.get("text", "")))
            if merged_dur <= 2.8 and merged_chars <= 18:
                out[i - 1] = merge_two_items(prev_it, cur)
                out[i]["_drop_after_duration_repair"] = True
                fixed += 1

    out = [x for x in out if not x.get("_drop_after_duration_repair")]
    globals()["_text_duration_mismatch_fixed"] = fixed
    return sort_items(out)

def finalize_items_profile_aware(sentence_items, speaker_segments, audio_len):
    if not sentence_items:
        reset_final_merge_counters()
        return []

    reset_final_merge_counters()
    items = cleanup_items(sentence_items, audio_len)
    items = pre_assign_split_items(items)
    items = cleanup_items(items, audio_len)
    items = assign_speakers(items, speaker_segments)
    items = smooth_balanced_speakers(items)
    items = make_secondary_overlap_items(items)
    items = merge_dependent_short_fragments(items)
    items = merge_tail_fragments(items)
    items = post_assign_split_items(items)
    if "window_majority_speaker_smoother" in globals():
        items = window_majority_speaker_smoother(items)
    items = post_split_speaker_repair(items)
    items = clamp_and_fix_order(items, audio_len)
    items = extend_tail_safely(items, speaker_segments, audio_len)
    items = protect_split_runs_from_early_end(items, audio_len)
    items = enforce_same_speaker_non_overlap(items, audio_len)
    items = normalize_micro_gaps_and_overlaps(items, audio_len)
    items = fix_obviously_early_starts(items, audio_len)
    items = rebalance_sentence_boundaries(items, audio_len)
    items = cleanup_tiny_final_items(items, audio_len)
    items = repair_text_duration_mismatch(items, audio_len)
    items = enforce_readable_item_durations(items, audio_len)
    items = drop_unverified_gap_edge_hallucinations(items, audio_len)
    items = cleanup_items(items, audio_len)
    items = sort_items(items)

    for i, it in enumerate(items):
        it["idx"] = i
        it["speaker"] = norm_speaker_key(it.get("speaker"))
        it["track"] = int(it.get("track", 0 if not ENABLE_MULTI_TRACK_ASS else it.get("track", 0)))
        it["is_overlap"] = bool(it.get("is_secondary_overlap")) or bool(it.get("secondary_speaker"))
        it["start"] = safe_round(it["start"], 3)
        it["end"] = safe_round(it["end"], 3)
        it["text_ass"] = ass_escape(to_ass_text(it.get("text", "")))

    return items

def finalize_items_with_runtime_speakers(sentence_source_items, speaker_segments_runtime, audio_len):
    if ENABLE_DIARIZATION and speaker_segments_runtime:
        return finalize_items_profile_aware(
            sentence_items=sentence_source_items,
            speaker_segments=speaker_segments_runtime,
            audio_len=audio_len,
        )

    reset_final_merge_counters()
    _tmp = cleanup_items(sentence_source_items, audio_len)
    _tmp = pre_assign_split_items(_tmp)
    for z in _tmp:
        z["speaker"] = z.get("speaker", "speaker_unknown")
        z["track"] = int(z.get("track", 0))
        z["secondary_speaker"] = None
        z["secondary_overlap_sec"] = 0.0
        z["has_overlap_region"] = False
    _tmp = merge_dependent_short_fragments(_tmp)
    _tmp = merge_tail_fragments(_tmp)
    _tmp = post_assign_split_items(_tmp)
    _tmp = post_split_speaker_repair(_tmp)
    _tmp = clamp_and_fix_order(_tmp, audio_len)
    _tmp = extend_tail_safely(_tmp, [], audio_len)
    _tmp = protect_split_runs_from_early_end(_tmp, audio_len)
    _tmp = enforce_same_speaker_non_overlap(_tmp, audio_len)
    _tmp = normalize_micro_gaps_and_overlaps(_tmp, audio_len)
    _tmp = fix_obviously_early_starts(_tmp, audio_len)
    _tmp = rebalance_sentence_boundaries(_tmp, audio_len)
    _tmp = cleanup_tiny_final_items(_tmp, audio_len)
    _tmp = repair_text_duration_mismatch(_tmp, audio_len)
    _tmp = enforce_readable_item_durations(_tmp, audio_len)
    _tmp = drop_unverified_gap_edge_hallucinations(_tmp, audio_len)
    _tmp = cleanup_items(_tmp, audio_len)
    final = sort_items(_tmp)
    for i, it in enumerate(final):
        it["idx"] = i
        it["speaker"] = norm_speaker_key(it.get("speaker"))
        it["track"] = int(it.get("track", 0))
        it["is_overlap"] = False
        it["start"] = safe_round(it["start"], 3)
        it["end"] = safe_round(it["end"], 3)
        it["text_ass"] = ass_escape(to_ass_text(it.get("text", "")))
    return final


In [ ]:
#@title **执行 finalize**
# =========================================================
# 执行
# =========================================================
assert "sentence_items" in globals(), "缺少 sentence_items，请先运行前面的对齐单元。"

audio_len = globals().get("AUDIO_LEN", globals().get("audio_len", None))
if audio_len is None:
    if sentence_items:
        audio_len = max(_f(x.get("end")) for x in sentence_items)
    else:
        audio_len = 0.0

speaker_segments_runtime = globals().get("speaker_segments", [])

final_items = finalize_items_with_runtime_speakers(sentence_items, speaker_segments_runtime, audio_len)
gap_rescue_sentence_items, gap_rescue_diagnostics = rescue_internal_speech_gaps(final_items, sentence_items, audio_len)
if len(gap_rescue_sentence_items) > len(sentence_items):
    print("[gap rescue] added sentence items:", len(gap_rescue_sentence_items) - len(sentence_items))
    final_items = finalize_items_with_runtime_speakers(gap_rescue_sentence_items, speaker_segments_runtime, audio_len)
else:
    print("[gap rescue] added sentence items: 0")

globals()["gap_rescue_sentence_items"] = gap_rescue_sentence_items
globals()["gap_rescue_diagnostics"] = gap_rescue_diagnostics
with open("gap_rescue_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(gap_rescue_diagnostics, f, ensure_ascii=False, indent=2)

globals()["final_items"] = final_items
globals()["base_items"] = final_items

with open("final_items.debug.json", "w", encoding="utf-8") as f:
    json.dump(final_items, f, ensure_ascii=False, indent=2)




In [ ]:
#@title **SRT / ASS 导出与统计**
# =========================================================
# 输出文件名
# =========================================================
assert "input_file" in globals(), "缺少 input_file，请先选择输入文件。"
base_name = os.path.splitext(os.path.basename(input_file))[0]
srt_out = f"{base_name}.srt"
ass_out = f"{base_name}.ass"


# =========================================================
# 导出 SRT
# =========================================================
# 使用 UTF-8 BOM，避免 Windows 下 SRT/ASS 中文被误判编码。
with open(srt_out, "w", encoding=SUBTITLE_TEXT_ENCODING, newline="\n") as f:
    for i, item in enumerate(final_items, 1):
        f.write(f"{i}\n")
        f.write(f"{format_srt_time(item['start'])} --> {format_srt_time(item['end'])}\n")
        f.write(f"{visible_text(item['text'])}\n\n")


# =========================================================
# 导出 ASS
# =========================================================
speaker_names = []
for item in final_items:
    style_name = normalize_speaker_name(item.get("speaker"))
    if style_name != "Default" and style_name not in speaker_names:
        speaker_names.append(style_name)

speaker_palette = {
    "Speaker0": "&H00FFFFFF",
    "Speaker1": "&H0058D8FF",
    "Speaker2": "&H00A0FF6B",
    "Speaker3": "&H00FF7AA2",
    "Speaker4": "&H006BD6FF",
    "Speaker5": "&H00D189FF",
    "Default": "&H00FFFFFF",
}

style_lines = []
all_style_names = ["Default"] + speaker_names

for spk_name in all_style_names:
    primary = speaker_palette.get(spk_name, "&H00FFFFFF")
    style_lines.append(
        f"Style: {spk_name},Arial,54,{primary},&H0000FFFF,&H00101010,&H64000000,"
        f"0,0,0,0,100,100,0,0,1,2.2,0.8,2,80,80,48,1"
    )

ass_header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1920
PlayResY: 1080
WrapStyle: 2
ScaledBorderAndShadow: yes
YCbCr Matrix: TV.601

[V4+ Styles]
Format: Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,Alignment,MarginL,MarginR,MarginV,Encoding
{chr(10).join(style_lines)}

[Events]
Format: Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text
"""

with open(ass_out, "w", encoding=SUBTITLE_TEXT_ENCODING, newline="\n") as f:
    f.write(ass_header)
    for item in final_items:
        start = format_ass_time(item["start"])
        end = format_ass_time(item["end"])
        txt = item["text_ass"]
        style_name = normalize_speaker_name(item.get("speaker"))

        track = int(item.get("track", 0))
        layer = ASS_LAYER_SUB if track == 1 else ASS_LAYER_MAIN
        margin_v = ASS_MARGIN_V_SUB if track == 1 else ASS_MARGIN_V_MAIN

        f.write(f"Dialogue: {layer},{start},{end},{style_name},,0000,0000,{margin_v},,{txt}\n")


# =========================================================
# 大段间隔诊断
# =========================================================
def _nearest_chunk_boundary(gap_mid):
    chunks = globals().get("SHARED_AUDIO_CHUNKS", []) or []
    audio_len = float(globals().get("AUDIO_LEN", globals().get("audio_len", 0.0)) or 0.0)
    boundaries = []
    for chunk in chunks:
        for key in ("base_start", "base_end"):
            val = float(chunk.get(key, 0.0) or 0.0)
            if val <= 0.0 or (audio_len > 0 and val >= audio_len - 1e-6):
                continue
            boundaries.append(val)
    if not boundaries:
        return None, None
    nearest = min(boundaries, key=lambda x: abs(x - gap_mid))
    return nearest, abs(nearest - gap_mid)


def _chunk_record_for_time(ts):
    debug = globals().get("_asr_chunk_debug", {}) or {}
    records = debug.get("chunks", []) or []
    if not records:
        return None
    containing = [r for r in records if float(r.get("base_start", 0.0)) <= ts <= float(r.get("base_end", 0.0))]
    if containing:
        r = containing[0]
    else:
        r = min(records, key=lambda x: min(abs(ts - float(x.get("base_start", 0.0))), abs(ts - float(x.get("base_end", 0.0)))))
    return {
        "index": int(r.get("index", -1)),
        "base_start": float(r.get("base_start", 0.0)),
        "base_end": float(r.get("base_end", 0.0)),
        "raw_segments": int(r.get("raw_segments", 0)),
        "kept_segments": int(r.get("kept_segments", 0)),
        "dedup_dropped": int(r.get("dedup_dropped", 0)),
    }


def _gap_rescue_record_for_window(prev_end, next_start):
    records = globals().get("gap_rescue_diagnostics", []) or []
    for rec in records:
        rec_start = float(rec.get("gap_start", rec.get("prev_end", -1.0)) or -1.0)
        rec_end = float(rec.get("gap_end", rec.get("next_start", -1.0)) or -1.0)
        if abs(rec_start - float(prev_end)) <= 0.75 and abs(rec_end - float(next_start)) <= 0.75:
            return rec
    return None


def build_subtitle_gap_diagnostics(items):
    threshold = float(globals().get("SUBTITLE_GAP_DIAG_THRESHOLD_SEC", 5.0))
    boundary_window = float(globals().get("SUBTITLE_GAP_CHUNK_BOUNDARY_WINDOW_SEC", 30.0))
    rows = []
    ordered = sort_items(items)
    for idx in range(len(ordered) - 1):
        prev_it = ordered[idx]
        next_it = ordered[idx + 1]
        if int(prev_it.get("track", 0)) != int(next_it.get("track", 0)):
            continue
        gap = _f(next_it.get("start")) - _f(prev_it.get("end"))
        if gap < threshold:
            continue
        mid = (_f(prev_it.get("end")) + _f(next_it.get("start"))) / 2.0
        boundary, boundary_dist = _nearest_chunk_boundary(mid)
        rescue_rec = _gap_rescue_record_for_window(_f(prev_it.get("end")), _f(next_it.get("start")))
        audio_stats = rescue_rec if rescue_rec else _audio_clip_activity_stats(_f(prev_it.get("end")), _f(next_it.get("start")))
        rows.append({
            "index": len(rows),
            "prev_item_index": idx,
            "next_item_index": idx + 1,
            "gap_sec": safe_round(gap, 3),
            "prev_end": safe_round(_f(prev_it.get("end")), 3),
            "next_start": safe_round(_f(next_it.get("start")), 3),
            "prev_text": visible_text(prev_it.get("text", ""))[:80],
            "next_text": visible_text(next_it.get("text", ""))[:80],
            "nearest_chunk_boundary": safe_round(boundary, 3) if boundary is not None else None,
            "distance_to_chunk_boundary": safe_round(boundary_dist, 3) if boundary_dist is not None else None,
            "near_chunk_boundary": bool(boundary_dist is not None and boundary_dist <= boundary_window),
            "chunk_record": _chunk_record_for_time(mid),
            "audio_active": audio_stats.get("audio_active"),
            "rms_db": audio_stats.get("rms_db"),
            "peak_db": audio_stats.get("peak_db"),
            "rescue_attempted": bool(rescue_rec.get("rescue_attempted")) if rescue_rec else False,
            "rescued_items": int(rescue_rec.get("rescued_items", 0)) if rescue_rec else 0,
            "rescue_reject_reason": rescue_rec.get("rescue_reject_reason") if rescue_rec else None,
        })
    return rows


gap_diagnostics = build_subtitle_gap_diagnostics(final_items)
with open("subtitle_gap_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(gap_diagnostics, f, ensure_ascii=False, indent=2)
globals()["subtitle_gap_diagnostics"] = gap_diagnostics


# =========================================================
# 简洁统计
# =========================================================
def _count_overlap_items(items):
    return sum(1 for x in items if bool(x.get("is_overlap")) or bool(x.get("is_secondary_overlap")))

def _speaker_counter(items):
    c = Counter(normalize_speaker_name(x.get("speaker")) for x in items)
    return dict(sorted(c.items(), key=lambda kv: kv[0]))

if PRINT_FINAL_COUNTS_ONLY:
    print(
        "[final]",
        f"items={len(final_items)}",
        f"overlap_items={_count_overlap_items(final_items)}",
        f"same_merge={int(globals().get('_same_speaker_merged', 0))}",
        f"dependent_merge={int(globals().get('_dependent_short_merged', 0))}",
        f"tail_merge={int(globals().get('_tail_fragment_merged', 0))}",
        f"tail_fragments_merged_or_dropped={int(globals().get('_tail_fragments_merged_or_dropped', 0))}",
        f"tail_rescue_deduped={int(globals().get('_tail_rescue_deduped', 0))}",
        f"gap_rescue_added={int(globals().get('_gap_rescue_added', 0))}",
        f"gap_edge_drop={int(globals().get('_gap_edge_hallucination_dropped', 0))}",
        f"readable_fixed={int(globals().get('_readable_duration_fixed', 0))}",
        f"speaker_smoothed={int(globals().get('_speaker_smoothed', 0))}",
        f"early_start_fixed={int(globals().get('_early_start_fixed', 0))}",
        f"secondary_overlap={int(globals().get('_overlap_secondary_made', 0))}",
        f"drop_empty={int(globals().get('_dropped_empty', 0))}",
        f"drop_bad={int(globals().get('_dropped_bad', 0))}",
        f"drop_hotword={int(globals().get('_dropped_hotword_hallucination', 0)) + int(globals().get('_hotword_hallucination_dropped', 0))}",
        f"drop_hotword_dup={int(globals().get('_dropped_hotword_duplicate', 0))}",
        f"gap_diag={len(gap_diagnostics)}",
        f"gap_near_chunk={sum(1 for x in gap_diagnostics if x.get('near_chunk_boundary'))}",
    )
else:
    print("[final summary]")
    print("items:", len(final_items))
    print("overlap_items:", _count_overlap_items(final_items))
    print("same_merge:", int(globals().get("_same_speaker_merged", 0)))
    print("dependent_merge:", int(globals().get("_dependent_short_merged", 0)))
    print("tail_merge:", int(globals().get("_tail_fragment_merged", 0)))
    print("tail_fragments_merged_or_dropped:", int(globals().get("_tail_fragments_merged_or_dropped", 0)))
    print("tail_rescue_deduped:", int(globals().get("_tail_rescue_deduped", 0)))
    print("gap_rescue_added:", int(globals().get("_gap_rescue_added", 0)))
    print("gap_edge_drop:", int(globals().get("_gap_edge_hallucination_dropped", 0)))
    print("readable_duration_fixed:", int(globals().get("_readable_duration_fixed", 0)), "merged:", int(globals().get("_readable_duration_merged", 0)))
    print("speaker_smoothed:", int(globals().get("_speaker_smoothed", 0)))
    print("early_start_fixed:", int(globals().get("_early_start_fixed", 0)))
    print("secondary_overlap:", int(globals().get("_overlap_secondary_made", 0)))
    print("drop_empty:", int(globals().get("_dropped_empty", 0)))
    print("drop_bad:", int(globals().get("_dropped_bad", 0)))
    print("drop_hotword:", int(globals().get("_dropped_hotword_hallucination", 0)) + int(globals().get("_hotword_hallucination_dropped", 0)))
    print("drop_hotword_dup:", int(globals().get("_dropped_hotword_duplicate", 0)))
    print("hotword_no_hotwords_rescue:", int(globals().get("_hotword_rescue_no_hotwords_used", 0)))
    print("gap_diag:", len(gap_diagnostics), "near_chunk:", sum(1 for x in gap_diagnostics if x.get("near_chunk_boundary")))
    print("speaker_count:", _speaker_counter(final_items))

print(f"saved: {srt_out} / {ass_out}")


## 下载文件


In [ ]:
#@title **下载结果 / Download outputs**
# @markdown <font size="2">Run the export cell first, then click the buttons below to download subtitle files.</font>
# @markdown <br/><font size="2">请先运行前面的导出单元，再点击下方按钮下载字幕文件。</font>

import os
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

def download_srt(_):
    if os.path.exists(srt_out):
        files.download(srt_out)
    else:
        print(f"File {srt_out} not found. Please run the previous cells first.")

def download_ass(_):
    if os.path.exists(ass_out):
        files.download(ass_out)
    else:
        print(f"File {ass_out} not found. Please run the previous cells first.")

btn_srt = widgets.Button(
    description='下载 SRT / Download SRT',
    icon='download',
    layout=globals().get("BTN_LAYOUT"),
    style=globals().get("BTN_STYLE")
)
btn_ass = widgets.Button(
    description='下载 ASS / Download ASS',
    icon='download',
    layout=globals().get("BTN_LAYOUT"),
    style=globals().get("BTN_STYLE")
)

btn_srt.on_click(download_srt)
btn_ass.on_click(download_ass)

display(widgets.HBox([btn_srt, btn_ass]))
